In [ ]:
#
# ▌BUSINESS ENTITY RESOLUTION: AWS SAGEMAKER TRAINING NOTEBOOK
#
# ▌Amazon ML Challenge 2026: matching S2/S3 business records to Source 1 entities
#
# This notebook runs end to end inside **AWS SageMaker** (Notebook Instance or Studio JupyterLab). It reads the challenge data from **Amazon S3**, trains the full pipeline on the **training split only**, and writes the trained artefacts and the two submission files back to S3:
#
# | file | content |
# |---|---|
# | `matching_results.tsv` | final matches per Source 1 entity (the scored file) |
# | `candidate_pairs.tsv` | the exact candidate set the matching model scored |
#
# **Data location:** `s3://aws-ml-162599956323-ap-south-1-an/6ab10eb3b23ba_student_resource/student_resource/` (`dataset/train`, `dataset/test`, `utils/`).
# **Checkpoints:** `.../student_resource/ber_outputs/checkpoints/`. Every finished piece of work (each blocking block, each feature part, each model fold) is uploaded as soon as it is complete.
# **Results location:** `.../student_resource/ber_outputs/runs/<profile>_<timestamp>/` (artefacts, output TSVs, submission zip).
#
# > **Resuming after a crash, a kernel restart or on a new instance:** run the notebook again from the top (*Run All*). Every finished stage, and every finished block, part or fold inside the interrupted stage, is restored from S3, and the work continues from the first missing piece. A resumed run gives bit-identical results to an uninterrupted one.
#
# **Pipeline**
#
# ```
# raw TSV ─► normalisation (any script → Latin, noise cleanup, learned aliases, locality phrases)
#         ─► DENSE VIEW (GPU): MiniLM-L12 bi-encoder (Apache-2.0, 118M) fine-tuned on training pairs
#         ─► BLOCKING: 5 sparse TF-IDF views + the dense view per (country, source), forward + reverse top-k
#                      ─► learned pruner (16 per entity per source)      ═► candidate_pairs.tsv
#         ─► STAGE 1: ~80 pairwise features → LightGBM (5-fold, grouped by entity, out-of-fold)
#         ─► CROSS-ENCODER (optional, GPU): mDeBERTa-v3-base (MIT, 278M) on uncertain pairs
#         ─► STAGE 2: + context features (competing candidates / claimants / sibling agreement) → LightGBM
#         ─► STAGE 3: context recomputed from the stage-2 probabilities → LightGBM (kept only if it wins out-of-fold)
#         ─► DECISION: per entity, the subset maximising the exact expected F0.5   ═► matching_results.tsv
# ```
#
# **Contents**
# 1. SageMaker environment & S3 data
# 2. Pipeline library (written to disk, identical to the submission `src/`)
# 3. Configuration
# 4. Load the training data
# 5. Exploratory data analysis
# 6. Text normalisation
# 7. Entity split
# 8. Learned aliases & noise tokens
# 9. Final views & corpus statistics
# 10. Candidate generation (blocking)
# 11. Stage-1 matcher
# 12. Cross-encoder (optional)
# 13. Stage-2 and stage-3 matchers
# 14. Decision rule, hold-out validation, where the score is lost, ambiguity floor
# 15. Error analysis and corpus-size check
# 16. Upload the trained artefacts to S3
# 17. Inference on the test set
# 18. Validate the submission files
# 19. Package the submission zip and upload it to S3
# 20. Final refit & tuning levers
# 21. Unseen-country check (optional)
#
# > **Fair play.** No external data, APIs or geocoders are used. Every learned artefact comes from `dataset/train`; the test split is downloaded and read only in Section 17, after training is finished, and nothing is ever fitted or tuned on it.

In [ ]:
#
# ▌1. SAGEMAKER ENVIRONMENT & S3 DATA
#
# **Recommended setup**
#
# | item | recommendation |
# |---|---|
# | instance | **`ml.g6.8xlarge`** (32 vCPU, 128 GiB RAM, 1× L4 24 GB) or `ml.g5.8xlarge` (1× A10G 24 GB): full profile + dense view + cross-encoder. The dense view and the cross-encoder need the GPU; without one both are skipped automatically. |
# | storage | EBS volume **≥ 100 GB** (data 2.5 GB + local working copies of the checkpoints, up to ~60 GB at full size). The default 5 GB volume is far too small. |
# | kernel | Notebook Instance: **`conda_pytorch_p310`**. Studio JupyterLab: the default **Python 3** kernel of the SageMaker Distribution image. Both ship PyTorch + CUDA. |
# | IAM | the SageMaker execution role needs `s3:ListBucket`, `s3:GetObject` and `s3:PutObject` on the bucket. |
# | internet | needed once to `pip install` packages and to download `microsoft/mdeberta-v3-base` and `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` from the Hugging Face Hub. Without internet, upload the models to S3 and set `CE_MODEL_S3_URI` / `DENSE_MODEL_S3_URI` in 3.1 (otherwise the stage is skipped automatically). |
#
# Everything runs on the instance's local EBS volume (fast working storage). S3 is the source of the data, the durable home of every checkpoint, and the destination of the results, so nothing is lost when the kernel dies or the instance is replaced.

In [ ]:
# ── 1.1 Inspect the instance: RAM, CPU cores, GPU ────────────────────────────
import os, sys, gc, json, time, shutil, zipfile, platform, subprocess, logging, random
from pathlib import Path

try:
    import psutil
except ImportError:
    %pip install -q psutil
    import psutil

# The cross-encoder runs on PyTorch: keep transformers from loading TensorFlow / JAX (preinstalled on
# SageMaker images), whose CUDA plugins only print "factory already registered" noise.
for _k, _v in (("USE_TF", "0"), ("USE_TORCH", "1"), ("USE_FLAX", "0"), ("USE_JAX", "0"),
               ("TRANSFORMERS_NO_TF", "1"), ("TF_CPP_MIN_LOG_LEVEL", "3")):
    os.environ.setdefault(_k, _v)

ON_SAGEMAKER = Path("/opt/ml/metadata/resource-metadata.json").exists()
RAM_GB = psutil.virtual_memory().total / 1e9
N_CPU = os.cpu_count()
try:
    GPU_INFO = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=20).stdout.strip()
except Exception:
    GPU_INFO = ""
print(f"SageMaker     : {ON_SAGEMAKER}")
print(f"Python        : {platform.python_version()}  ({sys.executable})")
print(f"RAM           : {RAM_GB:.1f} GB")
print(f"CPU cores     : {N_CPU}")
print(f"GPU (driver)  : {GPU_INFO or 'none'}")
# What actually matters: can PyTorch use the GPU? (the cross-encoder needs it)
try:
    import torch
    TORCH_CUDA = torch.cuda.is_available()
    if TORCH_CUDA:
        cap = torch.cuda.get_device_capability(0)
        print(f"PyTorch CUDA  : yes - {torch.cuda.get_device_name(0)}, compute {cap[0]}.{cap[1]}, "
              f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB, "
              f"{'bf16' if cap[0] >= 8 else 'fp16'} mixed precision (torch {torch.__version__})")
    else:
        print(f"PyTorch CUDA  : NO (torch {torch.__version__}) -> the cross-encoder will be skipped")
except ImportError:
    TORCH_CUDA = False
    print("PyTorch CUDA  : torch not installed -> the cross-encoder will be skipped "
          "(use the conda_pytorch_p310 / SageMaker Distribution kernel)")
if GPU_INFO and not TORCH_CUDA:
    print("WARNING: a GPU is attached but PyTorch cannot use it - pick a PyTorch GPU kernel to train the cross-encoder.")

In [ ]:
# ── 1.2 Install the pipeline dependencies ────────────────────────────────────
# numpy / pandas / scipy / scikit-learn / boto3 (and torch on the PyTorch kernels) are pre-installed.
import importlib.util
%pip install -q "lightgbm>=4.3" "rapidfuzz>=3.6" "anyascii>=0.3.2" "pyarrow>=14" boto3
if importlib.util.find_spec("torch") is not None:
    # cross-encoder extras; transformers is installed only if missing, so the kernel's torch stays compatible
    if importlib.util.find_spec("transformers") is None:
        %pip install -q "transformers>=4.40"
    %pip install -q sentencepiece protobuf
else:
    print("PyTorch not found in this kernel -> the cross-encoder will be skipped "
          "(use conda_pytorch_p310 / the SageMaker Distribution kernel to enable it)")

import numpy as np, pandas as pd, scipy, sklearn, lightgbm, rapidfuzz, anyascii, boto3
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
for m in (np, pd, scipy, sklearn, lightgbm, rapidfuzz, boto3):
    print(f"{m.__name__:<12} {m.__version__}")

In [ ]:
# ── 1.3 S3 locations and local working directories (EDIT HERE) ───────────────
S3_URI        = "s3://aws-ml-162599956323-ap-south-1-an/6ab10eb3b23ba_student_resource/student_resource/"  # <── EDIT if moved
S3_REGION     = "ap-south-1"                                   # region of the bucket
S3_OUTPUT_URI = S3_URI.rstrip("/") + "/ber_outputs/"           # artefacts, submissions and zips go here
S3_CKPT_URI   = S3_OUTPUT_URI + "checkpoints/"                 # every checkpoint is mirrored here (resume point)
S3_ENDPOINT   = os.environ.get("BER_S3_ENDPOINT") or None      # only for a custom / VPC endpoint URL

# Local EBS volume: /home/ec2-user/SageMaker (Notebook Instance) or /home/sagemaker-user (Studio).
LOCAL_ROOT = Path(os.environ.get("BER_LOCAL_ROOT") or next(
    (Path(p) / "ber" for p in ("/home/ec2-user/SageMaker", "/home/sagemaker-user") if Path(p).is_dir()),
    Path.home() / "ber"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SR = LOCAL_ROOT / "student_resource"                     # local mirror of the S3 prefix
WORK_DIR = LOCAL_ROOT / "work"                                 # local copies of the checkpoints + artefacts
LEGACY_WORK_DIR = LOCAL_ROOT / "ber_work"                      # earlier notebook version (its frames are reused if valid)
OUT_DIR = LOCAL_ROOT / "output"                                # submission TSVs
PKG = (LOCAL_ROOT / "business_entity_resolution").resolve().as_posix()   # library + submission code
for d in (WORK_DIR, OUT_DIR, Path(PKG) / "src" / "ber", Path(PKG) / "docs"):
    d.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(LOCAL_ROOT).free / 1e9
print("S3 data   :", S3_URI)
print("S3 output :", S3_OUTPUT_URI)
print("S3 ckpts  :", S3_CKPT_URI)
print("LOCAL_ROOT:", LOCAL_ROOT, f"({free_gb:.0f} GB free)")
if free_gb < 70:
    print("WARNING: full-size training needs ~70 GB free on the local volume - enlarge the space's storage "
          f"(Space settings, up to 100 GB), or delete {LEGACY_WORK_DIR} (only an earlier version's cache).")
if free_gb < 30 and not os.environ.get("BER_DEV_SAMPLE"):
    raise RuntimeError(f"only {free_gb:.0f} GB free on {LOCAL_ROOT}: a full-size run fails part-way with "
                       "'No space left on device'. Stop the space, set its storage to 100 GB, and start again "
                       "(finished work is restored from S3).")

In [ ]:
# ── 1.4 S3 helpers (boto3: parallel, resumable, size-checked) ────────────────
from concurrent.futures import ThreadPoolExecutor
from boto3.s3.transfer import TransferConfig
from botocore.config import Config as BotoConfig
from botocore.exceptions import ClientError, NoCredentialsError

S3 = boto3.client("s3", region_name=S3_REGION, endpoint_url=S3_ENDPOINT,
                  config=BotoConfig(max_pool_connections=64, retries={"max_attempts": 10, "mode": "adaptive"}))
XFER = TransferConfig(multipart_threshold=64 * 2**20, multipart_chunksize=64 * 2**20, max_concurrency=16)


def split_s3(uri: str) -> tuple:
    assert uri.startswith("s3://"), f"not an s3:// URI: {uri}"
    bucket, _, key = uri[5:].partition("/")
    return bucket, key


def s3_list(uri: str):
    # yields (key, size) for every object under the prefix
    bucket, prefix = split_s3(uri)
    for page in S3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            yield obj["Key"], obj["Size"]


def s3_download(uri: str, dest: Path) -> None:
    # mirror every object under the prefix into dest; complete files are skipped, so re-runs are free
    uri = uri if uri.endswith("/") else uri + "/"
    bucket, prefix = split_s3(uri)
    todo = []
    for key, size in s3_list(uri):
        rel = key[len(prefix):]
        if not rel or key.endswith("/") or "__pycache__" in rel:
            continue
        path = Path(dest) / rel
        if not (path.exists() and path.stat().st_size == size):
            todo.append((key, path, size))

    def get(item):
        key, path, _ = item
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".part")
        S3.download_file(bucket, key, str(tmp), Config=XFER)
        tmp.replace(path)                     # atomic: a half-downloaded file is never mistaken as complete

    with ThreadPoolExecutor(8) as ex:
        list(ex.map(get, todo))
    print(f"{uri} -> {dest}: {len(todo)} file(s) downloaded "
          f"({sum(t[2] for t in todo) / 1e9:.2f} GB), others already present")


def s3_upload(path: Path, uri: str) -> None:
    # upload one file (to uri, or into uri if it ends with '/') or a whole directory tree
    path = Path(path)
    bucket, key = split_s3(uri)
    if path.is_file():
        jobs = [(path, key + path.name if key.endswith("/") or not key else key)]
    else:
        jobs = [(p, f"{key.rstrip('/')}/{p.relative_to(path).as_posix()}")
                for p in path.rglob("*") if p.is_file() and "__pycache__" not in p.parts]
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(lambda j: S3.upload_file(str(j[0]), bucket, j[1], Config=XFER), jobs))
    print(f"{path} -> {uri} ({len(jobs)} file(s))")


def s3_latest_run(profile: str):
    # newest ber_outputs/runs/<profile>_<timestamp>/ that holds complete artefacts (meta.json)
    bucket, prefix = split_s3(S3_OUTPUT_URI + "runs/")
    runs = []
    for page in S3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix, Delimiter="/"):
        runs += [p["Prefix"] for p in page.get("CommonPrefixes", [])]
    for run in sorted(r for r in runs if r[len(prefix):].startswith(profile + "_"))[::-1]:
        if next(s3_list(f"s3://{bucket}/{run}artifacts/meta.json"), None):
            return f"s3://{bucket}/{run}"
    return None


# access check: fail early with a clear message instead of deep inside the pipeline
try:
    _root = split_s3(S3_URI)[1]
    listing = {k[len(_root):].split("/")[0] for k, _ in s3_list(S3_URI)}
except NoCredentialsError as e:
    raise RuntimeError("no AWS credentials - run this notebook on SageMaker with an execution role") from e
except ClientError as e:
    raise RuntimeError(f"cannot list {S3_URI}: {e.response['Error'].get('Code')} - the SageMaker execution "
                       "role needs s3:ListBucket / s3:GetObject / s3:PutObject on this bucket") from e
print("top-level entries under S3_URI:", sorted(listing))
assert "dataset" in listing, "S3_URI must point at the student_resource/ folder (it should contain dataset/)"

In [ ]:
# ── 1.5 Download the TRAINING split (+ validator) from S3 to the local volume ─
# The test split is downloaded only in Section 17, after training is finished.
s3_download(S3_URI.rstrip("/") + "/dataset/train/", LOCAL_SR / "dataset" / "train")
s3_download(S3_URI.rstrip("/") + "/utils/", LOCAL_SR / "utils")
DATA_DIR = LOCAL_SR / "dataset"
VALIDATOR = LOCAL_SR / "utils" / "validate_submission.py"
assert (DATA_DIR / "train" / "train_source1.tsv").exists(), f"train_source1.tsv missing under {DATA_DIR / 'train'}"
for f in sorted((DATA_DIR / "train").glob("*.tsv")):
    print(f"  {f.name:<28} {f.stat().st_size / 1e6:9.1f} MB")
print("validator present:", VALIDATOR.exists())

In [ ]:
# ── 1.6 Logging (every pipeline step prints start / end / duration) ─────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s | %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)], force=True)
logging.getLogger("ber").setLevel(logging.INFO)
logging.getLogger("botocore").setLevel(logging.WARNING)
random.seed(2026); np.random.seed(2026)

In [ ]:
#
# ▌2. PIPELINE LIBRARY (`BER` PACKAGE)
#
# The cells below write the library to `business_entity_resolution/src/ber/`. It is **the same code** as the submission package (`code/business_entity_resolution/src/`), so what you train here is exactly what you submit. Each module is small and single-purpose:
#
# | module | role |
# |---|---|
# | `config.py` | every hyper-parameter in one place |
# | `io.py` | tab-separated readers / writers |
# | `normalize.py` | transliteration, noise cleanup, canonicalisation, locality phrases, phonetic skeleton |
# | `aliases.py` | token aliases and noise tokens learned from training labels |
# | `blocking.py` | five-view sparse blocking (+ the dense view) and the learned pruner |
# | `dense.py` | learned dense retrieval view (bi-encoder training, embeddings, exact GPU top-k) |
# | `features.py` | pairwise, group and context features |
# | `models.py` | grouped K-fold LightGBM with out-of-fold predictions |
# | `cross_encoder.py` | optional multilingual cross-encoder |
# | `decision.py` | exact expected-F0.5 selection and the macro F0.5 metric |
# | `checkpoint.py` | stage checkpoints on local disk, mirrored to S3 |
# | `pipeline.py` | the resumable train / predict steps used below |

In [ ]:
#
# ▌2.1 Package marker
#
# Makes `ber` an importable package.

In [ ]:
%%writefile {PKG}/src/ber/__init__.py
"""Business Entity Resolution pipeline (ML Challenge 2026)."""

In [ ]:
#
# ▌2.2 Configuration (`config.py`)
#
# Blocking sizes, LightGBM parameters, cross-encoder and decision settings. Section 3 adapts a few of them to the runtime.

In [ ]:
%%writefile {PKG}/src/ber/config.py
"""Central configuration for the Business Entity Resolution pipeline.

Every tunable knob lives here so that a run is fully described by one object
(it is serialised next to the trained artefacts as ``config.json``).
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field, fields
from pathlib import Path


@dataclass
class BlockingConfig:
    # Per-view forward top-k (S1 -> target source) retrieved by sparse cosine.
    # At full scale (~1M entities per country block) a top-10 is crowded out by
    # look-alike names, which is why recall fell from 99.9% on a 20k slice to 98.1%.
    k_name_tok: int = 15
    k_name_char: int = 15
    k_name_skel: int = 15
    k_addr: int = 15
    k_name_addr: int = 15
    # Reverse top-k (target record -> S1) per view; protects records whose true
    # S1 is crowded out of the forward list by generic look-alikes.
    k_reverse: int = 5
    # Dense view (learned bi-encoder, see dense.py): exact top-k by embedding
    # cosine, forward per S1 entity and reverse per S2/S3 record. It has no
    # document-frequency cap, so it reaches records whose every name token is
    # too common for the sparse views. Used when DenseConfig.enabled.
    k_dense: int = 30
    k_dense_reverse: int = 10
    # Document-frequency caps (absolute counts inside one country block). Tokens
    # more frequent than this carry no blocking signal and blow up the join.
    max_df_name_tok: int = 3000
    max_df_name_char: int = 4000
    max_df_name_skel: int = 3000
    max_df_addr: int = 3000
    max_df_name_addr: int = 400
    # Final pruning: candidates kept per (S1 entity, target source) after the
    # learned blocking ranker. GT has at most 6 true matches per source; the
    # margin absorbs pruner mistakes (union -> candidate recall lost 0.16 pt at 12).
    keep_per_source: int = 16
    # Rows of S1 processed per sparse matmul chunk.
    chunk_rows: int = 1000


@dataclass
class ModelConfig:
    n_folds: int = 5
    seed: int = 2026
    # Fraction of training S1 entities held out (never trained on) to give an
    # honest validation F0.5, as the challenge statement recommends.
    holdout_frac: float = 0.15
    # Optional cap on S1 entities used to fit the GBDTs (None = all of the
    # non-holdout ones). Rows of the remaining entities are still scored.
    max_train_s1: int | None = None
    # S1 entities (disjoint from fit / hold-out when possible) used to train the
    # blocking pruner.
    ranker_s1: int = 150_000
    # Fraction of the non-hold-out S1 entities whose true pairs train the dense
    # bi-encoder (only when the dense view is enabled). These entities are kept
    # out of every other fit (pruner, GBDTs, cross-encoder): the encoder has
    # seen their pairs, so their embedding cosines would be optimistic - the
    # same stacking leak the cross-encoder avoids with its 2-fold scheme.
    dense_frac: float = 0.15
    lgb_stage1: dict = field(default_factory=lambda: dict(
        objective="binary", learning_rate=0.05, num_leaves=255,
        min_data_in_leaf=200, feature_fraction=0.7, bagging_fraction=0.8,
        bagging_freq=1, lambda_l2=10.0, max_bin=255, verbose=-1,
        num_threads=0,
    ))
    lgb_stage2: dict = field(default_factory=lambda: dict(
        objective="binary", learning_rate=0.03, num_leaves=127,
        min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
        bagging_freq=1, lambda_l2=10.0, max_bin=255, verbose=-1,
        num_threads=0,
    ))
    # Stage 3 (collective refinement): the competition / sibling-anchor context is
    # recomputed from the stage-2 probabilities - which already include the
    # cross-encoder - and a third grouped-OOF LightGBM is trained on it. Most
    # remaining errors are "which of several look-alike entities owns this
    # record", exactly what better competitor probabilities inform. The decision
    # step keeps stage 3 only if it beats stage 2 on out-of-fold data.
    stage3: bool = True
    lgb_stage3: dict = field(default_factory=lambda: dict(
        objective="binary", learning_rate=0.03, num_leaves=127,
        min_data_in_leaf=200, feature_fraction=0.8, bagging_fraction=0.8,
        bagging_freq=1, lambda_l2=10.0, max_bin=255, verbose=-1,
        num_threads=0,
    ))
    num_boost_round: int = 5000
    early_stopping: int = 100
    # Number of top candidates (by stage-1 prob) used as sibling "anchors".
    n_anchors: int = 3


@dataclass
class CrossEncoderConfig:
    enabled: bool = True
    # mDeBERTa-v3-base: MIT licence, 278M params, multilingual tokenizer that
    # natively covers Devanagari/Telugu/Tamil/Bengali/... and French accents.
    model_name: str = "microsoft/mdeberta-v3-base"
    # Two "name | address" records in Indic scripts easily exceed 96 pieces.
    max_len: int = 128
    # Only pairs whose stage-1 probability falls in this band are re-scored;
    # outside it the GBDT is already (near-)certain. Remaining hold-out errors
    # include confident ones, so the band is wider than the original 0.02-0.98.
    band_low: float = 0.01
    band_high: float = 0.99
    max_train_pairs: int = 3_000_000
    epochs: int = 1
    lr: float = 2e-5
    batch_size: int = 128
    infer_batch_size: int = 512
    warmup_frac: float = 0.06
    fp16: bool = True
    # DataLoader worker processes for training (speed only, not part of any checkpoint key);
    # 0 where /dev/shm is too small for worker processes (some container setups).
    num_workers: int = 2


@dataclass
class DenseConfig:
    """Learned dense retrieval view (dense.py): a siamese sentence encoder
    fine-tuned with in-batch-negative contrastive loss on training true pairs."""
    enabled: bool = True
    # Apache-2.0, 118M parameters, 384-dim embeddings, multilingual vocabulary.
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    max_len: int = 64
    epochs: int = 2
    lr: float = 5e-5
    batch_size: int = 512          # pairs per batch -> 511 in-batch negatives per pair
    temperature: float = 0.05
    warmup_frac: float = 0.05
    max_train_pairs: int = 2_000_000
    fp16: bool = True
    infer_batch_size: int = 2048   # speed only (not part of any checkpoint key)


@dataclass
class DecisionConfig:
    beta: float = 0.5
    # Candidates per S1 entering the exact expected-F optimisation.
    max_items: int = 20
    # Grid searched on out-of-fold predictions.
    temperatures: tuple = (0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.5, 1.75, 2.0, 2.5)
    thresholds: tuple = tuple(round(0.30 + 0.02 * i, 2) for i in range(34))
    # Calibration offsets (logit scale) tried around the best expected-F temperature: the
    # previous run's best rule was a plain threshold, a sign the probabilities are shifted.
    biases: tuple = (-1.0, -0.5, -0.25, 0.25, 0.5, 1.0)
    # How a record's probability is shared among its claimants (decision.PROB_MODES).
    prob_modes: tuple = ("raw", "renorm", "best")


def _build(cls, values: dict):
    """Dataclass from a dict, ignoring keys an older / newer version wrote."""
    known = {f.name for f in fields(cls)}
    return cls(**{k: v for k, v in values.items() if k in known})


@dataclass
class Config:
    blocking: BlockingConfig = field(default_factory=BlockingConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    cross_encoder: CrossEncoderConfig = field(default_factory=CrossEncoderConfig)
    dense: DenseConfig = field(default_factory=DenseConfig)
    decision: DecisionConfig = field(default_factory=DecisionConfig)
    n_jobs: int = -1
    # Minimum co-occurrence count for a learned token alias.
    alias_min_count: int = 25
    # A component seen at least this many times in a country's S1 records is
    # treated as a locality (city / district / state / region) component.
    geo_min_count: int = 30

    def save(self, path: Path) -> None:
        Path(path).write_text(json.dumps(asdict(self), indent=2, default=list))

    @classmethod
    def load(cls, path: Path) -> "Config":
        raw = json.loads(Path(path).read_text())
        cfg = cls()
        cfg.blocking = _build(BlockingConfig, raw["blocking"])
        cfg.model = _build(ModelConfig, raw["model"])
        cfg.cross_encoder = _build(CrossEncoderConfig, raw["cross_encoder"])
        # artefacts trained before the dense view existed have no "dense" section
        cfg.dense = _build(DenseConfig, raw["dense"]) if "dense" in raw else DenseConfig(enabled=False)
        dec = dict(raw["decision"])
        for k in ("temperatures", "thresholds", "biases", "prob_modes"):
            if k in dec:                          # older artefacts lack the newer grids
                dec[k] = tuple(dec[k])
        cfg.decision = _build(DecisionConfig, dec)
        for k in ("n_jobs", "alias_min_count", "geo_min_count"):
            setattr(cfg, k, raw[k])
        return cfg

In [ ]:
#
# ▌2.3 Input / output (`io.py`)
#
# All files are TAB-separated; ID lists are comma-separated. Outputs contain one row per Source 1 entity, de-duplicated, S2-/S3- IDs only.

In [ ]:
%%writefile {PKG}/src/ber/io.py
"""TSV readers / writers. Every file in the challenge is tab-separated."""
from __future__ import annotations

import csv
import logging
from pathlib import Path

import pandas as pd

log = logging.getLogger(__name__)

SOURCE_COLS = ["entity_id", "business_name", "business_address", "country"]


def read_source(path: Path) -> pd.DataFrame:
    """Read one source file as plain strings (empty string for missing)."""
    df = pd.read_csv(
        path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
        quoting=csv.QUOTE_NONE, engine="c", encoding="utf-8",
    )
    missing = [c for c in SOURCE_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}")
    df = df[SOURCE_COLS].copy()
    for c in SOURCE_COLS:
        df[c] = df[c].fillna("").astype(str)
    df["entity_id"] = df["entity_id"].str.strip()
    df["country"] = df["country"].str.strip()
    df = df[df["entity_id"] != ""].drop_duplicates("entity_id").reset_index(drop=True)
    log.info("read %s: %d rows", Path(path).name, len(df))
    return df


def read_split(data_dir: Path, split: str) -> dict[str, pd.DataFrame]:
    data_dir = Path(data_dir) / split
    return {
        src: read_source(data_dir / f"{split}_source{src[-1]}.tsv")
        for src in ("s1", "s2", "s3")
    }


def read_ground_truth(path: Path) -> dict[str, list[str]]:
    gt: dict[str, list[str]] = {}
    with open(path, encoding="utf-8") as f:
        next(f)
        for line in f:
            s1, _, rest = line.rstrip("\n").partition("\t")
            s1 = s1.strip()
            if s1:
                # a record listed twice is one match (the metric works on sets); an entity on
                # two lines keeps the records of both
                ids = gt.get(s1, []) + [x.strip() for x in rest.split(",") if x.strip()]
                gt[s1] = list(dict.fromkeys(ids))
    log.info("read ground truth: %d S1 entities", len(gt))
    return gt


def write_id_lists(path: Path, header: tuple[str, str], s1_ids, mapping: dict) -> None:
    """Write one row per S1 id; ids are de-duplicated preserving order."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(f"{header[0]}\t{header[1]}\n")
        for s1 in s1_ids:
            ids = mapping.get(s1, ())
            seen, out = set(), []
            for x in ids:
                if x not in seen and (x.startswith("S2-") or x.startswith("S3-")):
                    seen.add(x)
                    out.append(x)
            f.write(f"{s1}\t{','.join(out)}\n")
    log.info("wrote %s (%d rows)", path, len(s1_ids))

In [ ]:
#
# ▌2.4 Text normalisation (`normalize.py`)
#
# Static rules (any script → Latin with `anyascii`, OCR-digit repair, domain / DBA / acronym handling, legal-form and street-type canonicalisation incl. French forms, number parsing), locality-phrase joining, and a transliteration-aware **phonetic skeleton** (`sophtveyr` and `software` share a skeleton).

In [ ]:
%%writefile {PKG}/src/ber/normalize.py
"""Deterministic, country-agnostic text normalisation.

Two passes:
  * ``normalize_frame``   - static rules (transliteration, punctuation / noise
    cleanup, legal-form and street-type canonicalisation, number parsing).
  * ``finalize_frame``    - applies the aliases learned from training labels
    (see ``aliases.py``) and derives the views used by blocking and features
    (core name, no-space name, phonetic skeleton, address words / numbers).

Nothing here is specific to US / India: the rules are generic, the French legal
forms and street types are included because France is announced for the test
set, and every learned alias comes from the training data.
"""
from __future__ import annotations

import re
import unicodedata

import numpy as np
import pandas as pd
from anyascii import anyascii
from joblib import Parallel, delayed

# --------------------------------------------------------------------------
# Static dictionaries (generic domain knowledge, no external lookup)
# --------------------------------------------------------------------------
LEGAL_CANON = {
    # English / Indian
    "private": "pvt", "pvt": "pvt", "pte": "pvt", "prv": "pvt",
    "limited": "ltd", "ltd": "ltd", "limitd": "ltd", "ltda": "ltd",
    "incorporated": "inc", "inc": "inc", "incorporation": "inc",
    "corporation": "corp", "corp": "corp", "corpn": "corp",
    "company": "co", "co": "co", "cos": "co", "compnay": "co",
    "llc": "llc", "llp": "llp", "lp": "lp", "pc": "pc", "plc": "plc",
    "pllc": "pllc", "lllp": "lllp", "pa": "pa", "opc": "opc",
    # French / continental
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "sa": "sa", "eurl": "eurl",
    "sci": "sci", "snc": "snc", "scp": "scp", "selarl": "selarl", "scop": "scop",
    "scm": "scm", "sca": "sca", "eirl": "eirl", "gie": "gie", "societe": "ste",
    "ste": "ste", "gmbh": "gmbh", "ag": "ag", "bv": "bv", "nv": "nv", "srl": "srl",
    "spa": "spa",
}
LEGAL_TOKENS = set(LEGAL_CANON.values())

# Tokens that decorate a name without identifying it. The data-driven list
# learned in aliases.py is added on top of this one.
HONORIFIC_TOKENS = {
    "the", "m", "s", "ms", "messrs", "mr", "mrs", "dr", "shri", "sri", "smt",
    "sh", "and", "of", "dba", "aka", "le", "la", "les", "l", "du", "de", "des",
    "et", "d",
}

STREET_CANON = {
    "street": "st", "str": "st", "saint": "st", "st": "st", "sainte": "ste",
    "road": "rd", "rd": "rd", "avenue": "ave", "ave": "ave", "av": "ave",
    "drive": "dr", "dr": "dr", "drv": "dr", "lane": "ln", "ln": "ln",
    "boulevard": "blvd", "blvd": "blvd", "bd": "blvd", "bld": "blvd",
    "court": "ct", "ct": "ct", "place": "pl", "pl": "pl", "circle": "cir",
    "cir": "cir", "highway": "hwy", "hwy": "hwy", "parkway": "pkwy",
    "pkwy": "pkwy", "pky": "pkwy", "terrace": "ter", "ter": "ter",
    "trail": "trl", "trl": "trl", "square": "sq", "sq": "sq", "suite": "ste",
    "ste": "ste", "apartment": "apt", "apt": "apt", "building": "bldg",
    "bldg": "bldg", "floor": "fl", "flr": "fl", "fl": "fl", "north": "n",
    "south": "s", "east": "e", "west": "w", "northeast": "ne", "northwest": "nw",
    "southeast": "se", "southwest": "sw", "number": "no", "num": "no",
    "mount": "mt", "mt": "mt", "fort": "ft", "ft": "ft", "point": "pt",
    "center": "ctr", "centre": "ctr", "ctr": "ctr", "expressway": "expy",
    "freeway": "fwy", "way": "way", "plaza": "plz", "plz": "plz",
    # French
    "rue": "rue", "chemin": "ch", "impasse": "imp", "imp": "imp",
    "allee": "all", "route": "rte", "rte": "rte", "faubourg": "fbg",
    "fbg": "fbg", "quai": "quai", "cours": "crs", "residence": "res",
    "batiment": "bat", "bat": "bat", "etage": "fl", "lieu": "ld",
    # India
    "marg": "marg", "nagar": "nagar", "nr": "near", "opp": "opposite",
    "opposite": "opposite",
}

ORDINAL_WORDS = {
    "first": "1", "second": "2", "third": "3", "fourth": "4", "fifth": "5",
    "sixth": "6", "seventh": "7", "eighth": "8", "ninth": "9", "tenth": "10",
    "premier": "1", "premiere": "1", "deuxieme": "2", "troisieme": "3",
}

NULL_WORDS = {"null", "none", "nan", "nil", "undefined"}

_DOMAIN_RE = re.compile(
    r"\b(?:https?\s*:?\s*/*\s*)?(?:www\s*\.\s*)?([a-z0-9][a-z0-9\-]*)\s*\.\s*"
    r"(?:com|c0m|net|org|0rg|in|co|biz|info|us|fr|io|online|shop|store|app)\b"
)
_ACRONYM_RE = re.compile(r"\b(?:[a-z]\s?\.\s?){2,}")
_ID_NOISE_RE = re.compile(r"\(?\b(?:id|ref|no)\s*[:#]\s*\d+\)?")
_MS_RE = re.compile(r"\bm\s*/\s*s\b")
_DBA_RE = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|dba|aka|t/a|trading as)\b")
_POBOX_RE = re.compile(r"\b(?:p\s*\.?\s*o\s*\.?\s*box|post\s*box|boite\s*postale|bp)\s*#?\s*(\d+)")
_NULL_RE = re.compile(r"<\s*null\s*>|\bnull\b|\bnone\b|\bnan\b")
_ORDINAL_SUFFIX_RE = re.compile(r"\b(\d+)(?:st|nd|rd|th|er|e|eme)\b")
_TOKEN_RE = re.compile(r"[a-z]+|\d+")
_NONLATIN_RE = re.compile(r"[^\x00-\u024f\u1e00-\u1eff\u2000-\u206f\u20a0-\u20cf]")
_OCR_MAP = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "7": "t", "8": "b"})


def _fold(s: str) -> str:
    """Unicode-normalise, transliterate any script to ASCII, lowercase."""
    s = unicodedata.normalize("NFKC", s)
    return anyascii(s).lower()


def _fix_ocr(tok: str) -> str:
    """Fix digit-for-letter substitutions inside a word ('h0mes', 'zi1via')."""
    if tok.isalpha() or tok.isdigit():
        return tok
    letters = sum(ch.isalpha() for ch in tok)
    if letters >= 2 and letters >= 2 * (len(tok) - letters):
        return tok.translate(_OCR_MAP)
    return tok


def _name_tokens(s: str) -> list[str]:
    s = _MS_RE.sub(" ", s)
    s = s.replace("&", " and ").replace("+", " and ").replace("@", " at ")
    s = _ID_NOISE_RE.sub(" ", s)
    s = _DOMAIN_RE.sub(lambda m: " " + m.group(1).replace("-", "") + " ", s)
    s = _ACRONYM_RE.sub(lambda m: " " + re.sub(r"[\s.]", "", m.group(0)) + " ", s)
    s = s.replace("'", "").replace("`", "")
    toks = re.findall(r"[a-z0-9]+", s)
    out = []
    for t in toks:
        t = _fix_ocr(t)
        out.append(LEGAL_CANON.get(t, t))
    return out


def normalize_name(raw: str) -> tuple[str, str, int, int, int]:
    """Return (tokens, dba_alt_tokens, nonlatin, is_domain, has_dba)."""
    nonlatin = int(bool(_NONLATIN_RE.search(raw)))
    s = _fold(raw)
    is_domain = int(bool(_DOMAIN_RE.search(s)) or s.strip().startswith("#"))
    parts = _DBA_RE.split(s)
    has_dba = int(len(parts) > 1)
    toks = _name_tokens(" ".join(parts))
    alt = _name_tokens(parts[-1]) if has_dba else toks
    return " ".join(toks), " ".join(alt), nonlatin, is_domain, has_dba


def normalize_address(raw: str) -> tuple[str, str, int]:
    """Return (tokens, components joined by '|', has_pobox)."""
    s = _fold(raw)
    s = _NULL_RE.sub(" ", s)
    has_pobox = 0
    m = _POBOX_RE.search(s)
    if m:
        has_pobox = 1
        s = _POBOX_RE.sub(" ", s)
    s = _ORDINAL_SUFFIX_RE.sub(r"\1", s)
    comps_out, all_toks = [], []
    for comp in s.split(","):
        toks = []
        for t in _TOKEN_RE.findall(comp):
            if t.isdigit():
                t = t.lstrip("0")
                if not t:
                    continue
            else:
                if t in NULL_WORDS:
                    continue
                t = ORDINAL_WORDS.get(t, t)
                t = STREET_CANON.get(t, t)
            toks.append(t)
        if toks:
            comps_out.append(" ".join(toks))
            all_toks.extend(toks)
    return " ".join(all_toks), "|".join(comps_out), has_pobox


def _normalize_chunk(names: list[str], addrs: list[str]):
    n_out = [normalize_name(x) for x in names]
    a_out = [normalize_address(x) for x in addrs]
    return n_out, a_out


def normalize_frame(df: pd.DataFrame, n_jobs: int = -1, chunk: int = 50_000) -> pd.DataFrame:
    """Static normalisation of a source frame (parallel over chunks)."""
    names = df["business_name"].tolist()
    addrs = df["business_address"].tolist()
    spans = [(i, min(i + chunk, len(df))) for i in range(0, len(df), chunk)]
    res = Parallel(n_jobs=n_jobs, batch_size=1)(
        delayed(_normalize_chunk)(names[a:b], addrs[a:b]) for a, b in spans
    )
    n_rows = [r for part in res for r in part[0]]
    a_rows = [r for part in res for r in part[1]]
    out = pd.DataFrame({
        "entity_id": df["entity_id"].values,
        "country": df["country"].values,
        "raw_name": df["business_name"].values,
        "raw_addr": df["business_address"].values,
    })
    out["name_tok"] = [r[0] for r in n_rows]
    out["name_alt_tok"] = [r[1] for r in n_rows]
    out["name_nonlatin"] = np.array([r[2] for r in n_rows], dtype=np.int8)
    out["name_is_domain"] = np.array([r[3] for r in n_rows], dtype=np.int8)
    out["name_has_dba"] = np.array([r[4] for r in n_rows], dtype=np.int8)
    out["addr_tok"] = [r[0] for r in a_rows]
    out["addr_comps"] = [r[1] for r in a_rows]
    out["addr_pobox"] = np.array([r[2] for r in a_rows], dtype=np.int8)
    return out


# --------------------------------------------------------------------------
# Multi-word locality phrases ("uttar pradesh", "north carolina",
# "nouvelle aquitaine") are joined into one token so that an abbreviation or a
# transliteration can be aliased to the whole phrase (up -> uttar_pradesh).
# The phrase list is read, without labels, from the S1 records of whichever
# corpus is being processed, so it adapts to an unseen country.
# --------------------------------------------------------------------------
def learn_phrases(S1: pd.DataFrame, min_count: int) -> set:
    counts: dict = {}
    for cs in S1["addr_comps"].values:
        for c in cs.split("|") if cs else ():
            toks = c.split()
            if 2 <= len(toks) <= 3 and not any(t.isdigit() for t in toks):
                counts[c] = counts.get(c, 0) + 1
    return {tuple(c.split()) for c, n in counts.items() if n >= min_count}


def _join_chunk(comps_list: list[str], phrases: set) -> tuple[list[str], list[str]]:
    out_c, out_t = [], []
    for cs in comps_list:
        new_comps = []
        for c in cs.split("|") if cs else ():
            toks = c.split()
            res, i = [], 0
            while i < len(toks):
                if i + 3 <= len(toks) and tuple(toks[i:i + 3]) in phrases:
                    res.append("_".join(toks[i:i + 3]))
                    i += 3
                elif i + 2 <= len(toks) and tuple(toks[i:i + 2]) in phrases:
                    res.append("_".join(toks[i:i + 2]))
                    i += 2
                else:
                    res.append(toks[i])
                    i += 1
            new_comps.append(" ".join(res))
        out_c.append("|".join(new_comps))
        out_t.append(" ".join(new_comps))
    return out_c, out_t


def join_phrases(df: pd.DataFrame, phrases: set, n_jobs: int = -1, chunk: int = 200_000) -> None:
    """Rewrite addr_comps / addr_tok in place with joined locality phrases."""
    if not phrases:
        return
    comps = df["addr_comps"].tolist()
    spans = [(i, min(i + chunk, len(comps))) for i in range(0, len(comps), chunk)]
    res = Parallel(n_jobs=n_jobs, batch_size=1)(
        delayed(_join_chunk)(comps[a:b], phrases) for a, b in spans
    )
    df["addr_comps"] = [c for part in res for c in part[0]]
    df["addr_tok"] = [t for part in res for t in part[1]]


# --------------------------------------------------------------------------
# Phonetic skeleton: bridges transliterations ('praivet' ~ 'private',
# 'kelksi knstrksns' ~ 'galaxy constructions', 'sophtveyr' ~ 'software').
# --------------------------------------------------------------------------
_DIGRAPHS = [("ph", "f"), ("bh", "v"), ("sh", "s"), ("ch", "c"), ("th", "t"),
             ("dh", "t"), ("kh", "k"), ("gh", "k"), ("ck", "k"), ("qu", "k"),
             ("x", "ks")]
_SKEL_MAP = str.maketrans({"b": "p", "d": "t", "g": "k", "j": "k", "c": "k",
                           "q": "k", "z": "s", "w": "v"})


def skeleton(tok: str) -> str:
    if not tok or tok.isdigit():
        return tok
    t = tok
    for a, b in _DIGRAPHS:
        t = t.replace(a, b)
    t = t.translate(_SKEL_MAP)
    head, tail = t[0], re.sub(r"[aeiouyh]", "", t[1:])
    out = [head]
    for ch in tail:
        if ch != out[-1]:
            out.append(ch)
    return "".join(out)


def _apply_alias(tokstr: str, amap: dict) -> str:
    if not amap or not tokstr:
        return tokstr
    return " ".join(amap.get(t, t) for t in tokstr.split())


def finalize_frame(df: pd.DataFrame, name_alias: dict, addr_alias: dict,
                   noise_tokens: set) -> pd.DataFrame:
    """Apply learned aliases and derive every string view used downstream."""
    drop = LEGAL_TOKENS | HONORIFIC_TOKENS | set(noise_tokens)
    soft_drop = LEGAL_TOKENS | {"the", "m", "s", "and", "of"}

    name_full, name_core, name_ns, name_skel, legal, alt_core = [], [], [], [], [], []
    for tok, alt in zip(df["name_tok"].values, df["name_alt_tok"].values):
        toks = _apply_alias(tok, name_alias).split()
        core = [t for t in toks if t not in drop]
        if not core:
            core = [t for t in toks if t not in soft_drop] or toks
        atoks = _apply_alias(alt, name_alias).split()
        acore = [t for t in atoks if t not in drop] or core
        name_full.append(" ".join(toks))
        name_core.append(" ".join(core))
        name_ns.append("".join(core))
        name_skel.append(" ".join(skeleton(t) for t in core))
        legal.append(" ".join(sorted({t for t in toks if t in LEGAL_TOKENS})))
        alt_core.append(" ".join(acore))

    addr_full, addr_words, addr_nums, addr_comps = [], [], [], []
    for tok, comps in zip(df["addr_tok"].values, df["addr_comps"].values):
        toks = _apply_alias(tok, addr_alias).split()
        addr_full.append(" ".join(toks))
        addr_words.append(" ".join(t for t in toks if not t.isdigit()))
        addr_nums.append(" ".join(t for t in toks if t.isdigit()))
        addr_comps.append("|".join(_apply_alias(c, addr_alias) for c in comps.split("|")) if comps else "")

    out = df[["entity_id", "country", "raw_name", "raw_addr", "name_nonlatin",
              "name_is_domain", "name_has_dba", "addr_pobox"]].copy()
    out["name_full"] = name_full
    out["name_core"] = name_core
    out["name_ns"] = name_ns
    out["name_skel"] = name_skel
    out["name_legal"] = legal
    out["name_alt"] = alt_core
    out["addr_full"] = addr_full
    out["addr_words"] = addr_words
    out["addr_nums"] = addr_nums
    out["addr_comps"] = addr_comps
    out["addr_empty"] = (out["addr_full"] == "").astype(np.int8)
    return out

In [ ]:
#
# ▌2.5 Learned aliases (`aliases.py`)
#
# Token substitutions mined from training true pairs (e.g. `texas→tx`, `up→uttar_pradesh`, `praivet→pvt`), with a plausibility check, plus noise tokens inserted by the sources (`center`, `services`, ...).

In [ ]:
%%writefile {PKG}/src/ber/aliases.py
"""Data-driven token aliases and noise tokens, learned from training labels only.

For every labelled (S1, S2/S3) true pair we look at the tokens that differ
between the two records. Token ``u`` that repeatedly appears in the S2/S3
record exactly when token ``v`` is missing from it (and ``u`` is rare in the
clean S1 source) is a spelling/abbreviation/transliteration variant of ``v``:
e.g. texas->tx, tg->telangana, praivet->pvt, madras->chennai. S1 is the clean
reference, so its form is the canonical one.

Name tokens that are frequently *inserted* into S2/S3 names without being in
the S1 name (center, services, shri, ...) are learned as noise tokens and are
dropped from the "core" name view.
"""
from __future__ import annotations

import logging
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz.distance import Indel

from .normalize import HONORIFIC_TOKENS, skeleton

log = logging.getLogger(__name__)


def _resolve_chains(amap: dict) -> dict:
    out = {}
    for u, v in amap.items():
        seen = {u}
        while v in amap and v not in seen:
            seen.add(v)
            v = amap[v]
        if v != u:
            out[u] = v
    return out


def _mine(s1_tok: np.ndarray, x_tok: np.ndarray, s1_corpus: pd.Series,
          min_count: int, max_diff: int = 4) -> dict:
    s1_freq = Counter(t for s in s1_corpus.values for t in s.split())
    co: Counter = Counter()
    du: Counter = Counter()
    for a, b in zip(s1_tok, x_tok):
        ta, tb = set(a.split()), set(b.split())
        d1 = [t for t in ta - tb if not t.isdigit()]
        dx = [t for t in tb - ta if not t.isdigit()]
        if not d1 or not dx or len(d1) > max_diff or len(dx) > max_diff:
            continue
        for u in dx:
            du[u] += 1
            for v in d1:
                co[(u, v)] += 1
    cands: dict = {}
    for (u, v), c in co.items():
        if c >= min_count:
            cands.setdefault(u, []).append((c, v))
    amap = {}
    for u, lst in cands.items():
        if u in HONORIFIC_TOKENS:
            continue
        for c, v in sorted(lst, reverse=True):
            if (c / du[u] >= 0.3 and u != v and _plausible(u, v)
                    and s1_freq.get(u, 0) <= 0.05 * s1_freq.get(v, 0)):
                amap[u] = v
                break
    return _resolve_chains(amap)


def _plausible(u: str, v: str) -> bool:
    """An alias must look like its target: same phonetic onset or similar spelling.

    Rejects co-occurrence artefacts such as 'mr'->'ltd' or 'li'->'pvt' while
    keeping abbreviations (tg->telangana), transliterations (phud->food) and
    typos (lnc->inc).
    """
    su, sv = skeleton(u.replace("_", "")), skeleton(v.replace("_", ""))
    return (su[:1] == sv[:1]) or u[:1] == v[:1] or Indel.normalized_similarity(u, v) >= 0.5


def _noise_tokens(s1_tok: np.ndarray, x_tok: np.ndarray, min_count: int = 200,
                  min_rate: float = 0.6) -> set:
    ins: Counter = Counter()
    tot: Counter = Counter()
    for a, b in zip(s1_tok, x_tok):
        ta = set(a.split())
        for t in set(b.split()):
            tot[t] += 1
            if t not in ta:
                ins[t] += 1
    return {t for t, n in tot.items()
            if n >= min_count and ins[t] / n >= min_rate and not t.isdigit()}


def learn_aliases(s1n: pd.DataFrame, xn: pd.DataFrame, pairs: pd.DataFrame,
                  min_count: int, max_pairs: int = 3_000_000, seed: int = 0):
    """pairs: DataFrame with columns s1_id, x_id (true matches, training split)."""
    if len(pairs) > max_pairs:
        pairs = pairs.sample(max_pairs, random_state=seed)
    s1_pos = pd.Series(np.arange(len(s1n)), index=s1n["entity_id"].values)
    x_pos = pd.Series(np.arange(len(xn)), index=xn["entity_id"].values)
    ok = pairs["s1_id"].isin(s1_pos.index) & pairs["x_id"].isin(x_pos.index)
    pairs = pairs[ok]
    ia = s1_pos.loc[pairs["s1_id"].values].values
    ib = x_pos.loc[pairs["x_id"].values].values

    name_s1 = s1n["name_tok"].values[ia]
    name_x = xn["name_tok"].values[ib]
    addr_s1 = s1n["addr_tok"].values[ia]
    addr_x = xn["addr_tok"].values[ib]

    name_alias = _mine(name_s1, name_x, s1n["name_tok"], min_count)
    addr_alias = _mine(addr_s1, addr_x, s1n["addr_tok"], min_count)
    noise = _noise_tokens(np.array([" ".join(name_alias.get(t, t) for t in s.split()) for s in name_s1]),
                          np.array([" ".join(name_alias.get(t, t) for t in s.split()) for s in name_x]))
    log.info("learned %d name aliases, %d address aliases, %d noise tokens",
             len(name_alias), len(addr_alias), len(noise))
    return name_alias, addr_alias, noise

In [ ]:
#
# ▌2.6 Blocking (`blocking.py`)
#
# Five TF-IDF views per (country, source) block, forward top-k per entity and reverse top-k per record, then a learned LightGBM pruner applied block by block (memory-safe).

In [ ]:
%%writefile {PKG}/src/ber/blocking.py
"""Multi-view sparse candidate generation (blocking).

Per (country, target source) block, five TF-IDF "views" are built on the
records of that block only (IDF is a property of the corpus being matched, so
the same procedure adapts to an unseen country such as France):

  name_tok   core-name word tokens + the no-space name (typo-free matches,
             word-order transpositions, domain-style names)
  name_char  character 4-grams of the no-space core name (typos, OCR noise,
             partial transliterations)
  name_skel  phonetic-skeleton tokens of the core name (Indic-script names
             transliterated to Latin: 'teknalaji' ~ 'technology')
  addr       address words, house numbers and (number x word) conjunctions
             inside one address component (robust to component reordering)
  name_addr  (core-name token x address token) conjunctions: very specific
             keys that survive even when every single token is common

Tokens whose document frequency exceeds a cap are dropped (they carry no
blocking signal and make the sparse join quadratic). For every view we take
the forward top-k S2/S3 records per S1 entity and the reverse top-k S1
entities per S2/S3 record, and union everything (one block at a time, so a
run can checkpoint and resume block by block).

A sixth, dense view (dense.py) adds the exact embedding-cosine top-k in both
directions when record embeddings are given; it has no frequency cap. Every
union pair carries ``via``, a bitmask of the (view, direction) lists that
retrieved it (VIA_BIT), and ``cos_dense``.

The union is then pruned to ``keep_per_source`` candidates per (S1, source)
by a small learned ranker on the view cosines and the retrieval bitmask,
trained on the union rows of a sample of training entities - exactly the
distribution it prunes. Records whose single best S1 is this entity are always
kept. The pruned set is exactly what the matching model scores and what is
written to candidate_pairs.tsv.
"""
from __future__ import annotations

import logging
import re

import lightgbm as lgb
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer

from . import dense
from .config import BlockingConfig

log = logging.getLogger(__name__)

VIEWS = ("name_tok", "name_char", "name_skel", "addr", "name_addr")     # sparse views
SPARSE_COS = [f"cos_{v}" for v in VIEWS]
COS_COLS = SPARSE_COS + ["cos_dense"]            # cos_dense is 0 when the dense view is off
# ``via`` bits: forward / reverse list of each view that retrieved the pair
VIA_BIT = {**{(v, d): 1 << (2 * i + j) for i, v in enumerate(VIEWS) for j, d in enumerate(("fwd", "rev"))},
           ("dense", "fwd"): 1 << 10, ("dense", "rev"): 1 << 11}
VIA_SPARSE = (1 << 10) - 1
VIA_DENSE = (1 << 10) | (1 << 11)
_POPCOUNT = np.array([bin(i).count("1") for i in range(1 << 12)], dtype=np.uint8)


def n_via(via: np.ndarray, mask: int = VIA_SPARSE | VIA_DENSE) -> np.ndarray:
    """Number of retrieval lists (of ``mask``) that produced each pair."""
    return _POPCOUNT[np.asarray(via, dtype=np.int64) & mask]


# --------------------------------------------------------------------------
# View documents
# --------------------------------------------------------------------------
def _doc_name_tok(df: pd.DataFrame) -> list[str]:
    out = []
    for core, ns, alt in zip(df["name_core"].values, df["name_ns"].values, df["name_alt"].values):
        toks = set(core.split()) | set(alt.split())
        if ns:
            toks.add("ns:" + ns)
        out.append(" ".join(sorted(toks)))   # sorted: identical matrices on every run
    return out


def _doc_name_skel(df: pd.DataFrame) -> list[str]:
    out = []
    for sk in df["name_skel"].values:
        toks = {t for t in sk.split() if len(t) >= 2}
        if sk:
            toks.add("sk:" + sk.replace(" ", ""))
        out.append(" ".join(sorted(toks)))   # sorted: identical matrices on every run
    return out


def _doc_name_char(df: pd.DataFrame) -> list[str]:
    return [f" {ns} " if ns else "" for ns in df["name_ns"].values]


def _doc_addr(df: pd.DataFrame) -> list[str]:
    out = []
    for comps in df["addr_comps"].values:
        toks = set()
        for comp in comps.split("|") if comps else ():
            ct = comp.split()
            nums = [t for t in ct if t.isdigit()]
            words = [t for t in ct if not t.isdigit() and len(t) >= 2]
            toks.update(words)
            toks.update("n:" + n for n in nums)
            for n in nums[:3]:
                for w in words:
                    if len(w) >= 3:
                        toks.add(f"{n}_{w}")
        out.append(" ".join(sorted(toks)))   # sorted: identical matrices on every run
    return out


def _doc_name_addr(df: pd.DataFrame) -> list[str]:
    out = []
    for core, nums, words in zip(df["name_core"].values, df["addr_nums"].values,
                                 df["addr_words"].values):
        ntoks = [t for t in dict.fromkeys(core.split()) if len(t) >= 2][:4]
        akeys = list(dict.fromkeys(nums.split()))[:4] + \
            [w for w in dict.fromkeys(words.split()) if len(w) >= 4][:8]
        out.append(" ".join(f"{n}^{a}" for n in ntoks for a in akeys))
    return out


_DOC_BUILDERS = {
    "name_tok": _doc_name_tok, "name_char": _doc_name_char, "name_skel": _doc_name_skel,
    "addr": _doc_addr, "name_addr": _doc_name_addr,
}


def _split(s: str) -> list[str]:
    return s.split()


def _vectorizer(view: str, max_df: int) -> TfidfVectorizer:
    if view == "name_char":
        return TfidfVectorizer(analyzer="char", ngram_range=(4, 4), lowercase=False,
                               min_df=2, max_df=max_df, sublinear_tf=True,
                               dtype=np.float32)
    return TfidfVectorizer(analyzer=_split, lowercase=False, min_df=2,
                           max_df=max_df, sublinear_tf=True, dtype=np.float32)


# Columns each view's document builder reads (only these are shipped to workers).
_VIEW_COLS = {"name_tok": ["name_core", "name_ns", "name_alt"], "name_char": ["name_ns"],
              "name_skel": ["name_skel"], "addr": ["addr_comps"],
              "name_addr": ["name_core", "addr_nums", "addr_words"]}


def _fit_view(view: str, s1: pd.DataFrame, xt: pd.DataFrame, max_df: int):
    """(A, B) L2-normalised TF-IDF matrices of one view; vocabulary / IDF fitted on S1 + target."""
    da = _DOC_BUILDERS[view](s1)
    db = _DOC_BUILDERS[view](xt)
    # Absolute max_df must stay above min_df for tiny blocks.
    vec = _vectorizer(view, max(max_df, 3))
    try:
        M = vec.fit_transform(da + db).tocsr()   # identical to fit() then transform(), one tokenisation
    except ValueError:  # empty vocabulary in a tiny block
        return (sp.csr_matrix((len(da), 1), dtype=np.float32),
                sp.csr_matrix((len(db), 1), dtype=np.float32))
    return M[:len(da)], M[len(da):]


def build_view_matrices(s1: pd.DataFrame, xt: pd.DataFrame, cfg: BlockingConfig, n_jobs: int = -1):
    """Return {view: (A, B)} for one block; the five views are built in parallel processes."""
    caps = {"name_tok": cfg.max_df_name_tok, "name_char": cfg.max_df_name_char,
            "name_skel": cfg.max_df_name_skel, "addr": cfg.max_df_addr,
            "name_addr": cfg.max_df_name_addr}
    workers = len(VIEWS) if n_jobs == -1 else max(1, min(len(VIEWS), n_jobs))
    res = Parallel(n_jobs=workers)(
        delayed(_fit_view)(v, s1[_VIEW_COLS[v]], xt[_VIEW_COLS[v]], caps[v]) for v in VIEWS)
    return dict(zip(VIEWS, res))


# --------------------------------------------------------------------------
# Sparse top-k join
# --------------------------------------------------------------------------
def _topk_chunk(A_chunk: sp.csr_matrix, BT: sp.csc_matrix, k: int, row_offset: int):
    M = (A_chunk @ BT).tocsr()
    if M.nnz == 0:
        e = np.empty(0, dtype=np.int64)
        return e, e
    indptr = M.indptr
    rows = np.repeat(np.arange(M.shape[0]), np.diff(indptr))
    order = np.lexsort((-M.data, rows))
    rank = np.arange(len(order)) - indptr[rows[order]]
    keep = order[rank < k]
    return rows[keep].astype(np.int64) + row_offset, M.indices[keep].astype(np.int64)


def topk_join(A: sp.csr_matrix, B: sp.csr_matrix, k: int, chunk_rows: int, n_jobs: int):
    if A.shape[0] == 0 or B.shape[0] == 0 or k <= 0:
        e = np.empty(0, dtype=np.int64)
        return e, e
    BT = B.T.tocsc()
    spans = [(i, min(i + chunk_rows, A.shape[0])) for i in range(0, A.shape[0], chunk_rows)]
    res = Parallel(n_jobs=n_jobs, batch_size=1)(
        delayed(_topk_chunk)(A[a:b], BT, k, a) for a, b in spans
    )
    ra = np.concatenate([r[0] for r in res]) if res else np.empty(0, np.int64)
    rb = np.concatenate([r[1] for r in res]) if res else np.empty(0, np.int64)
    return ra, rb


def rowwise_cosine(A: sp.csr_matrix, B: sp.csr_matrix, ia: np.ndarray, ib: np.ndarray,
                   chunk: int = 1_000_000) -> np.ndarray:
    out = np.zeros(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        e = min(s + chunk, len(ia))
        out[s:e] = np.asarray(A[ia[s:e]].multiply(B[ib[s:e]]).sum(axis=1)).ravel()
    return out


# --------------------------------------------------------------------------
# Candidate generation, one (country, target source) block at a time
# --------------------------------------------------------------------------
def block_keys(S1: pd.DataFrame, X: pd.DataFrame) -> list[tuple[str, int, str]]:
    """(country, source, file tag) of every non-empty block, in a fixed order."""
    keys = []
    countries = sorted(set(S1["country"].unique()) | set(X["country"].unique()))
    for i, country in enumerate(countries):
        if not (S1["country"].values == country).any():
            continue
        safe = re.sub(r"[^A-Za-z0-9]+", "_", str(country)).strip("_") or "none"
        for src in (2, 3):
            if ((X["country"].values == country) & (X["src"].values == src)).any():
                keys.append((country, src, f"{i:03d}_{safe}_S{src}"))
    return keys


def block_union(S1: pd.DataFrame, X: pd.DataFrame, country: str, src: int,
                cfg: BlockingConfig, n_jobs: int = -1, emb=None) -> pd.DataFrame:
    """Unpruned candidate union of one block, for every S1 entity of the country.

    ``emb``: optional (S1 embeddings, S2/S3 embeddings), row-aligned with S1 / X,
    which adds the dense view.
    Columns: s1_idx, x_idx, src, cos_* (five sparse view cosines + cos_dense), via.
    """
    ks = {"name_tok": cfg.k_name_tok, "name_char": cfg.k_name_char,
          "name_skel": cfg.k_name_skel, "addr": cfg.k_addr, "name_addr": cfg.k_name_addr}
    s1_pos = np.flatnonzero(S1["country"].values == country)
    x_pos = np.flatnonzero((X["country"].values == country) & (X["src"].values == src))
    mats = build_view_matrices(S1.iloc[s1_pos], X.iloc[x_pos], cfg, n_jobs)
    ra_all, rb_all, bits = [], [], []

    def add(ra, rb, bit):
        ra_all.append(ra)
        rb_all.append(rb)
        bits.append(np.full(len(ra), bit, dtype=np.uint16))

    for view in VIEWS:
        A, B = mats[view]
        fa, fb = topk_join(A, B, ks[view], cfg.chunk_rows, n_jobs)
        rb_, ra_ = topk_join(B, A, cfg.k_reverse, cfg.chunk_rows, n_jobs)
        add(fa, fb, VIA_BIT[(view, "fwd")])
        add(ra_, rb_, VIA_BIT[(view, "rev")])
    E1 = EX = None
    if emb is not None:
        E1, EX = emb[0][s1_pos], emb[1][x_pos]
        fa, fb = dense.topk_dense(E1, EX, cfg.k_dense)
        rb_, ra_ = dense.topk_dense(EX, E1, cfg.k_dense_reverse)
        add(fa, fb, VIA_BIT[("dense", "fwd")])
        add(ra_, rb_, VIA_BIT[("dense", "rev")])
    # union of every list, OR-ing the retrieval bits of duplicate pairs
    key_all = np.concatenate(ra_all) * len(x_pos) + np.concatenate(rb_all)
    bits_all = np.concatenate(bits)
    del ra_all, rb_all, bits
    order = np.argsort(key_all, kind="stable")
    key_all, bits_all = key_all[order], bits_all[order]
    del order
    start = np.r_[0, np.flatnonzero(key_all[1:] != key_all[:-1]) + 1] if len(key_all) else np.zeros(0, np.int64)
    key = key_all[start]
    via = np.bitwise_or.reduceat(bits_all, start) if len(start) else np.zeros(0, np.uint16)
    del key_all, bits_all
    la, lb = key // len(x_pos), key % len(x_pos)
    block = pd.DataFrame({
        "s1_idx": s1_pos[la].astype(np.int32),
        "x_idx": x_pos[lb].astype(np.int32),
        "src": np.int8(src),
    })
    for view in VIEWS:
        A, B = mats[view]
        block[f"cos_{view}"] = rowwise_cosine(A, B, la, lb)
    block["cos_dense"] = dense.rowwise_cos(E1, EX, la, lb) if E1 is not None else np.float32(0.0)
    block["via"] = via.astype(np.uint16)
    log.info("blocking %s / S%d: %d S1 x %d records -> %d union pairs (%d from the dense view only)",
             country, src, len(s1_pos), len(x_pos), len(block), int(((via & VIA_SPARSE) == 0).sum()))
    return block


# --------------------------------------------------------------------------
# Learned pruning
# --------------------------------------------------------------------------
def _ranker_features(cand: pd.DataFrame, X: pd.DataFrame) -> np.ndarray:
    # the first nine columns are the layout of pruners trained before the dense view
    c = cand[SPARSE_COS].values.astype(np.float32)
    src = X["src"].values[cand["x_idx"].values].astype(np.float32)[:, None]
    empty = X["addr_empty"].values[cand["x_idx"].values].astype(np.float32)[:, None]
    via = cand["via"].values
    return np.hstack([c, c.max(1, keepdims=True), c.sum(1, keepdims=True), src, empty,
                      cand[["cos_dense"]].values.astype(np.float32),
                      n_via(via, VIA_SPARSE).astype(np.float32)[:, None],
                      n_via(via, VIA_DENSE).astype(np.float32)[:, None]])


def fit_ranker(cand: pd.DataFrame, X: pd.DataFrame, labels: np.ndarray, seed: int = 0):
    feats = _ranker_features(cand, X)
    params = dict(objective="binary", learning_rate=0.1, num_leaves=63,
                  min_data_in_leaf=500, verbose=-1, seed=seed)
    return lgb.train(params, lgb.Dataset(feats, labels.astype(np.float32)), num_boost_round=300)


def prune_candidates(cand: pd.DataFrame, X: pd.DataFrame, ranker, keep: int) -> pd.DataFrame:
    """Keep the top ``keep`` per (S1, source) plus every record's best S1.

    Called per (country, source) block: every record belongs to exactly one
    block, so "best S1 for this record" is global.
    """
    cand = cand.copy()
    feats = _ranker_features(cand, X)
    cand["block_score"] = ranker.predict(feats[:, :ranker.num_feature()]).astype(np.float32)
    del feats
    cand["src"] = X["src"].values[cand["x_idx"].values].astype(np.int8)
    cand = cand.sort_values(["s1_idx", "src", "block_score"], ascending=[True, True, False])
    rank = cand.groupby(["s1_idx", "src"], sort=False).cumcount()
    best_for_x = cand.groupby("x_idx")["block_score"].transform("max") == cand["block_score"]
    return cand[(rank < keep) | best_for_x].reset_index(drop=True)

In [ ]:
#
# ▌2.7 Features (`features.py`)
#
# Pairwise similarities (rapidfuzz, vectorised), house-number and locality agreement, genericness, blocking-rank features, and stage-2 context features.

In [ ]:
%%writefile {PKG}/src/ber/features.py
"""Pairwise and contextual features for candidate (S1, S2/S3) pairs.

Stage-1 features describe one pair in isolation (name / address similarity on
several normalised views, house-number agreement, locality agreement, name
genericness, blocking scores). Stage-2 context features describe the pair
relative to its competitors: the other candidates of the same S1 entity, the
other S1 entities claiming the same S2/S3 record, and the agreement of the
record with the S1 entity's most probable matches ("anchors").

All features are country-agnostic similarities / counts; no country one-hot.
"""
from __future__ import annotations

import logging
from collections import Counter

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein, Prefix
from rapidfuzz.process import cpdist

from .blocking import COS_COLS, VIA_DENSE, VIA_SPARSE, n_via

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# Record-level statistics (computed on the corpus being matched)
# --------------------------------------------------------------------------
def add_record_stats(S1: pd.DataFrame, X: pd.DataFrame, geo_min_count: int) -> set:
    """Adds genericness columns in place; returns the locality vocabulary."""
    both = pd.concat([S1[["country", "name_core"]], X[["country", "name_core"]]], ignore_index=True)
    key = both["country"] + "|" + both["name_core"]
    cnt = key.value_counts()
    S1["name_dup"] = np.log1p((S1["country"] + "|" + S1["name_core"]).map(cnt).fillna(0).values).astype(np.float32)
    X["name_dup"] = np.log1p((X["country"] + "|" + X["name_core"]).map(cnt).fillna(0).values).astype(np.float32)

    toks = both.assign(t=both["name_core"].str.split()).explode("t").dropna(subset=["t"])
    tdf = (toks["country"] + "|" + toks["t"]).value_counts()
    for frame in (S1, X):
        ex = frame[["country", "name_core"]].assign(t=frame["name_core"].str.split()).explode("t")
        d = (ex["country"] + "|" + ex["t"].fillna("")).map(tdf).fillna(1.0)
        g = d.groupby(level=0)
        frame["name_tok_mindf"] = np.log1p(g.min().reindex(frame.index).fillna(1.0).values).astype(np.float32)
        frame["name_tok_maxdf"] = np.log1p(g.max().reindex(frame.index).fillna(1.0).values).astype(np.float32)

    # Locality vocabulary: digit-free address components that recur in S1.
    comps = Counter()
    for country, cs in zip(S1["country"].values, S1["addr_comps"].values):
        for c in cs.split("|") if cs else ():
            if c and not any(ch.isdigit() for ch in c):
                comps[(country, c)] += 1
    geo = {k for k, v in comps.items() if v >= geo_min_count}
    log.info("locality vocabulary: %d components", len(geo))
    return geo


# --------------------------------------------------------------------------
# Python-loop features (numbers / components), parallel over chunks
# --------------------------------------------------------------------------
def _num_comp_chunk(n1, n2, c1, c2, ctry, geo):
    out = np.zeros((len(n1), 15), dtype=np.float32)
    for i in range(len(n1)):
        a = n1[i].split()
        b = n2[i].split()
        sa, sb = set(a), set(b)
        inter = len(sa & sb)
        uni = len(sa | sb)
        best = 0.0
        if a and b:
            for x in sa:
                for y in sb:
                    if len(x) >= 2 or len(y) >= 2:
                        s = Levenshtein.normalized_similarity(x, y)
                        if s > best:
                            best = s
        out[i, 0] = len(sa)
        out[i, 1] = len(sb)
        out[i, 2] = inter
        out[i, 3] = inter / uni if uni else np.nan
        out[i, 4] = float(a[0] == b[0]) if a and b else np.nan
        out[i, 5] = float(a[0] in sb) if a and b else np.nan
        out[i, 6] = best if a and b else np.nan
        big_a = {x for x in sa if len(x) >= 3}
        out[i, 7] = float(bool(big_a & sb)) if big_a and b else np.nan

        ca = [c for c in c1[i].split("|") if c]
        cb = [c for c in c2[i].split("|") if c]
        ga = {c for c in ca if (ctry[i], c) in geo}
        gb = {c for c in cb if (ctry[i], c) in geo}
        gi = len(ga & gb)
        out[i, 8] = len(ga)
        out[i, 9] = len(gb)
        out[i, 10] = gi
        out[i, 11] = float(gi == 0) if ga and gb else np.nan
        sca, scb = set(ca), set(cb)
        cu = len(sca | scb)
        out[i, 12] = len(sca & scb) / cu if cu else np.nan
        out[i, 13] = float(ga <= gb or gb <= ga) if ga and gb else np.nan
        if a and b and len(a[0]) <= 9 and len(b[0]) <= 9:
            out[i, 14] = np.log1p(abs(int(a[0]) - int(b[0])))
        else:
            out[i, 14] = np.nan
    return out


_NUM_COMP_COLS = ["num_n1", "num_n2", "num_inter", "num_jacc", "num_first_eq",
                  "num_first_in", "num_best_sim", "num_big_overlap", "geo_n1",
                  "geo_n2", "geo_inter", "geo_conflict", "comp_jacc", "geo_subset",
                  "num_first_logdiff"]


def _street_part(comps: np.ndarray, ctry: np.ndarray, geo: set) -> list[str]:
    out = []
    for cs, c in zip(comps, ctry):
        out.append(" ".join(p for p in cs.split("|") if p and (c, p) not in geo) if cs else "")
    return out


def _sim(q, c, scorer, **kw) -> np.ndarray:
    return cpdist(q, c, scorer=scorer, workers=-1, dtype=np.float32, **kw)


# --------------------------------------------------------------------------
# Stage-1 feature matrix
# --------------------------------------------------------------------------
def pair_features(cand: pd.DataFrame, S1: pd.DataFrame, X: pd.DataFrame, geo: set,
                  n_jobs: int = -1, chunk: int = 2_000_000) -> pd.DataFrame:
    parts = []
    for s in range(0, len(cand), chunk):
        parts.append(_pair_features_chunk(cand.iloc[s:s + chunk], S1, X, geo, n_jobs))
        log.info("features: %d / %d pairs", min(s + chunk, len(cand)), len(cand))
    feats = pd.concat(parts, ignore_index=True)
    feats.index = cand.index
    return feats


def _pair_features_chunk(cand, S1, X, geo, n_jobs) -> pd.DataFrame:
    ia = cand["s1_idx"].values
    ib = cand["x_idx"].values
    g1 = lambda col: S1[col].values[ia]  # noqa: E731
    g2 = lambda col: X[col].values[ib]  # noqa: E731
    f = {}
    for c in COS_COLS + ["block_score"]:
        f[c] = cand[c].values.astype(np.float32)
    f["src"] = cand["src"].values.astype(np.float32)
    # how many retrieval lists found the pair (agreement of independent views)
    f["via_n_sparse"] = n_via(cand["via"].values, VIA_SPARSE).astype(np.float32)
    f["via_n_dense"] = n_via(cand["via"].values, VIA_DENSE).astype(np.float32)

    a_full, b_full = list(g1("name_full")), list(g2("name_full"))
    a_core, b_core = list(g1("name_core")), list(g2("name_core"))
    a_ns, b_ns = list(g1("name_ns")), list(g2("name_ns"))
    a_sk, b_sk = list(g1("name_skel")), list(g2("name_skel"))
    b_alt = list(g2("name_alt"))

    f["nm_full_ratio"] = _sim(a_full, b_full, fuzz.ratio) / 100
    f["nm_full_tset"] = _sim(a_full, b_full, fuzz.token_set_ratio) / 100
    f["nm_full_partial"] = _sim(a_full, b_full, fuzz.partial_ratio) / 100
    f["nm_core_ratio"] = _sim(a_core, b_core, fuzz.ratio) / 100
    f["nm_core_tset"] = _sim(a_core, b_core, fuzz.token_set_ratio) / 100
    f["nm_core_tsort"] = _sim(a_core, b_core, fuzz.token_sort_ratio) / 100
    f["nm_core_jw"] = _sim(a_core, b_core, JaroWinkler.normalized_similarity)
    f["nm_ns_lev"] = _sim(a_ns, b_ns, Levenshtein.normalized_similarity)
    f["nm_ns_partial"] = _sim(a_ns, b_ns, fuzz.partial_ratio) / 100
    f["nm_ns_prefix"] = _sim(a_ns, b_ns, Prefix.normalized_similarity)
    f["nm_ns_eq"] = (np.array(a_ns, dtype=object) == np.array(b_ns, dtype=object)).astype(np.float32)
    f["nm_skel_ratio"] = _sim(a_sk, b_sk, fuzz.ratio) / 100
    f["nm_skel_tset"] = _sim(a_sk, b_sk, fuzz.token_set_ratio) / 100
    f["nm_alt_tset"] = _sim(a_core, b_alt, fuzz.token_set_ratio) / 100
    f["nm_len1"] = np.array([len(x) for x in a_ns], dtype=np.float32)
    f["nm_len2"] = np.array([len(x) for x in b_ns], dtype=np.float32)
    f["nm_ntok1"] = np.array([x.count(" ") + 1 if x else 0 for x in a_core], dtype=np.float32)
    f["nm_ntok2"] = np.array([x.count(" ") + 1 if x else 0 for x in b_core], dtype=np.float32)

    la, lb = g1("name_legal"), g2("name_legal")
    both = (la != "") & (lb != "")
    f["legal_both"] = both.astype(np.float32)
    f["legal_eq"] = np.where(both, (la == lb).astype(np.float32), np.nan).astype(np.float32)
    disjoint = np.array([not (set(x.split()) & set(y.split())) for x, y in zip(la, lb)])
    f["legal_conflict"] = np.where(both, disjoint.astype(np.float32), np.nan).astype(np.float32)

    for col in ("name_nonlatin", "name_is_domain", "name_has_dba", "addr_empty", "addr_pobox"):
        f[f"x_{col}"] = g2(col).astype(np.float32)
    for col in ("name_dup", "name_tok_mindf", "name_tok_maxdf"):
        f[f"s1_{col}"] = g1(col)
        f[f"x_{col}"] = g2(col)

    a_ad, b_ad = list(g1("addr_full")), list(g2("addr_full"))
    a_aw, b_aw = list(g1("addr_words")), list(g2("addr_words"))
    f["ad_full_ratio"] = _sim(a_ad, b_ad, fuzz.ratio) / 100
    f["ad_full_tset"] = _sim(a_ad, b_ad, fuzz.token_set_ratio) / 100
    f["ad_full_tsort"] = _sim(a_ad, b_ad, fuzz.token_sort_ratio) / 100
    f["ad_words_tset"] = _sim(a_aw, b_aw, fuzz.token_set_ratio) / 100
    f["ad_words_partial"] = _sim(a_aw, b_aw, fuzz.partial_token_set_ratio) / 100
    ctry = g1("country")
    sa = _street_part(g1("addr_comps"), ctry, geo)
    sb = _street_part(g2("addr_comps"), ctry, geo)
    f["ad_street_tset"] = _sim(sa, sb, fuzz.token_set_ratio) / 100
    f["ad_street_ratio"] = _sim(sa, sb, fuzz.ratio) / 100
    f["ad_len1"] = np.array([len(x) for x in a_ad], dtype=np.float32)
    f["ad_len2"] = np.array([len(x) for x in b_ad], dtype=np.float32)

    n1, n2 = g1("addr_nums"), g2("addr_nums")
    c1, c2 = g1("addr_comps"), g2("addr_comps")
    step = 200_000
    spans = [(i, min(i + step, len(cand))) for i in range(0, len(cand), step)]
    res = Parallel(n_jobs=n_jobs, batch_size=1)(
        delayed(_num_comp_chunk)(n1[a:b], n2[a:b], c1[a:b], c2[a:b], ctry[a:b], geo)
        for a, b in spans
    )
    nc = np.vstack(res) if res else np.zeros((0, len(_NUM_COMP_COLS)), np.float32)
    for j, col in enumerate(_NUM_COMP_COLS):
        f[col] = nc[:, j]

    # Joint name+address evidence.
    f["joint_min"] = np.minimum(f["nm_core_tset"], f["ad_full_tset"])
    f["joint_prod"] = f["nm_core_tset"] * f["ad_full_tset"]
    return pd.DataFrame(f)


def group_features(cand: pd.DataFrame) -> pd.DataFrame:
    """Rank / competition features on the blocking score, over ALL candidates."""
    bs = cand["block_score"]
    g_s1s = cand.groupby(["s1_idx", "src"])["block_score"]
    g_x = cand.groupby("x_idx")["block_score"]
    dn = cand["cos_dense"]
    d_s1s = cand.groupby(["s1_idx", "src"])["cos_dense"]
    d_x = cand.groupby("x_idx")["cos_dense"]
    out = pd.DataFrame({
        "blk_rank_s1src": g_s1s.rank(ascending=False, method="first").values,
        "blk_gap_s1src": (bs - g_s1s.transform("max")).values,
        "blk_rank_x": g_x.rank(ascending=False, method="first").values,
        "blk_gap_x": (bs - g_x.transform("max")).values,
        "n_claim_x": cand.groupby("x_idx")["s1_idx"].transform("size").values,
        "n_cand_s1": cand.groupby("s1_idx")["x_idx"].transform("size").values,
        # the same competition view on the embedding cosine (ties -> equal rank, so a
        # constant cos_dense when the dense view is off gives a constant feature)
        "dns_rank_s1src": d_s1s.rank(ascending=False, method="min").values,
        "dns_gap_s1src": (dn - d_s1s.transform("max")).values,
        "dns_rank_x": d_x.rank(ascending=False, method="min").values,
        "dns_gap_x": (dn - d_x.transform("max")).values,
    })
    return out.astype(np.float32)


def stage1_matrix(cand: pd.DataFrame, rows: np.ndarray, S1: pd.DataFrame, X: pd.DataFrame,
                  geo: set, grp: pd.DataFrame, n_jobs: int = -1,
                  cols: list | None = None) -> pd.DataFrame:
    """Stage-1 features for candidate rows ``rows`` (pairwise + group columns)."""
    F = pair_features(cand.iloc[rows], S1, X, geo, n_jobs).reset_index(drop=True)
    for c in grp.columns:
        F[c] = grp[c].values[rows]
    if cols is not None:
        F = F[cols]
    return F


# --------------------------------------------------------------------------
# Stage-2 context features from stage-1 probabilities
# --------------------------------------------------------------------------
def _second_largest(keys: np.ndarray, p: np.ndarray) -> np.ndarray:
    order = np.lexsort((-p, keys))
    ks, ps = keys[order], p[order]
    first = np.r_[True, ks[1:] != ks[:-1]]
    rank = np.arange(len(ks)) - np.maximum.accumulate(np.where(first, np.arange(len(ks)), 0))
    sec = pd.Series(ps[rank == 1], index=ks[rank == 1])
    return pd.Series(keys).map(sec).fillna(0.0).values


def _top2_rows(keys: np.ndarray, p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """For every row, the row index of the highest and of the second-highest ``p``
    among the rows sharing its key (-1 when the key has a single row)."""
    n = len(keys)
    top1 = np.full(n, -1, np.int64)
    top2 = np.full(n, -1, np.int64)
    if n == 0:
        return top1, top2
    order = np.lexsort((-p, keys))
    ks = keys[order]
    first = np.r_[True, ks[1:] != ks[:-1]]
    start = np.maximum.accumulate(np.where(first, np.arange(n), 0))
    grp = np.cumsum(first) - 1
    size = np.bincount(grp)[grp]
    top1[order] = order[start]
    top2[order] = np.where(size >= 2, order[np.minimum(start + 1, n - 1)], -1)
    return top1, top2


def _competitor_row(keys: np.ndarray, p: np.ndarray) -> np.ndarray:
    """For every row, the most probable OTHER row with the same key (-1 when none)."""
    top1, top2 = _top2_rows(keys, p)
    return np.where(top1 == np.arange(len(keys)), top2, top1)


def _ownership_features(df: pd.DataFrame, S1: pd.DataFrame | None) -> dict:
    """Collective "who owns this record" evidence, over ALL candidates.

    Most remaining errors are records (often with an empty address) that several
    look-alike S1 entities compete for. Besides the pair's own evidence, the owner
    is revealed by the competitors: a rival that already has a strong match in the
    same source, or a rival with exactly the same name (then the record is a coin
    flip, and a precision-weighted metric prefers leaving it out).
    """
    p = df["p"].values.astype(np.float64)
    s1 = df["s1"].values.astype(np.int64)
    x = df["x"].values.astype(np.int64)
    src = df["src"].values.astype(np.int64)
    out = {}
    # the entity's best OTHER candidate in the same source, and its best one in the other source
    k_s1src = s1 * 8 + src
    o = _competitor_row(k_s1src, p)
    own_other = np.where(o >= 0, p[np.maximum(o, 0)], 0.0)
    out["own_max_other_src"] = own_other
    gmax = pd.Series(p).groupby(k_s1src).max()
    out["own_max_other_source"] = pd.Series(s1 * 8 + (5 - src)).map(gmax).fillna(0.0).values
    # the record's strongest rival claimant, and how well that rival is already served
    c = _competitor_row(x, p)
    has_c = c >= 0
    ci = np.maximum(c, 0)
    out["rival_p"] = np.where(has_c, p[ci], np.nan)
    out["rival_own_max_other_src"] = np.where(has_c, own_other[ci], np.nan)
    if S1 is not None:
        key = pd.Series(S1["country"].values).astype(str) + "|" + pd.Series(S1["name_core"].values).astype(str)
        code_s1, uniq = pd.factorize(key)
        n_same = np.bincount(code_s1, minlength=len(uniq))
        nc = code_s1[s1].astype(np.int64)
        out["s1_same_name_entities"] = np.log1p(n_same[nc]).astype(np.float64)
        out["rival_same_name"] = np.where(has_c, (nc[ci] == nc).astype(np.float64), np.nan)
        # claimants of this record carrying exactly this entity's name, and their probability mass
        _, inv, cnt = np.unique(x * (len(uniq) + 1) + nc, return_inverse=True, return_counts=True)
        inv = inv.ravel()
        out["x_same_name_claimants"] = cnt[inv].astype(np.float64)
        out["x_same_name_rival_p"] = np.bincount(inv, weights=p)[inv] - p
    return out


def context_features(cand: pd.DataFrame, p1: np.ndarray, X: pd.DataFrame,
                     n_anchors: int, rows: np.ndarray | None = None,
                     S1: pd.DataFrame | None = None) -> pd.DataFrame:
    """Context features for candidate ``rows`` (default: all rows).

    Aggregates are computed over ALL candidates (every competing S1 / record),
    anchors only for the requested rows. ``S1`` adds the same-name rivalry features.
    """
    df = pd.DataFrame({"s1": cand["s1_idx"].values, "x": cand["x_idx"].values,
                       "src": cand["src"].values, "p": p1.astype(np.float32)})
    if rows is None:
        rows = np.arange(len(df))
    g_s1 = df.groupby("s1")["p"]
    g_s1s = df.groupby(["s1", "src"])["p"]
    g_x = df.groupby("x")["p"]
    hi = (df["p"] > 0.5).astype(np.float32)
    out = {"p1": df["p"].values[rows]}
    out["p1_rank_s1"] = g_s1.rank(ascending=False, method="first").values[rows]
    out["p1_rank_s1src"] = g_s1s.rank(ascending=False, method="first").values[rows]
    out["p1_gap_s1src"] = (df["p"] - g_s1s.transform("max")).values[rows]
    out["p1_sum_s1"] = g_s1.transform("sum").values[rows]
    out["p1_sum_s1src"] = g_s1s.transform("sum").values[rows]
    out["p1_nhi_s1"] = hi.groupby(df["s1"]).transform("sum").values[rows]
    out["p1_rank_x"] = g_x.rank(ascending=False, method="first").values[rows]
    out["p1_gap_x"] = (df["p"] - g_x.transform("max")).values[rows]
    out["p1_x_second"] = _second_largest(df["x"].values, df["p"].values)[rows]
    out["p1_nhi_x"] = hi.groupby(df["x"]).transform("sum").values[rows]
    for k, v in _ownership_features(df, S1).items():
        out[k] = v[rows].astype(np.float32)

    # Anchor agreement: similarity of this record to the entity's most
    # probable other candidates (siblings of the same real-world business).
    order = df.sort_values(["s1", "p"], ascending=[True, False])
    order["r"] = order.groupby("s1").cumcount()
    top = order[order["r"] < n_anchors]
    anc_x = top.pivot(index="s1", columns="r", values="x")
    anc_p = top.pivot(index="s1", columns="r", values="p")
    names = X["name_full"].values
    addrs = X["addr_full"].values
    s1r = df["s1"].values[rows]
    xr = df["x"].values[rows]
    best_n = np.zeros(len(rows), np.float32)
    best_a = np.zeros(len(rows), np.float32)
    best_j = np.zeros(len(rows), np.float32)
    for r in anc_x.columns:
        ax = anc_x[r].reindex(s1r).values.astype(np.float64)
        ap = anc_p[r].reindex(s1r).values.astype(np.float64)
        valid = ~np.isnan(ax)
        valid[valid] &= ax[valid].astype(np.int64) != xr[valid]
        idx = np.flatnonzero(valid)
        if len(idx) == 0:
            continue
        axi = ax[idx].astype(np.int64)
        xi = xr[idx]
        ns = _sim(list(names[xi]), list(names[axi]), fuzz.token_set_ratio) / 100
        as_ = _sim(list(addrs[xi]), list(addrs[axi]), fuzz.token_set_ratio) / 100
        w = ap[idx].astype(np.float32)
        best_n[idx] = np.maximum(best_n[idx], w * ns)
        best_a[idx] = np.maximum(best_a[idx], w * as_)
        best_j[idx] = np.maximum(best_j[idx], w * np.minimum(ns, as_))
    out["anc_name"] = best_n
    out["anc_addr"] = best_a
    out["anc_joint"] = best_j
    return pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in out.items()})


def stage3_context(cand: pd.DataFrame, p2: np.ndarray, X: pd.DataFrame,
                   n_anchors: int, rows: np.ndarray | None = None,
                   S1: pd.DataFrame | None = None) -> pd.DataFrame:
    """Stage-3 context: the same competition / anchor features recomputed from
    the stage-2 probabilities (prefix ``q_``), plus the record's margin over its
    best competing claimant - the quantity that decides who owns a record."""
    ctx = context_features(cand, p2, X, n_anchors, rows, S1).add_prefix("q_")
    rows = np.arange(len(cand)) if rows is None else rows
    p = p2.astype(np.float32)
    best_x = pd.Series(p).groupby(cand["x_idx"].values).transform("max").values
    second_x = _second_largest(cand["x_idx"].values, p)
    # own p minus the strongest OTHER claimant of the same record
    other = np.where(p >= best_x, second_x, best_x)
    ctx["q_claim_margin"] = (p - other)[rows].astype(np.float32)
    return ctx

In [ ]:
#
# ▌2.8 Gradient-boosted models (`models.py`)
#
# LightGBM trained with 5 folds **grouped by Source 1 entity**; out-of-fold predictions for fitting rows, fold-average predictions for everything else.

In [ ]:
%%writefile {PKG}/src/ber/models.py
"""LightGBM training with entity-grouped K-fold (out-of-fold predictions).

Folds are assigned per S1 entity so that every candidate row of an entity is
in the same fold: out-of-fold predictions are then honest inputs for the
stage-2 context features and for tuning the decision rule. Rows of entities
not used for fitting (hold-out, or beyond ``max_train_s1``) are scored by the
average of the fold models, exactly like the test set.

Feature matrices never have to fit in RAM: they live on disk as float32 .npy
parts (``PartsMatrix``) and are streamed into LightGBM's binned dataset, which
takes about one byte per value. Each finished fold (model + its predictions)
is checkpointed, so an interrupted training resumes at the next fold.
"""
from __future__ import annotations

import logging
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# On-disk feature matrix
# --------------------------------------------------------------------------
class PartsMatrix:
    """Row-wise concatenation of float32 .npy parts (memory-mapped), plus
    optional extra in-memory columns appended on the right."""

    def __init__(self, paths: list[Path], extra: np.ndarray | None = None):
        self.parts = [np.load(p, mmap_mode="r") for p in paths]
        self.bounds = np.cumsum([0] + [len(p) for p in self.parts])
        self.n = int(self.bounds[-1])
        self.extra = extra
        if extra is not None:
            assert len(extra) == self.n, (len(extra), self.n)
        self.ncol = self.parts[0].shape[1] + (0 if extra is None else extra.shape[1])

    def __len__(self) -> int:
        return self.n

    def rows(self, a: int, b: int) -> np.ndarray:
        """Contiguous rows [a, b) as an in-memory float32 matrix."""
        out, s, i = [], a, int(np.searchsorted(self.bounds, a, side="right") - 1)
        while s < b:
            hi = min(b, self.bounds[i + 1])
            out.append(self.parts[i][s - self.bounds[i]:hi - self.bounds[i]])
            s, i = hi, i + 1
        M = np.concatenate(out) if len(out) != 1 else np.array(out[0])
        if self.extra is not None:
            M = np.hstack([M, self.extra[a:b]])
        return np.ascontiguousarray(M, dtype=np.float32)

    def take(self, idx: np.ndarray) -> np.ndarray:
        """Rows at sorted positions ``idx`` as an in-memory float32 matrix."""
        idx = np.asarray(idx, dtype=np.int64)
        part_of = np.searchsorted(self.bounds, idx, side="right") - 1
        out = [self.parts[i][idx[part_of == i] - self.bounds[i]] for i in np.unique(part_of)]
        M = np.concatenate(out) if out else np.zeros((0, self.parts[0].shape[1]), np.float32)
        if self.extra is not None:
            M = np.hstack([M, self.extra[idx]])
        return np.ascontiguousarray(M, dtype=np.float32)


class _PartSeq(lgb.Sequence):
    """Rows [a, b) of a PartsMatrix as a LightGBM Sequence (streamed construction)."""
    batch_size = 65_536

    def __init__(self, M: PartsMatrix, a: int, b: int):
        self.M, self.a, self.b = M, a, b

    def __len__(self) -> int:
        return self.b - self.a

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            start, stop, step = idx.indices(len(self))
            assert step == 1
            return self.M.rows(self.a + start, self.a + stop)
        if isinstance(idx, (int, np.integer)):
            # single rows feed LightGBM's bin-boundary sampling, which requires float64
            # (the same float32 values, widened - the bins are unchanged)
            i = int(idx) + (len(self) if idx < 0 else 0)
            return self.M.rows(self.a + i, self.a + i + 1)[0].astype(np.float64)
        return self.M.take(self.a + np.asarray(idx))


def _sequences(M: PartsMatrix, n: int) -> list[_PartSeq]:
    """Sequences covering the first n rows, split at part boundaries."""
    edges = sorted({0, n, *[int(b) for b in M.bounds if 0 < b < n]})
    return [_PartSeq(M, a, b) for a, b in zip(edges[:-1], edges[1:])]


def _predict_rows(booster, M: PartsMatrix, idx: np.ndarray, chunk: int = 2_000_000) -> np.ndarray:
    out = np.zeros(len(idx), dtype=np.float64)
    for s in range(0, len(idx), chunk):
        out[s:s + chunk] = booster.predict(M.take(idx[s:s + chunk]))
    return out


# --------------------------------------------------------------------------
# Grouped K-fold with per-fold checkpoints
# --------------------------------------------------------------------------
def train_oof(M: PartsMatrix, cols: list, y: np.ndarray, fold: np.ndarray, n_fit: int,
              params: dict, n_folds: int, rounds: int, early_stopping: int, seed: int,
              stage, tag: str):
    """Grouped K-fold LightGBM.

    M      feature matrix whose FIRST ``n_fit`` rows are the fitting rows (their
           ``fold`` ids are used); the remaining rows (hold-out) are only scored.
    stage  checkpoint Stage: ``<tag>_fold<k>.txt`` plus the fold's predictions
           are saved after every fold, and restored instead of retrained.
    Returns (models, predictions): out-of-fold for the first n_fit rows,
    fold-average for the rest.
    """
    pred = np.zeros(M.n, dtype=np.float64)
    fold_fit = fold[:n_fit]
    rest = np.arange(n_fit, M.n)
    models, imp, ds = [], np.zeros(len(cols)), None
    for k in range(n_folds):
        f_model, f_va, f_rest = f"{tag}_fold{k}.txt", f"{tag}_oof{k}.npy", f"{tag}_rest{k}.npy"
        va = np.flatnonzero(fold_fit == k)
        if all(stage.has(f) for f in (f_model, f_va, f_rest)):
            booster = lgb.Booster(model_file=str(stage.fetch(f_model)))
            p_va, p_rest = stage.load_npy(f_va), stage.load_npy(f_rest)
            log.info("%s fold %d restored from checkpoint", tag, k)
        else:
            if ds is None:
                ds = lgb.Dataset(_sequences(M, n_fit), label=y[:n_fit].astype(np.float32),
                                 feature_name=cols, free_raw_data=True,
                                 params={"max_bin": params.get("max_bin", 255), "verbose": -1})
                ds.construct()
            tr = np.flatnonzero(fold_fit != k)
            d_tr, d_va = ds.subset(tr), ds.subset(va)
            booster = lgb.train(
                dict(params, seed=seed + k), d_tr, num_boost_round=rounds,
                valid_sets=[d_va], valid_names=["valid"],
                callbacks=[lgb.early_stopping(early_stopping, verbose=False),
                           lgb.log_evaluation(250)],
            )
            del d_tr, d_va
            log.info("%s fold %d: best_iter=%d valid=%s", tag, k, booster.best_iteration,
                     dict(booster.best_score["valid"]))
            p_va = _predict_rows(booster, M, va)
            p_rest = _predict_rows(booster, M, rest)
            # predictions first, model last: the model file marks the fold complete
            stage.save_npy(f_va, p_va)
            stage.save_npy(f_rest, p_rest)
            stage.save_model(f_model, booster, num_iteration=booster.best_iteration)
        pred[va] = p_va
        pred[n_fit:] += p_rest / n_folds
        imp += booster.feature_importance("gain")
        models.append(booster)
    del ds
    stage.save_bytes(f"{tag}_importance.csv", pd.Series(imp / n_folds, index=cols)
                     .sort_values(ascending=False).to_csv(header=["gain"]).encode("utf-8"))
    return models, pred


def load_models(out_dir: Path, tag: str, n_folds: int):
    return [lgb.Booster(model_file=str(out_dir / f"{tag}_fold{k}.txt")) for k in range(n_folds)]


def predict_avg(models, Xm, chunk: int = 2_000_000) -> np.ndarray:
    """Fold-average prediction for an in-memory array or a PartsMatrix."""
    out = np.zeros(len(Xm), dtype=np.float64)
    for s in range(0, len(Xm), chunk):
        e = min(s + chunk, len(Xm))
        Xc = Xm.rows(s, e) if isinstance(Xm, PartsMatrix) else np.asarray(Xm[s:e], dtype=np.float32)
        out[s:e] = np.mean([m.predict(Xc) for m in models], axis=0)
    return out

In [ ]:
#
# ▌2.9 Cross-encoder (`cross_encoder.py`)
#
# `microsoft/mdeberta-v3-base` (MIT, 278M parameters, multilingual tokenizer) fine-tuned on uncertain pairs; mixed precision on GPU.

In [ ]:
%%writefile {PKG}/src/ber/cross_encoder.py
"""Optional multilingual cross-encoder re-scorer (mDeBERTa-v3-base, MIT licence).

The GBDT is excellent on engineered similarities but cannot "read" a Telugu
business name against its English form, or reason about token-level
correspondences across two noisy strings. A cross-encoder attends jointly over
both records ("name | address" [SEP] "name | address") with a multilingual
SentencePiece vocabulary that natively covers Indic scripts and French.

It is trained only on training pairs and applied only to the uncertain band of
stage-1 probabilities; its score becomes a stage-2 feature (NaN outside the
band), so the GBDT learns how much to trust it.
"""
from __future__ import annotations

import logging
import math
import os
import warnings
from pathlib import Path

import numpy as np

log = logging.getLogger(__name__)

# The model runs on PyTorch. Keep transformers from importing TensorFlow / JAX
# (preinstalled on SageMaker images), whose CUDA plugins only print
# "factory already registered" noise. Must be set before transformers loads.
# The DataLoader's worker processes are forked after the tokenizer was used in the
# main process: Rust-side parallelism must be off to avoid the fork warning / deadlock.
for _k, _v in (("USE_TF", "0"), ("USE_TORCH", "1"), ("USE_FLAX", "0"), ("USE_JAX", "0"),
               ("TRANSFORMERS_NO_TF", "1"), ("TF_CPP_MIN_LOG_LEVEL", "3"), ("TOKENIZERS_PARALLELISM", "false")):
    os.environ.setdefault(_k, _v)


def _quiet_transformers() -> None:
    """Silence expected messages: the freshly initialised classification head
    (it is trained here) and the fast-tokenizer byte-fallback note."""
    import transformers
    transformers.logging.set_verbosity_error()
    warnings.filterwarnings("ignore", message=".*byte fallback option.*")


def _require_torch():
    try:
        import torch  # noqa: F401
        import transformers  # noqa: F401
    except ImportError as e:  # pragma: no cover
        raise ImportError(
            "The cross-encoder stage needs `torch` and `transformers` "
            "(pip install -r requirements.txt), or run with --no-cross-encoder."
        ) from e


def gpu_available() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except ImportError:
        return False


def _grad_scaler(torch, enabled: bool):
    try:  # torch >= 2.3
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def pair_text(names: np.ndarray, addrs: np.ndarray) -> list[str]:
    return [f"{' '.join(n.split())} | {' '.join(a.split())}" for n, a in zip(names, addrs)]


def _amp_dtype(torch, fp16: bool):
    if not fp16 or not torch.cuda.is_available():
        return None
    # Native bf16 needs Ampere+ (A100 / L4 / H100); newer torch reports emulated
    # bf16 as "supported" on a T4, which is far slower than fp16 there.
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def train_cross_encoder(text_a: list[str], text_b: list[str], labels: np.ndarray,
                        cfg, out_dir: Path, seed: int) -> Path:
    _require_torch()
    import torch
    from torch.utils.data import DataLoader
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              get_linear_schedule_with_warmup)

    _quiet_transformers()
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info("cross-encoder training on %s", torch.cuda.get_device_name(0) if device == "cuda" else "CPU")
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=1).to(device)
    idx = np.random.default_rng(seed).permutation(len(labels))

    def collate(batch_ids):
        enc = tok([text_a[i] for i in batch_ids], [text_b[i] for i in batch_ids],
                  truncation=True, max_length=cfg.max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(labels[batch_ids], dtype=torch.float32)
        return enc

    loader = DataLoader(idx, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate,
                        num_workers=getattr(cfg, "num_workers", 2), drop_last=False)
    steps = cfg.epochs * len(loader)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_frac * steps), steps)
    amp = _amp_dtype(torch, cfg.fp16)
    scaler = _grad_scaler(torch, amp == torch.float16)
    lossf = torch.nn.BCEWithLogitsLoss()
    model.train()
    step = 0
    for _ in range(cfg.epochs):
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            y = batch.pop("labels")
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                logits = model(**batch).logits.squeeze(-1)
            loss = lossf(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 500 == 0:
                log.info("cross-encoder step %d/%d loss %.4f", step, steps, loss.item())
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir


def score_pairs(model_dir: Path, text_a: list[str], text_b: list[str], cfg) -> np.ndarray:
    _require_torch()
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    if not text_a:
        return np.zeros(0, dtype=np.float32)
    _quiet_transformers()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    amp = _amp_dtype(torch, cfg.fp16)
    lengths = np.array([len(a) + len(b) for a, b in zip(text_a, text_b)])
    order = np.argsort(lengths)
    out = np.zeros(len(text_a), dtype=np.float32)
    bs = cfg.infer_batch_size
    with torch.no_grad():
        for s in range(0, len(order), bs):
            ids = order[s:s + bs]
            enc = tok([text_a[i] for i in ids], [text_b[i] for i in ids], truncation=True,
                      max_length=cfg.max_len, padding=True, return_tensors="pt").to(device)
            with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
                logits = model(**enc).logits.squeeze(-1).float()
            out[ids] = torch.sigmoid(logits).cpu().numpy()
            if (s // bs) % 2000 == 0:
                log.info("cross-encoder scoring %d / %d", s, len(order))
    return out


def sample_training_pairs(band_idx: np.ndarray, other_idx: np.ndarray, labels: np.ndarray,
                          max_pairs: int, seed: int) -> np.ndarray:
    """Mostly uncertain pairs, plus 20% confident ones for calibration."""
    rng = np.random.default_rng(seed)
    n_band = min(len(band_idx), int(math.ceil(0.8 * max_pairs)))
    n_other = min(len(other_idx), max_pairs - n_band)
    pick = np.concatenate([rng.choice(band_idx, n_band, replace=False) if n_band else band_idx[:0],
                           rng.choice(other_idx, n_other, replace=False) if n_other else other_idx[:0]])
    log.info("cross-encoder training pairs: %d (positives %.3f)", len(pick),
             labels[pick].mean() if len(pick) else 0.0)
    return pick

In [ ]:
#
# ▌2.9b Dense retrieval view (`dense.py`)
#
# `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Apache-2.0, 118M parameters) fine-tuned as a siamese bi-encoder with the symmetric in-batch-negative contrastive loss on training true pairs (country-homogeneous batches, sibling records masked); exact cosine top-k on the GPU. Unlike the sparse views it has no document-frequency cap.

In [ ]:
%%writefile {PKG}/src/ber/dense.py
"""Learned dense retrieval view (bi-encoder), a sixth blocking view.

The sparse TF-IDF views drop every token above a document-frequency cap, and
at full scale (about a million entities per country block) that removes most
of a generic name ("Prime Money", "Shree Ganesh Traders"): such records keep
an almost empty vector and their true entity is crowded out of every top-k.
This is why candidate recall fell from 99.9% on a 20k-entity slice to 98.1%
on the full training set.

A dense view has no cap. Every record is embedded as "name | address"
(normalised text) by a siamese sentence encoder and retrieved by exact cosine
top-k on the GPU, so a common name is still disambiguated by its address and
a transliterated one by its sub-word pieces.

Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
(Apache-2.0, 118M parameters, 384-dim). It is fine-tuned on training true pairs
with the symmetric in-batch-negative contrastive loss (InfoNCE / "multiple
negatives ranking", as in DPR and Sentence-BERT):

  * batches are drawn inside one country, so the negatives are hard;
  * records of the same entity inside a batch are masked out as negatives;
  * the pairs come from a separate slice of training entities that no other
    model is fitted on, so the embedding cosine stays an honest feature for
    the pruner and the GBDTs (see ModelConfig.dense_frac).
"""
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd

from .cross_encoder import _amp_dtype, _grad_scaler, _quiet_transformers, _require_torch

log = logging.getLogger(__name__)


def record_text(frame: pd.DataFrame, idx: np.ndarray | None = None) -> list[str]:
    """Encoder input of every record (or of rows ``idx``): normalised name | address."""
    names, addrs = frame["name_full"].values, frame["addr_full"].values
    if idx is not None:
        names, addrs = names[idx], addrs[idx]
    return [f"{n} | {a}" for n, a in zip(names, addrs)]


def _embed(torch, model, tok, texts: list[str], max_len: int, device, amp):
    """Mean-pooled, L2-normalised sentence embeddings (float32 tensor)."""
    enc = tok(texts, truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(device)
    with torch.autocast("cuda", dtype=amp, enabled=amp is not None):
        h = model(**enc).last_hidden_state
    m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
    v = (h * m).sum(1).float() / m.sum(1).float().clamp(min=1.0)
    return torch.nn.functional.normalize(v, dim=-1)


def _batches(groups: np.ndarray, batch_size: int, rng) -> list[np.ndarray]:
    """Shuffled batches, each drawn inside one group (country)."""
    out = []
    for g in np.unique(groups):
        idx = rng.permutation(np.flatnonzero(groups == g))
        out += [idx[s:s + batch_size] for s in range(0, len(idx), batch_size) if len(idx) - s >= 2]
    return [out[i] for i in rng.permutation(len(out))]


def train_encoder(text_a: list[str], text_b: list[str], entity: np.ndarray, groups: np.ndarray,
                  cfg, out_dir: Path, seed: int) -> Path:
    """Fine-tune the siamese encoder: text_a[i] (S1) and text_b[i] (S2/S3) are a
    true pair of S1 entity ``entity[i]``; ``groups[i]`` (country) keeps batches
    homogeneous."""
    _require_torch()
    import torch
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

    _quiet_transformers()
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    log.info("dense encoder training on %s: %d pairs, %d epoch(s)",
             torch.cuda.get_device_name(0) if device == "cuda" else "CPU", len(text_a), cfg.epochs)
    tok = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModel.from_pretrained(cfg.model_name).to(device)
    rng = np.random.default_rng(seed)
    entity = np.asarray(entity)
    n_batches = len(_batches(groups, cfg.batch_size, np.random.default_rng(seed)))
    steps = max(1, cfg.epochs * n_batches)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_frac * steps), steps)
    amp = _amp_dtype(torch, cfg.fp16)
    scaler = _grad_scaler(torch, amp == torch.float16)
    ce = torch.nn.functional.cross_entropy
    model.train()
    step = 0
    for _ in range(cfg.epochs):
        for b in _batches(groups, cfg.batch_size, rng):
            ea = _embed(torch, model, tok, [text_a[i] for i in b], cfg.max_len, device, amp)
            eb = _embed(torch, model, tok, [text_b[i] for i in b], cfg.max_len, device, amp)
            logits = ea @ eb.T / cfg.temperature
            e = torch.as_tensor(entity[b], device=device)
            # another record of the same entity is a true match, not a negative
            sibling = (e[:, None] == e[None, :]) & ~torch.eye(len(b), dtype=torch.bool, device=device)
            logits = logits.masked_fill(sibling, -1e4)
            target = torch.arange(len(b), device=device)
            loss = (ce(logits, target) + ce(logits.T, target)) / 2
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            step += 1
            if step % 250 == 0:
                log.info("dense encoder step %d/%d loss %.4f", step, steps, loss.item())
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir


def encode(model_dir: Path, texts: list[str], cfg) -> np.ndarray:
    """L2-normalised float16 embeddings of ``texts`` (batched by length)."""
    _require_torch()
    import torch
    from transformers import AutoModel, AutoTokenizer

    _quiet_transformers()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModel.from_pretrained(model_dir).to(device).eval()
    amp = _amp_dtype(torch, cfg.fp16)
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float16)
    order = np.argsort(np.fromiter((len(t) for t in texts), dtype=np.int64, count=len(texts)), kind="stable")
    bs = cfg.infer_batch_size
    with torch.inference_mode():
        for s in range(0, len(order), bs):
            ids = order[s:s + bs]
            e = _embed(torch, model, tok, [texts[i] for i in ids], cfg.max_len, device, amp)
            out[ids] = e.half().cpu().numpy()
            if (s // bs) % 500 == 0:
                log.info("dense encoding %d / %d", s, len(order))
    return out


def topk_dense(A: np.ndarray, B: np.ndarray, k: int):
    """For every row of A, the k rows of B with the highest cosine (rows are
    L2-normalised). Exact search in row chunks on the GPU (CPU fallback).
    Returns (row of A, row of B) int64 arrays."""
    empty = np.empty(0, dtype=np.int64)
    if len(A) == 0 or len(B) == 0 or k <= 0:
        return empty, empty
    _require_torch()
    import torch

    k = min(k, len(B))
    cuda = torch.cuda.is_available()
    dev = torch.device("cuda" if cuda else "cpu")
    dt = torch.float16 if cuda else torch.float32
    Bt = torch.from_numpy(np.ascontiguousarray(B)).to(dev, dt)
    # score-matrix rows per chunk: a quarter of the free GPU memory (speed only; results are identical)
    budget = torch.cuda.mem_get_info()[0] // 4 if cuda else 2 * 2**30
    chunk = int(max(16, min(len(A), budget // (len(B) * (2 if cuda else 4)))))
    ia, ib = [], []
    with torch.inference_mode():
        for s in range(0, len(A), chunk):
            a = torch.from_numpy(np.ascontiguousarray(A[s:s + chunk])).to(dev, dt)
            idx = torch.topk(a @ Bt.T, k, dim=1, sorted=False).indices
            ib.append(idx.cpu().numpy().astype(np.int64).ravel())
            ia.append(np.repeat(np.arange(s, s + len(a), dtype=np.int64), k))
    del Bt
    if cuda:
        torch.cuda.empty_cache()
    return np.concatenate(ia), np.concatenate(ib)


def rowwise_cos(A: np.ndarray, B: np.ndarray, ia: np.ndarray, ib: np.ndarray,
                chunk: int = 2_000_000) -> np.ndarray:
    out = np.zeros(len(ia), dtype=np.float32)
    for s in range(0, len(ia), chunk):
        e = min(s + chunk, len(ia))
        out[s:e] = np.einsum("ij,ij->i", A[ia[s:e]].astype(np.float32), B[ib[s:e]].astype(np.float32))
    return out

In [ ]:
#
# ▌2.10 Decision rule & metric (`decision.py`)
#
# Macro F0.5 (singletons included) and the exact expected-F0.5 subset selection via Poisson-binomial dynamic programming.

In [ ]:
%%writefile {PKG}/src/ber/decision.py
"""From pair probabilities to per-entity match lists, optimised for macro F0.5.

The metric is F_beta computed per S1 entity and averaged, with singletons
scoring 1 for an empty prediction and 0 otherwise. For one entity with
independent match probabilities p_1 >= p_2 >= ... the optimal prediction is
always a top-k prefix, and its expected F_beta can be computed exactly:

  E[F | k] = sum_{a,b} P(TP=a | top-k) * P(FN=b | rest) * (1+b^2) a / (b^2 (a+b) + k)
  E[F | 0] = P(no true match at all)

(Poisson-binomial distributions via DP). We choose k* = argmax_k E[F | k]
per entity - this handles singletons, multi-match entities and ambiguous
cases in one principled rule instead of a single global threshold. A
temperature on the logits (fitted on out-of-fold predictions) corrects
calibration. A global-threshold rule and an "each S2/S3 record belongs to at
most one S1" exclusivity step are evaluated as alternatives; the best rule on
out-of-fold data is kept.
"""
from __future__ import annotations

import logging

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)


# --------------------------------------------------------------------------
# Metric
# --------------------------------------------------------------------------
def macro_fbeta(s1_codes: np.ndarray, label: np.ndarray, selected: np.ndarray,
                n_gt: np.ndarray, beta: float = 0.5) -> float:
    """Vectorised macro F_beta.

    s1_codes : int array (rows) - dense code of the S1 entity of each candidate row
    label    : 0/1 per row (is the candidate a true match)
    selected : bool per row (is it predicted)
    n_gt     : per-entity number of true matches (length = number of entities,
               including entities without any candidate row)
    """
    n = len(n_gt)
    tp = np.bincount(s1_codes, weights=(label * selected).astype(np.float64), minlength=n)
    npred = np.bincount(s1_codes, weights=selected.astype(np.float64), minlength=n)
    b2 = beta * beta
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(npred > 0, tp / npred, 0.0)
        rec = np.where(n_gt > 0, tp / n_gt, 0.0)
        f = np.where(tp > 0, (1 + b2) * prec * rec / (b2 * prec + rec), 0.0)
    f = np.where((n_gt == 0) & (npred == 0), 1.0, f)
    return float(f.mean())


# --------------------------------------------------------------------------
# Expected-F optimal selection
# --------------------------------------------------------------------------
def _weights(m: int, beta: float) -> np.ndarray:
    b2 = beta * beta
    W = np.zeros((m + 1, m + 1, m + 1))
    for k in range(1, m + 1):
        for a in range(1, k + 1):
            for b in range(0, m - k + 1):
                W[k, a, b] = (1 + b2) * a / (b2 * (a + b) + k)
    return W


def _poisson_binomial_prefix(P: np.ndarray) -> np.ndarray:
    n, m = P.shape
    out = np.zeros((n, m + 1, m + 1))
    out[:, 0, 0] = 1.0
    for k in range(1, m + 1):
        p = P[:, k - 1][:, None]
        out[:, k, :] = out[:, k - 1, :] * (1 - p)
        out[:, k, 1:] += out[:, k - 1, :-1] * p
    return out


def _expected_best_k(P: np.ndarray, W: np.ndarray, chunk: int = 20_000) -> np.ndarray:
    n, m = P.shape
    best = np.zeros(n, dtype=np.int64)
    for s in range(0, n, chunk):
        Pc = P[s:s + chunk]
        pre = _poisson_binomial_prefix(Pc)
        suf = _poisson_binomial_prefix(Pc[:, ::-1])[:, ::-1, :]  # suf[:, k] = items k..m-1
        # E[n, k] = sum_{a,b} pre[n,k,a] * W[k,a,b] * suf[n,k,b], as m+1 small matrix products
        E = np.empty((len(Pc), m + 1))
        for k in range(m + 1):
            E[:, k] = ((pre[:, k, :] @ W[k]) * suf[:, k, :]).sum(axis=1)
        E[:, 0] = suf[:, 0, 0]
        best[s:s + chunk] = E.argmax(axis=1)
    return best


def _temper(p: np.ndarray, T: float, bias: float = 0.0) -> np.ndarray:
    """Platt-style recalibration on the logit scale: sigmoid(logit(p) / T + bias)."""
    if T == 1.0 and bias == 0.0:
        return p
    p = np.clip(p, 1e-6, 1 - 1e-6)
    z = np.log(p / (1 - p)) / T + bias
    return 1.0 / (1.0 + np.exp(-z))


# How a record's probability is shared among the S1 entities that claim it. A record
# belongs to at most one S1 entity, but each pair was scored on its own:
#   raw     the pair probabilities as they are
#   renorm  divided by the record's total claimed probability when it exceeds 1
#           (the claims of one record are mutually exclusive events)
#   best    only the record's most probable claimant keeps its probability, so the
#           per-entity choice below is made knowing which records it cannot win
PROB_MODES = ("raw", "renorm", "best")


def transform_probs(x_codes: np.ndarray, p: np.ndarray, mode: str) -> np.ndarray:
    if mode == "raw" or len(p) == 0:
        return p
    if mode == "renorm":
        total = np.bincount(x_codes, weights=p)[x_codes]
        return p / np.maximum(total, 1.0)
    if mode == "best":
        best = pd.Series(p).groupby(x_codes).transform("max").values
        return np.where(p >= best, p, 0.0)
    raise ValueError(f"unknown probability mode {mode!r}")


def _groups(s1_codes: np.ndarray, p: np.ndarray, max_items: int):
    """Per-entity ranking of rows by probability. A temperature is monotone, so
    the same ranking serves every temperature of the grid."""
    order = np.lexsort((-p, s1_codes))
    codes_sorted = s1_codes[order]
    first = np.r_[True, codes_sorted[1:] != codes_sorted[:-1]]
    start = np.maximum.accumulate(np.where(first, np.arange(len(order)), 0))
    rank = np.arange(len(order)) - start
    grp = np.cumsum(first) - 1
    n_grp = int(grp[-1] + 1) if len(grp) else 0
    return order, rank, grp, n_grp, rank < max_items


def select_expected_f(s1_codes: np.ndarray, p: np.ndarray, beta: float, max_items: int,
                      temperature: float = 1.0, groups=None, W=None, bias: float = 0.0) -> np.ndarray:
    order, rank, grp, n_grp, keep = groups if groups is not None else _groups(s1_codes, p, max_items)
    P = np.zeros((n_grp, max_items))
    P[grp[keep], rank[keep]] = _temper(p[order][keep], temperature, bias)
    kstar = _expected_best_k(P, W if W is not None else _weights(max_items, beta))
    selected = np.zeros(len(p), dtype=bool)
    selected[order] = rank < kstar[grp]
    return selected


def apply_exclusivity(x_codes: np.ndarray, p: np.ndarray, selected: np.ndarray,
                      p_order: np.ndarray | None = None) -> np.ndarray:
    """Each S2/S3 record may belong to at most one S1 entity: keep the most probable."""
    if not selected.any():
        return selected
    if p_order is None:
        p_order = np.argsort(-p, kind="stable")
    idx = p_order[selected[p_order]]                      # selected rows, most probable first
    _, first = np.unique(x_codes[idx], return_index=True)
    out = np.zeros_like(selected)
    out[idx[first]] = True
    return out


def decide(s1_codes, x_codes, p, params: dict, beta: float, max_items: int) -> np.ndarray:
    """Selected rows. Every row of the candidate graph must be passed together:
    the probability sharing and the exclusivity look at all claimants of a record."""
    q = transform_probs(x_codes, p, params.get("prob", "raw"))
    if params["rule"] == "expf":
        sel = select_expected_f(s1_codes, q, beta, max_items, params["temperature"],
                                bias=params.get("bias", 0.0))
    else:
        sel = q >= params["threshold"]
    if params["exclusive"]:
        sel = apply_exclusivity(x_codes, q, sel)
    return sel


def tune_decision(s1_codes, x_codes, p, cfg, score) -> tuple[dict, float, list]:
    """Grid-search the decision rule; ``score(selected) -> macro F`` evaluates a
    selection of all rows on the tuning entities (out-of-fold predictions).

    Rules: exact expected-F (temperature grid, then a calibration-offset grid
    around the best temperature) and a global threshold, for every probability
    mode, each with and without exclusivity. The per-entity ranking, the F-weights
    and the probability order are computed once per mode.
    """
    W = _weights(cfg.max_items, cfg.beta)
    modes = tuple(getattr(cfg, "prob_modes", ("raw",)))
    results = []
    prepared = {}

    def prep(mode):
        if mode not in prepared:
            q = transform_probs(x_codes, p, mode)
            prepared.clear()                      # one mode in memory at a time
            prepared[mode] = (q, _groups(s1_codes, q, cfg.max_items), np.argsort(-q, kind="stable"))
        return prepared[mode]

    def run(base, mode, sel, q, q_order):
        for excl in ((False,) if mode == "best" else (False, True)):     # "best" is exclusive already
            s = apply_exclusivity(x_codes, q, sel, q_order) if excl else sel
            results.append((score(s), {**base, "prob": mode, "exclusive": excl}))

    for mode in modes:
        q, groups, q_order = prep(mode)
        for T in cfg.temperatures:
            run({"rule": "expf", "temperature": float(T), "bias": 0.0}, mode,
                select_expected_f(s1_codes, q, cfg.beta, cfg.max_items, T, groups, W), q, q_order)
        for t in cfg.thresholds:
            run({"rule": "threshold", "threshold": float(t)}, mode, q >= t, q, q_order)
    # calibration offset around the best expected-F setting
    best = max((r for r in results if r[1]["rule"] == "expf"), key=lambda r: r[0], default=None)
    if best is not None and getattr(cfg, "biases", ()):
        mode, T = best[1]["prob"], best[1]["temperature"]
        q, groups, q_order = prep(mode)
        for b in cfg.biases:
            if b != 0.0:
                run({"rule": "expf", "temperature": T, "bias": float(b)}, mode,
                    select_expected_f(s1_codes, q, cfg.beta, cfg.max_items, T, groups, W, bias=b), q, q_order)
    prepared.clear()
    results.sort(key=lambda r: -r[0])
    for s, params in results[:5]:
        log.info("decision candidate %.5f %s", s, params)
    return results[0][1], results[0][0], [(s, p_) for s, p_ in results]

In [ ]:
#
# ▌2.10b Diagnostics (`diagnostics.py`)
#
# The measurements of cells 10.1/10.2/14.1/14.3-14.5/15.2 as JSON-ready functions, used by the SageMaker training jobs (`sagemaker/`). The notebook shows the same numbers in its own cells.

In [ ]:
%%writefile {PKG}/src/ber/diagnostics.py
"""Read-only diagnostics of a trained state, as JSON-ready dicts.

The same measurements the notebook prints in 10.1 / 10.2 (blocking), 14.1 /
14.3 / 14.4 (score and where it is lost), 14.5 (ambiguity ceiling) and 15.2
(corpus-size check), for runs without a notebook (SageMaker training jobs).
Training data only; nothing here changes a model or a checkpoint.

Entities evaluated: the hold-out when the run has one; otherwise (hold-out
fraction 0) the fit entities scored by their out-of-fold predictions.
"""
from __future__ import annotations

import logging
import time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

from .blocking import VIA_BIT, VIA_DENSE, VIA_SPARSE, VIEWS
from .decision import decide

log = logging.getLogger(__name__)


def _f(x) -> float | None:
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else float(x)


def per_entity_f(codes, label, sel, n_gt, beta: float = 0.5):
    """Per-entity F_beta (1 for a singleton predicted empty) plus tp and #predicted."""
    n = len(n_gt)
    label, sel = np.asarray(label, bool), np.asarray(sel, bool)
    tp = np.bincount(codes, weights=(label & sel).astype(float), minlength=n)
    npred = np.bincount(codes, weights=sel.astype(float), minlength=n)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.where(npred > 0, tp / npred, 0.0)
        R = np.where(n_gt > 0, tp / n_gt, 0.0)
        f = np.where(tp > 0, (1 + beta ** 2) * P * R / (beta ** 2 * P + R), 0.0)
    return np.where((n_gt == 0) & (npred == 0), 1.0, f), tp, npred


def eval_view(st) -> SimpleNamespace:
    """Rows, final probabilities and selection of the evaluated entities."""
    nf, dc = st.n_fit_rows, st.cfg.decision
    if st.is_hold.any():
        ents, rows, p = np.flatnonzero(st.is_hold), st.model_rows[nf:], st.pfin[nf:]
        sel, name = np.asarray(st.holdout_sel, bool), "hold-out"
    else:
        ents, rows, p = np.flatnonzero(st.is_fit), st.model_rows[:nf], st.pfin[:nf]
        sel, name = None, "out-of-fold (fit entities; no hold-out in this run)"
    codes = np.searchsorted(ents, st.cand["s1_idx"].values[rows])
    if sel is None and getattr(st, "sel_full", None) is not None:
        sel = st.sel_full[rows]               # the rule applied to the whole candidate graph, as at test time
    if sel is None:
        sel = decide(codes, st.cand["x_idx"].values[rows], p, st.decision, dc.beta, dc.max_items)
    return SimpleNamespace(ents=ents, rows=rows, p=p, sel=np.asarray(sel, bool), codes=codes,
                           y=st.y[rows].astype(bool), n_gt=st.n_gt[ents], name=name)


def blocking_report(st) -> dict:
    """10.1 metrics + 10.2 retrieval breakdown of the true candidates."""
    keys = ("union_pairs", "cand_pairs", "cand_per_s1", "union_recall_eval_sparse_only",
            "union_recall_eval", "cand_recall_eval")
    out = {k: _f(st.metrics.get(k)) for k in keys}
    cand = st.cand
    is_true = st.owner[cand["x_idx"].values] == cand["s1_idx"].values
    ev = (st.is_hold if st.is_hold.any() else st.is_fit)[cand["s1_idx"].values] & is_true
    via = cand["via"].values[ev]
    sp, dn = (via & VIA_SPARSE) > 0, (via & VIA_DENSE) > 0
    if len(via):
        out["true_candidates_found_by"] = {"sparse and dense": _f((sp & dn).mean()),
                                           "sparse views only": _f((sp & ~dn).mean()),
                                           "dense view only": _f((~sp & dn).mean())}
        out["true_candidates_found_by_view"] = {
            v: _f(((via & (VIA_BIT[(v, "fwd")] | VIA_BIT[(v, "rev")])) > 0).mean()) for v in (*VIEWS, "dense")}
        rank = cand.groupby(["s1_idx", "src"], sort=False).cumcount().values + 1
        rt = rank[ev]
        out["true_pruner_rank_1_3"] = _f((rt <= 3).mean())
        out["true_pruner_rank_1_6"] = _f((rt <= 6).mean())
    return out


def score_report(st) -> dict:
    """14.1 metrics (stages, decision rule, hold-out scores)."""
    keys = ("cand_recall_eval", "oof_macro_f05_stage2", "oof_macro_f05_stage3", "oof_macro_f05_stage23",
            "final_stage", "oof_macro_f05", "holdout_macro_f05_stage2", "holdout_macro_f05_stage3",
            "holdout_macro_f05_stage23", "holdout_macro_f05",
            "holdout_macro_f05_stage1_t05", "holdout_macro_f05_oracle", "holdout_pair_precision",
            "holdout_pair_recall")
    out = {k: (st.metrics.get(k) if k == "final_stage" else _f(st.metrics.get(k))) for k in keys}
    out["decision"] = st.metrics.get("decision")
    return out


def loss_report(st, ev) -> dict:
    """14.3 by number of true matches + 14.4 where the F0.5 is lost."""
    f_act, tp, npred = per_entity_f(ev.codes, ev.y, ev.sel, ev.n_gt)
    f_orc, reach, _ = per_entity_f(ev.codes, ev.y, ev.y, ev.n_gt)
    n_gt, n = ev.n_gt, len(ev.ents)
    kind = np.select(
        [(n_gt == 0) & (npred > 0), (n_gt > 0) & (npred == 0) & (reach > 0),
         (n_gt > 0) & (npred > tp), (n_gt > 0) & (tp < reach)],
        ["singleton given a match (scores 0)", "matchable entity left empty (scores 0)",
         "entity with >=1 wrong match", "entity missing reachable matches (no wrong one)"], "correct")
    loss = pd.DataFrame({"kind": kind, "loss": f_orc - f_act}).groupby("kind")["loss"]
    by_kind = {k: {"entities": int(g.size), "loss_points": _f(g.sum() / n)} for k, g in loss}
    by_n = pd.DataFrame({"m": np.minimum(n_gt, 8), "f": f_act}).groupby("m")["f"].agg(["mean", "size"])
    xi = st.cand["x_idx"].values[ev.rows]
    fp, fn = ev.sel & ~ev.y, ~ev.sel & ev.y
    q = [0.1, 0.25, 0.5, 0.75, 0.9]

    def share(col, mask):
        return _f(st.X[col].values[xi[mask]].mean()) if mask.any() else None

    ctry = st.S1["country"].values[ev.ents]
    by_c = pd.DataFrame({"country": ctry, "f": f_act, "ceiling": f_orc}).groupby("country")
    return {
        "evaluated_on": ev.name, "entities": n,
        "macro_f05": _f(f_act.mean()), "ceiling_with_these_candidates": _f(f_orc.mean()),
        "lost_at_blocking": _f(1 - f_orc.mean()), "lost_by_model_and_decision": _f(f_orc.mean() - f_act.mean()),
        "loss_by_kind": by_kind,
        "f05_by_true_matches": {(str(int(m)) if m < 8 else "8+"): {"f05": _f(r["mean"]), "entities": int(r["size"])}
                                for m, r in by_n.iterrows()},
        "wrong_matches": int(fp.sum()),
        "wrong_match_record_owned_by_other_s1": _f((st.owner[xi[fp]] >= 0).mean()) if fp.any() else None,
        "p_of_missed_true_matches_q10_25_50_75_90": [_f(v) for v in np.quantile(ev.p[fn], q)] if fn.any() else None,
        "p_of_wrong_matches_q10_25_50_75_90": [_f(v) for v in np.quantile(ev.p[fp], q)] if fp.any() else None,
        "by_country": {c: {"f05": _f(g["f"].mean()), "ceiling": _f(g["ceiling"].mean()), "entities": int(len(g))}
                       for c, g in by_c},
        "record_name_non_latin_share": {"all true pairs": share("name_nonlatin", ev.y),
                                        "missed (FN)": share("name_nonlatin", fn), "wrong (FP)": share("name_nonlatin", fp)},
        "record_address_empty_share": {"all true pairs": share("addr_empty", ev.y),
                                       "missed (FN)": share("addr_empty", fn), "wrong (FP)": share("addr_empty", fp)},
    }


def ambiguity_report(st, ev) -> dict:
    """14.5: estimated ceiling when records with an empty address whose entity shares its
    exact core name with other S1 entities of the country are left out (unresolvable)."""
    key = pd.Series(st.S1["country"].values).astype(str) + "|" + pd.Series(st.S1["name_core"].values)
    n_same = key.map(key.value_counts()).values
    gx = pd.Index(np.asarray(st.X["entity_id"].values, dtype=object)).get_indexer(st.gt_x)
    ok = (st.gt_gi >= 0) & (gx >= 0)
    gi, gxx = st.gt_gi[ok], gx[ok]
    in_ev = np.zeros(len(st.S1), bool)
    in_ev[ev.ents] = True
    gi, gxx = gi[in_ev[gi]], gxx[in_ev[gi]]
    amb = (st.X["addr_empty"].values[gxx] == 1) & (n_same[gi] >= 2)
    tp_floor = np.bincount(np.searchsorted(ev.ents, gi[~amb]), minlength=len(ev.ents)).astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):
        R = np.where(ev.n_gt > 0, tp_floor / ev.n_gt, 0.0)
        f_ceiling = np.where(tp_floor > 0, 1.25 * R / (0.25 + R), 0.0)
    f_ceiling = np.where(ev.n_gt == 0, 1.0, f_ceiling)
    has_amb = np.bincount(np.searchsorted(ev.ents, gi[amb]), minlength=len(ev.ents)) > 0
    f_now, _, _ = per_entity_f(ev.codes, ev.y, ev.sel, ev.n_gt)
    lost = 1 - f_now
    n = max(len(lost), 1)
    return {"evaluated_on": ev.name, "true_pairs": int(len(amb)), "ambiguous_true_pairs": int(amb.sum()),
            "ambiguous_share": _f(amb.mean()) if len(amb) else 0.0,
            "entities_with_ambiguous_record": int(has_amb.sum()),
            "AMBIGUITY_CEILING": _f(f_ceiling.mean()), "current_macro_f05": _f(f_now.mean()),
            "loss_points_on_entities_with_ambiguous_records": _f(lost[has_amb].sum() / n),
            "loss_points_on_other_entities": _f(lost[~has_amb].sum() / n),
            "irreducible_loss_points": _f((1 - f_ceiling[has_amb]).sum() / n)}


def _f05(pred: set, gt: set) -> float:
    if not gt:
        return float(not pred)
    tp = len(pred & gt)
    if tp == 0:
        return 0.0
    P, R = tp / len(pred), tp / len(gt)
    return 1.25 * P * R / (0.25 * P + R)


def _dump(path: Path, frame: pd.DataFrame, mask: np.ndarray) -> None:
    # the raw text exactly as read (no field contains a tab), in the challenge's TSV layout
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        fh.write("entity_id\tbusiness_name\tbusiness_address\tcountry\n")
        for r in zip(*(frame[c].values[mask] for c in ("entity_id", "raw_name", "raw_addr", "country"))):
            fh.write("\t".join(r) + "\n")


def corpus_size_check(st, ev, fracs, sim_root: Path, out_root: Path, predict_fn, **predict_kw) -> list[dict]:
    """15.2: rebuild smaller corpora from the TRAINING files (a fraction of the S1 entities,
    all their records, the same fraction of unowned records), run the trained pipeline on each
    exactly as on the test set, and score the hold-out entities inside. Needs a hold-out."""
    if not fracs or not st.is_hold.any():
        return []
    owned = st.owner >= 0
    f_full = pd.Series(per_entity_f(ev.codes, ev.y, ev.sel, ev.n_gt)[0], index=st.S1["entity_id"].values[ev.ents])
    rng = np.random.default_rng(11)
    res = []
    for frac in fracs:
        name = f"sim{int(round(frac * 1000)):04d}"
        pick = rng.random(len(st.S1)) < frac
        keep_x = np.where(owned, pick[np.clip(st.owner, 0, None)], rng.random(len(st.X)) < frac)
        d = Path(sim_root) / name
        if not (d / f"{name}_source3.tsv").exists():
            d.mkdir(parents=True, exist_ok=True)
            _dump(d / f"{name}_source1.tsv", st.S1, pick)
            for s in (2, 3):
                _dump(d / f"{name}_source{s}.tsv", st.X, keep_x & (st.X["src"].values == s))
        out = Path(out_root) / name
        t0 = time.time()
        predict_fn(Path(sim_root), name, st.work, out, **predict_kw)
        pred = pd.read_csv(out / "matching_results.tsv", sep="\t", dtype=str, keep_default_na=False)
        pm = dict(zip(pred["source1_entity_id"], pred["matched_entity_ids"]))
        ids = st.S1["entity_id"].values[pick & st.is_hold]
        f_sim = np.array([_f05(set(filter(None, pm.get(i, "").split(","))), set(st.gt.get(i, []))) for i in ids])
        res.append({"corpus_size": frac, "s1_entities": int(pick.sum()), "records": int(keep_x.sum()),
                    "holdout_entities_scored": int(len(ids)),
                    "f05_in_this_corpus": _f(f_sim.mean()) if len(ids) else None,
                    "same_entities_full_corpus": _f(f_full.loc[ids].mean()) if len(ids) else None,
                    "minutes": round((time.time() - t0) / 60, 1)})
        log.info("corpus-size check: %s", res[-1])
    return res


def full_report(st) -> dict:
    ev = eval_view(st)
    return {"blocking": blocking_report(st), "score": score_report(st),
            "loss": loss_report(st, ev), "ambiguity": ambiguity_report(st, ev)}

In [ ]:
#
# ▌2.11 Checkpoints (`checkpoint.py`)
#
# Every expensive step writes its outputs into a stage folder whose name carries a hash of the data, the settings and the upstream stages. Each file is uploaded to S3 the moment it is complete, so a new session (or a new instance) restores finished work and resumes at the first missing file.

In [ ]:
%%writefile {PKG}/src/ber/checkpoint.py
"""Stage checkpoints on local disk, mirrored to S3 so that any instance can resume.

Every expensive step writes its outputs into a *stage* directory whose name
carries a hash of everything those outputs depend on (data, settings and the
upstream stages):

    <root>/<stage>-<hash12>/<file>          local copy (fast working storage)
    <s3_uri><stage>-<hash12>/<file>         durable copy

Each file is written atomically and uploaded as soon as it is complete (one
blocking block, one feature part, one model fold, ...), and a stage is marked
finished by ``_DONE.json``, written last. A later session - after a kernel
crash, or on a brand-new instance with an empty disk - looks for every file
locally first and then on S3, so it continues from the last completed file
instead of from the start of the step.

Settings that only change speed or memory are left out of the hashes. Changing
any other setting starts fresh stages for that step and everything downstream
of it, while the upstream stages are reused.
"""
from __future__ import annotations

import hashlib
import json
import logging
import os
import pickle
import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

log = logging.getLogger(__name__)

DONE = "_DONE.json"


def fingerprint(obj) -> str:
    """Short stable hash of a JSON-serialisable object."""
    text = json.dumps(obj, sort_keys=True, default=list)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


def link_or_copy(src: Path, dst: Path) -> None:
    """Hard-link a file (no extra disk) or copy it when linking is impossible."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        dst.unlink()
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


class Checkpoints:
    """Factory of stages under one local root and (optionally) one S3 prefix."""

    def __init__(self, root: Path, s3_uri: str | None = None, region: str | None = None,
                 endpoint: str | None = None, enabled: bool = True):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.enabled = enabled
        self.s3_uri = (s3_uri.rstrip("/") + "/") if (s3_uri and enabled) else None
        self.s3 = None
        if self.s3_uri:
            import boto3
            from boto3.s3.transfer import TransferConfig
            from botocore.config import Config as BotoConfig

            assert self.s3_uri.startswith("s3://"), f"not an s3:// URI: {s3_uri}"
            self.bucket, _, self.prefix = self.s3_uri[5:].partition("/")
            self.s3 = boto3.client("s3", region_name=region, endpoint_url=endpoint, config=BotoConfig(
                max_pool_connections=32, retries={"max_attempts": 10, "mode": "adaptive"}))
            self.xfer = TransferConfig(multipart_threshold=64 * 2**20, multipart_chunksize=64 * 2**20,
                                       max_concurrency=16)

    def stage(self, name: str, fp: dict) -> "Stage":
        return Stage(self, name, fp)


class Stage:
    """One step's checkpoint directory (local + S3)."""

    def __init__(self, ck: Checkpoints, name: str, fp: dict):
        self.ck = ck
        self.name = name
        self.hash = fingerprint(fp)
        self.key = f"{name}-{self.hash}"
        self.dir = ck.root / self.key
        self.dir.mkdir(parents=True, exist_ok=True)
        fp_path = self.dir / "_fingerprint.json"
        if not fp_path.exists():
            fp_path.write_text(json.dumps(fp, indent=1, sort_keys=True, default=list))
        self._remote: dict | None = None

    def __repr__(self) -> str:
        return f"Stage({self.key}, done={self.done})"

    # ---------------------------------------------------------------- lookup
    def _s3_key(self, rel: str) -> str:
        return f"{self.ck.prefix}{self.key}/{rel}"

    def remote(self) -> dict:
        """rel path -> size of every object of this stage on S3 (listed once)."""
        if self._remote is None:
            self._remote = {}
            if self.ck.s3 is not None:
                pre = self._s3_key("")
                pages = self.ck.s3.get_paginator("list_objects_v2").paginate(
                    Bucket=self.ck.bucket, Prefix=pre)
                for page in pages:
                    for obj in page.get("Contents", []):
                        self._remote[obj["Key"][len(pre):]] = obj["Size"]
        return self._remote

    def path(self, rel: str) -> Path:
        return self.dir / rel

    def has(self, rel: str) -> bool:
        """Is this file complete, locally or on S3? (local files are written atomically)"""
        if not self.ck.enabled:
            return False
        return self.path(rel).exists() or rel in self.remote()

    @property
    def done(self) -> bool:
        return self.has(DONE)

    def fetch(self, rel: str) -> Path:
        """Local path of a checkpoint file, downloading it from S3 when needed."""
        p = self.path(rel)
        if p.exists():
            return p
        if rel not in self.remote():
            raise FileNotFoundError(f"{rel} is in neither {self.dir} nor {self.ck.s3_uri}{self.key}/")
        p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_name(p.name + ".part")
        self.ck.s3.download_file(self.ck.bucket, self._s3_key(rel), str(tmp), Config=self.ck.xfer)
        tmp.replace(p)
        return p

    def fetch_many(self, rels: list[str]) -> list[Path]:
        missing = [r for r in rels if not self.path(r).exists()]
        if missing:
            log.info("downloading %d checkpoint file(s) of %s from S3 (%.2f GB)", len(missing), self.key,
                     sum(self.remote().get(r, 0) for r in missing) / 1e9)
            with ThreadPoolExecutor(8) as ex:
                list(ex.map(self.fetch, missing))
        return [self.path(r) for r in rels]

    def fetch_dir(self, rel_dir: str) -> Path:
        pre = rel_dir.rstrip("/") + "/"
        self.fetch_many([r for r in self.remote() if r.startswith(pre)])
        return self.path(rel_dir)

    # ---------------------------------------------------------------- upload
    def commit(self, rel: str) -> Path:
        """Upload a completed local file (or a whole directory) to S3."""
        p = self.path(rel)
        files = [p] if p.is_file() else sorted(q for q in p.rglob("*") if q.is_file())
        if self.ck.s3 is not None and files:
            def up(q: Path) -> None:
                r = q.relative_to(self.dir).as_posix()
                self.ck.s3.upload_file(str(q), self.ck.bucket, self._s3_key(r), Config=self.ck.xfer)
                self.remote()[r] = q.stat().st_size

            with ThreadPoolExecutor(8) as ex:
                list(ex.map(up, files))
        return p

    def _atomic(self, rel: str, write) -> Path:
        p = self.path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_name(p.name + ".tmp")
        write(tmp)
        tmp.replace(p)
        return self.commit(rel)

    def save_bytes(self, rel: str, data: bytes) -> Path:
        return self._atomic(rel, lambda t: t.write_bytes(data))

    def save_json(self, rel: str, obj) -> Path:
        return self.save_bytes(rel, json.dumps(obj, indent=1, default=float).encode("utf-8"))

    def save_pickle(self, rel: str, obj) -> Path:
        return self.save_bytes(rel, pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL))

    def save_npy(self, rel: str, arr: np.ndarray) -> Path:
        def write(t: Path) -> None:
            with open(t, "wb") as f:
                np.save(f, np.asarray(arr))
        return self._atomic(rel, write)

    def save_frame(self, rel: str, df: pd.DataFrame) -> Path:
        return self._atomic(rel, lambda t: df.to_parquet(t, index=False))

    def save_model(self, rel: str, booster, num_iteration: int | None = None) -> Path:
        return self._atomic(rel, lambda t: booster.save_model(str(t), num_iteration=num_iteration))

    # ---------------------------------------------------------------- load
    def load_json(self, rel: str):
        return json.loads(self.fetch(rel).read_text(encoding="utf-8"))

    def load_pickle(self, rel: str):
        return pickle.loads(self.fetch(rel).read_bytes())

    def load_npy(self, rel: str, mmap: bool = False) -> np.ndarray:
        return np.load(self.fetch(rel), mmap_mode="r" if mmap else None)

    def load_frame(self, rel: str) -> pd.DataFrame:
        return pd.read_parquet(self.fetch(rel))

    # ---------------------------------------------------------------- lifecycle
    def finish(self, info: dict | None = None) -> None:
        """Mark the stage complete (after uploading anything a crash left local-only)."""
        if self.ck.s3 is not None:
            remote = self.remote()
            for q in self.dir.rglob("*"):
                r = q.relative_to(self.dir).as_posix()
                if q.is_file() and r != DONE and r not in remote and not q.name.endswith((".tmp", ".part")):
                    self.commit(r)
        self.save_json(DONE, info or {})

    def drop_local(self, rel: str) -> None:
        """Free local disk: delete a local copy that is safely on S3 (kept when there is no S3)."""
        p = self.path(rel)
        if self.ck.s3 is None or not p.exists():
            return
        files = [p] if p.is_file() else [q for q in p.rglob("*") if q.is_file()]
        if all(q.relative_to(self.dir).as_posix() in self.remote() for q in files):
            shutil.rmtree(p) if p.is_dir() else p.unlink()

In [ ]:
#
# ▌2.12 Orchestration (`pipeline.py`)
#
# Resumable steps with S3-mirrored checkpoints. The notebook calls them one by one below.

In [ ]:
%%writefile {PKG}/src/ber/pipeline.py
"""End-to-end orchestration as resumable steps (used by run.py and the notebook).

TRAIN (dataset/train only)                                        checkpoint stage
  train_load        read ground truth + sources (or restore frames)
  train_normalize   static normalisation + locality phrases          static
  train_split       hold-out / dense / fit / ranker entity split,     (cheap, recomputed)
                    K folds
  train_aliases     aliases + noise tokens from training labels only  prep
  train_finalize    derived views, genericness stats, locality vocab  prep
  train_dense       bi-encoder fine-tuned on the dense entities'      dense  (per part)
                    true pairs (GPU), embeddings of every record
  train_blocking    candidate union per (country, source) block,      block  (per block)
                    pruner trained on the ranker entities' union
                    rows, then every block pruned
  train_stage1      features for fit + hold-out rows as on-disk       stage1 (per part / fold)
                    parts, grouped-OOF LightGBM, fold-average p1
                    for every other candidate (competitors)
  train_cross_encoder  optional mDeBERTa on uncertain pairs (GPU)     ce     (per half)
  train_stage2      context (+ ownership) features + grouped-OOF       stage2 (per fold)
                    LightGBM; fold-average p2 for competitor rows
  train_stage3      context recomputed from p2 + grouped-OOF LightGBM stage3 (per fold)
  train_decision    final probability (stage 2, 3 or their mean) and  decision
                    decision rule chosen on OOF over the whole
                    candidate graph, honest hold-out score

PREDICT (any split, e.g. test): the same transforms with the frozen
artefacts, checkpointed in one stage per (split, artefacts).

Checkpoints (checkpoint.py): each stage's hash covers the data, the settings
the stage depends on and its upstream stages. Every completed file is uploaded
to S3 immediately, so a resumed session - on the same instance or a new one -
restores the finished work and continues from the first missing file.
"""
from __future__ import annotations

import gc
import hashlib
import json
import logging
import shutil
import time
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace

import lightgbm as lgb
import numpy as np
import pandas as pd

from . import cross_encoder as ce
from . import dense
from .aliases import learn_aliases
from .blocking import COS_COLS, VIA_SPARSE, block_keys, block_union, fit_ranker, prune_candidates
from .checkpoint import Checkpoints, fingerprint, link_or_copy
from .config import Config
from .decision import decide, macro_fbeta, tune_decision
from .features import add_record_stats, context_features, group_features, stage1_matrix, stage3_context
from .io import read_ground_truth, read_split, write_id_lists
from .models import PartsMatrix, load_models, predict_avg, train_oof
from .normalize import finalize_frame, join_phrases, learn_phrases, normalize_frame

log = logging.getLogger(__name__)

CKPT_VERSION = 2          # bump when a code change invalidates earlier checkpoints
PART_ROWS = 2_000_000     # candidate rows per on-disk feature part (checkpoint granularity)
EMB_ROWS = 2_000_000      # records per on-disk embedding part (checkpoint granularity)


class Timer:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t = time.time()
        log.info("[start] %s", self.name)

    def __exit__(self, exc_type, *a):
        log.info("[%s] %s after %.1fs", "FAILED" if exc_type else "done ", self.name, time.time() - self.t)


# --------------------------------------------------------------------------
# Shared helpers
# --------------------------------------------------------------------------
def static_normalize(raw: dict, n_jobs: int, phrase_min_count: int):
    S1 = normalize_frame(raw["s1"], n_jobs)
    parts = []
    for src in (2, 3):
        f = normalize_frame(raw[f"s{src}"], n_jobs)
        f["src"] = np.int8(src)
        parts.append(f)
    X = pd.concat(parts, ignore_index=True)
    phrases = learn_phrases(S1, phrase_min_count)
    join_phrases(S1, phrases, n_jobs)
    join_phrases(X, phrases, n_jobs)
    log.info("joined %d multi-word locality phrases", len(phrases))
    return S1, X


def finalize(S1s, Xs, name_alias, addr_alias, noise):
    S1 = finalize_frame(S1s, name_alias, addr_alias, noise)
    X = finalize_frame(Xs, name_alias, addr_alias, noise)
    X["src"] = Xs["src"].values.astype(np.int8)
    return S1, X


def _spans(n: int) -> list[tuple[int, int]]:
    return [(a, min(a + PART_ROWS, n)) for a in range(0, n, PART_ROWS)]


def _no(d: dict, *keys) -> dict:
    return {k: v for k, v in d.items() if k not in keys}


def _publish(art: Path, stage, rels: list[str]) -> None:
    """Expose checkpoint files of a stage in the artefacts folder."""
    for rel, src in zip(rels, stage.fetch_many(rels)):
        link_or_copy(src, art / Path(rel).name)


def _publish_dir(art: Path, stage, rel_dir: str) -> None:
    root = stage.fetch_dir(rel_dir)
    for f in root.rglob("*"):
        if f.is_file():
            link_or_copy(f, art / rel_dir / f.relative_to(root))


def _require_disk(path: Path, gb: float, what: str) -> None:
    """Fail BEFORE an expensive step when the local volume cannot hold its output (a full
    disk otherwise surfaces only when the result is saved, after hours of GPU work)."""
    Path(path).mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(path).free / 1e9
    log.info("free local disk before %s: %.1f GB", what, free)
    if free < gb:
        raise RuntimeError(f"only {free:.1f} GB free on the local volume - {what} needs at least {gb:.0f} GB. "
                           f"Enlarge the space's storage or delete unneeded local files (everything finished "
                           f"is on S3 and is restored from there).")


def _drop_embeddings(stg) -> None:
    """The record embeddings (~11 GB at full size) are only needed to build the candidate
    union; once it exists, keep them on S3 only."""
    for rel in ("emb_s1", "emb_x"):
        stg.drop_local(rel)


def _release_frames(st) -> None:
    """Free the normalised text frames (the largest fixed memory) while LightGBM
    trains; they are reloaded from the prep checkpoint afterwards."""
    st.S1 = st.X = None
    gc.collect()


def _reload_frames(st) -> None:
    if st.S1 is None:
        prep = st.stg["prep"]
        st.S1, st.X = prep.load_frame("s1_final.parquet"), prep.load_frame("x_final.parquet")


def _ce_texts(frame: pd.DataFrame, idx: np.ndarray) -> list[str]:
    return ce.pair_text(frame["raw_name"].values[idx], frame["raw_addr"].values[idx])


def _ce_score(model_dirs, cand, S1, X, rows, cfg) -> np.ndarray:
    ta = _ce_texts(S1, cand["s1_idx"].values[rows])
    tb = _ce_texts(X, cand["x_idx"].values[rows])
    return np.mean([ce.score_pairs(d, ta, tb, cfg) for d in model_dirs], axis=0)


def _embed_parts(stg, name: str, model_dir: Path, texts: list[str], dc) -> np.ndarray:
    """Embeddings of ``texts``, encoded and checkpointed EMB_ROWS at a time."""
    spans = [(a, min(a + EMB_ROWS, len(texts))) for a in range(0, len(texts), EMB_ROWS)]
    parts = []
    for i, (a, b) in enumerate(spans):
        rel = f"{name}/part_{i:04d}.npy"
        if not stg.has(rel):
            stg.save_npy(rel, dense.encode(model_dir, texts[a:b], dc))
            log.info("embedding part %s saved (%d / %d records)", rel, b, len(texts))
        parts.append(stg.load_npy(rel))
    return np.concatenate(parts) if parts else np.zeros((0, 1), np.float16)


def _require_embeddings(cfg: Config, emb, step: str) -> None:
    """A block built without the embeddings would silently get cos_dense = 0 while the
    models expect the dense view - refuse instead of producing skewed candidates."""
    if cfg.dense.enabled and emb is None:
        raise RuntimeError(f"the dense view is enabled but the record embeddings are not loaded: "
                           f"run {step} before building candidates (in the notebook: the cell before this one)")


def _ce_usable(cfg: Config) -> bool:
    if not cfg.cross_encoder.enabled:
        return False
    if not ce.gpu_available():
        log.warning("cross-encoder requested but no CUDA GPU (or torch) found - skipping it")
        return False
    return True


def write_outputs(out_dir: Path, S1: pd.DataFrame, X: pd.DataFrame, cand: pd.DataFrame,
                  selected: np.ndarray, p: np.ndarray) -> dict:
    s1_ids = np.asarray(S1["entity_id"].values, dtype=object)
    x_ids = np.asarray(X["entity_id"].values, dtype=object)
    c = cand[["s1_idx", "x_idx", "block_score"]].copy()
    c["p"] = p
    c["sel"] = selected
    cands = (c.sort_values(["s1_idx", "block_score"], ascending=[True, False])
             .groupby("s1_idx")["x_idx"].apply(lambda v: list(x_ids[v.values])))
    matches = (c[c["sel"]].sort_values(["s1_idx", "p"], ascending=[True, False])
               .groupby("s1_idx")["x_idx"].apply(lambda v: list(x_ids[v.values])))
    cand_map = {s1_ids[k]: v for k, v in cands.items()}
    match_map = {s1_ids[k]: v for k, v in matches.items()}
    write_id_lists(Path(out_dir) / "candidate_pairs.tsv",
                   ("source1_entity_id", "candidate_entity_ids"), s1_ids, cand_map)
    write_id_lists(Path(out_dir) / "matching_results.tsv",
                   ("source1_entity_id", "matched_entity_ids"), s1_ids, match_map)
    summary = {"s1_entities": int(len(s1_ids)),
               "s1_with_matches": int(len(match_map)),
               "matched_ids": int(sum(len(v) for v in match_map.values())),
               "candidate_ids": int(len(cand))}
    log.info("outputs: %s", summary)
    return summary


# --------------------------------------------------------------------------
# TRAIN
# --------------------------------------------------------------------------
def _train_files(data_dir: Path) -> dict:
    files = [data_dir / "train" / f"train_source{k}.tsv" for k in (1, 2, 3)]
    files.append(data_dir / "train" / "train_ground_truth.tsv")
    return {f.name: f.stat().st_size for f in files if f.exists()}


def stage_fingerprints(cfg: Config, data_dir: Path) -> dict:
    """What each training stage depends on (speed / memory settings left out).
    The cross-encoder, stage-2 and decision stages are added as they run."""
    mc = cfg.model
    static = {"v": CKPT_VERSION, "data": _train_files(Path(data_dir)), "geo_min_count": cfg.geo_min_count}
    prep = {"static": fingerprint(static), "seed": mc.seed, "holdout_frac": mc.holdout_frac,
            "alias_min_count": cfg.alias_min_count}
    dns = {"prep": fingerprint(prep), "enabled": cfg.dense.enabled}
    if cfg.dense.enabled:
        dns.update(dense=_no(asdict(cfg.dense), "infer_batch_size", "enabled"), dense_frac=mc.dense_frac)
    block = {"prep": fingerprint(prep), "dense": fingerprint(dns),
             "blocking": _no(asdict(cfg.blocking), "chunk_rows"), "ranker_s1": mc.ranker_s1}
    stage1 = {"block": fingerprint(block), "max_train_s1": mc.max_train_s1, "n_folds": mc.n_folds,
              "lgb": _no(mc.lgb_stage1, "num_threads"), "rounds": mc.num_boost_round,
              "early_stopping": mc.early_stopping}
    return {"static": static, "prep": prep, "dense": dns, "block": block, "stage1": stage1}


def train_init(data_dir, work_dir, cfg: Config, use_cache: bool = True, ckpt_uri: str | None = None,
               s3_region: str | None = None, s3_endpoint: str | None = None,
               legacy_work_dir=None) -> SimpleNamespace:
    """Training state. ``ckpt_uri`` (s3://...) mirrors every checkpoint to S3;
    ``use_cache=False`` recomputes everything. ``legacy_work_dir`` lets a run of
    the earlier local-cache layout donate its normalised frames when they were
    built from the same data and settings."""
    if cfg.dense.enabled and not ce.gpu_available():
        # training the encoder and embedding ~12M records needs a GPU; decided before the
        # checkpoint keys are computed, so the keys always describe what actually runs
        log.warning("dense view requested but no CUDA GPU (or torch) found - blocking without it")
        cfg.dense.enabled = False
    st = SimpleNamespace(cfg=cfg, data_dir=Path(data_dir), work=Path(work_dir), use_cache=use_cache,
                         metrics={}, S1=None, X=None, S1s=None, Xs=None, raw=None, emb=None)
    st.art = st.work / "artifacts"
    if st.art.exists():                       # rebuilt from the checkpoints of this configuration
        shutil.rmtree(st.art)
    st.art.mkdir(parents=True)
    st.ck = Checkpoints(st.work / "ckpt", ckpt_uri, s3_region, s3_endpoint, enabled=use_cache)
    st.stg = {k: st.ck.stage(k, fp) for k, fp in stage_fingerprints(cfg, st.data_dir).items()}
    cfg.save(st.art / "config.json")
    if legacy_work_dir is not None and use_cache:
        _adopt_legacy_cache(st, Path(legacy_work_dir))
    log.info("checkpoints: %s", st.ck.s3_uri or st.ck.root)
    for name, stg in st.stg.items():
        log.info("  %-8s %s  %s", name, stg.key, "complete" if stg.done else "to do")
    return st


def _adopt_legacy_cache(st, legacy: Path) -> None:
    """Reuse the normalised frames of a run made with the earlier local-cache layout."""
    prep = st.stg["prep"]
    fp_path = legacy / "cache" / "fingerprint.json"
    files = {"s1_final.parquet": legacy / "cache" / "s1_final.parquet",
             "x_final.parquet": legacy / "cache" / "x_final.parquet",
             "geo.pkl": legacy / "cache" / "geo.pkl",
             "aliases.json": legacy / "artifacts" / "aliases.json"}
    if prep.done or not fp_path.exists() or not all(p.exists() for p in files.values()):
        return
    mc = st.cfg.model
    try:                                      # an unreadable / other-layout fingerprint: just don't reuse
        old = json.loads(fp_path.read_text())
        same = (old.get("data") == _train_files(st.data_dir) and old["model"]["seed"] == mc.seed
                and old["model"]["holdout_frac"] == mc.holdout_frac
                and old.get("alias_min_count") == st.cfg.alias_min_count
                and old.get("geo_min_count") == st.cfg.geo_min_count)
    except (ValueError, KeyError, TypeError, AttributeError):
        same = False
    if not same:
        log.info("earlier cache in %s was built with other data or settings - not reused", legacy)
        return
    log.info("reusing the normalised frames of the earlier run in %s", legacy)
    for rel, src in files.items():
        link_or_copy(src, prep.path(rel))
        prep.commit(rel)
    prep.finish({"adopted_from": str(legacy)})


def train_load(st) -> None:
    with Timer("read training data"):
        st.gt = read_ground_truth(st.data_dir / "train" / "train_ground_truth.tsv")
        prep, static = st.stg["prep"], st.stg["static"]
        if prep.done:
            st.S1 = prep.load_frame("s1_final.parquet")
            st.X = prep.load_frame("x_final.parquet")
            st.geo = prep.load_pickle("geo.pkl")
            _publish(st.art, prep, ["aliases.json"])
            for rel in ("s1_static.parquet", "x_static.parquet"):   # superseded by the final frames
                static.drop_local(rel)
            log.info("restored normalised frames from checkpoint (%d S1, %d S2+S3)", len(st.S1), len(st.X))
        elif static.done:
            st.S1s = static.load_frame("s1_static.parquet")
            st.Xs = static.load_frame("x_static.parquet")
            log.info("restored statically normalised frames from checkpoint")
        else:
            st.raw = read_split(st.data_dir, "train")


def train_normalize(st) -> None:
    if st.S1 is not None or st.S1s is not None:
        return
    with Timer("static normalisation"):
        st.S1s, st.Xs = static_normalize(st.raw, st.cfg.n_jobs, st.cfg.geo_min_count)
        st.raw = None
        stg = st.stg["static"]
        stg.save_frame("s1_static.parquet", st.S1s)
        stg.save_frame("x_static.parquet", st.Xs)
        stg.finish()


def train_split(st) -> None:
    mc = st.cfg.model
    S1 = st.S1 if st.S1 is not None else st.S1s
    X = st.X if st.X is not None else st.Xs
    n1 = len(S1)
    rng = np.random.default_rng(mc.seed)
    perm = rng.permutation(n1)
    n_hold = int(round(mc.holdout_frac * n1))
    st.is_hold = np.zeros(n1, bool)
    st.is_hold[perm[:n_hold]] = True
    train_pos = perm[n_hold:]
    # Dense entities: their true pairs train the bi-encoder and nothing else is fitted on them
    # (their embedding cosines are optimistic). Their candidates are still generated and
    # scored as competitors, exactly like the test set's.
    n_dense = int(round(mc.dense_frac * len(train_pos))) if st.cfg.dense.enabled else 0
    st.is_dense = np.zeros(n1, bool)
    st.is_dense[train_pos[:n_dense]] = True
    train_pos = train_pos[n_dense:]
    n_fit = min(len(train_pos), mc.max_train_s1 or len(train_pos))
    fit_pos = train_pos[:n_fit]
    # Ranker entities: the tail of the permutation - independent of max_train_s1
    # (so the blocking checkpoint survives a profile change) and disjoint from
    # the fit entities whenever enough entities are left over.
    ranker_pos = train_pos[-mc.ranker_s1:] if mc.ranker_s1 > 0 else train_pos[:0]
    st.is_fit = np.zeros(n1, bool)
    st.is_fit[fit_pos] = True
    st.is_ranker = np.zeros(n1, bool)
    st.is_ranker[ranker_pos] = True
    st.fold_s1 = np.full(n1, -1, np.int64)
    st.fold_s1[fit_pos] = np.arange(len(fit_pos)) % mc.n_folds

    s1_index = pd.Index(np.asarray(S1["entity_id"].values, dtype=object))
    x_index = pd.Index(np.asarray(X["entity_id"].values, dtype=object))
    st.gt_s1 = np.asarray([s for s, lst in st.gt.items() for _ in lst], dtype=object)
    st.gt_x = np.asarray([x for _, lst in st.gt.items() for x in lst], dtype=object)
    gi = s1_index.get_indexer(st.gt_s1)
    gx = x_index.get_indexer(st.gt_x)
    st.gt_ok = (gi >= 0) & (gx >= 0)
    st.gt_gi = gi
    st.owner = np.full(len(X), -1, np.int64)
    st.owner[gx[st.gt_ok]] = gi[st.gt_ok]
    st.n_gt = np.bincount(gi[gi >= 0], minlength=n1).astype(np.int64)
    st.metrics.update(n_s1=int(n1), n_x=int(len(X)), singleton_rate=float((st.n_gt == 0).mean()),
                      n_fit_s1=int(st.is_fit.sum()), n_holdout_s1=int(st.is_hold.sum()),
                      n_ranker_s1=int(st.is_ranker.sum()), n_dense_s1=int(st.is_dense.sum()))
    log.info("entities: fit=%d hold-out=%d ranker=%d dense=%d (of %d)", st.is_fit.sum(),
             st.is_hold.sum(), st.is_ranker.sum(), st.is_dense.sum(), n1)


def train_aliases(st) -> None:
    if st.S1 is not None:
        return
    with Timer("learn aliases"):
        keep = st.gt_ok & ~st.is_hold[np.clip(st.gt_gi, 0, None)]
        pairs = pd.DataFrame({"s1_id": st.gt_s1[keep], "x_id": st.gt_x[keep]})
        st.name_alias, st.addr_alias, st.noise = learn_aliases(
            st.S1s, st.Xs, pairs, st.cfg.alias_min_count, seed=st.cfg.model.seed)
        (st.art / "aliases.json").write_text(json.dumps(
            {"name_alias": st.name_alias, "addr_alias": st.addr_alias,
             "noise_tokens": sorted(st.noise)}, indent=1))


def train_finalize(st) -> None:
    if st.S1 is not None:
        return
    with Timer("finalise normalisation"):
        st.S1, st.X = finalize(st.S1s, st.Xs, st.name_alias, st.addr_alias, st.noise)
        st.S1s = st.Xs = None
        st.geo = add_record_stats(st.S1, st.X, st.cfg.geo_min_count)
        prep = st.stg["prep"]
        prep.save_frame("s1_final.parquet", st.S1)
        prep.save_frame("x_final.parquet", st.X)
        prep.save_pickle("geo.pkl", st.geo)
        link_or_copy(st.art / "aliases.json", prep.path("aliases.json"))
        prep.commit("aliases.json")
        prep.finish()
        for rel in ("s1_static.parquet", "x_static.parquet"):       # superseded by the final frames
            st.stg["static"].drop_local(rel)


def train_dense(st) -> None:
    """Fine-tune the bi-encoder on the dense entities' true pairs and embed every record.

    The embeddings are only needed to build the candidate union: when the
    blocking stage, or every block's union, is already complete they are not
    loaded at all.
    """
    dc, mc, stg = st.cfg.dense, st.cfg.model, st.stg["dense"]
    st.emb = None
    if not dc.enabled:
        return
    if not stg.has("dense_encoder.ok"):
        xs = np.flatnonzero(st.owner >= 0)
        xs = xs[st.is_dense[st.owner[xs]]]
        if len(xs) > dc.max_train_pairs:
            xs = np.sort(np.random.default_rng(mc.seed).choice(xs, dc.max_train_pairs, replace=False))
        s1s = st.owner[xs]
        with Timer(f"dense encoder training ({len(xs):,} true pairs of {st.is_dense.sum():,} entities)"):
            dense.train_encoder(dense.record_text(st.S1, s1s), dense.record_text(st.X, xs), s1s,
                                st.S1["country"].values[s1s], dc, stg.path("dense_encoder"), mc.seed)
        stg.commit("dense_encoder")
        stg.save_json("dense_encoder.ok", {"pairs": int(len(xs))})   # marks the encoder complete
    enc = stg.fetch_dir("dense_encoder")
    blk = st.stg["block"]
    if not blk.done and not all(blk.has(f"union/{t}.parquet") for *_, t in block_keys(st.S1, st.X)):
        with Timer("dense embeddings of every record"):
            st.emb = (_embed_parts(stg, "emb_s1", enc, dense.record_text(st.S1), dc),
                      _embed_parts(stg, "emb_x", enc, dense.record_text(st.X), dc))
        if not stg.done:
            stg.finish({"dim": int(st.emb[0].shape[1])})
    _publish_dir(st.art, stg, "dense_encoder")


def _read_cand(stg, tags: list[str]) -> pd.DataFrame:
    paths = stg.fetch_many([f"cand/{t}.parquet" for t in tags])
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)


def train_blocking(st) -> None:
    cfg, mc, stg = st.cfg, st.cfg.model, st.stg["block"]
    if stg.done:
        st.cand = _read_cand(stg, stg.load_json("blocks.json"))
        st.metrics.update(stg.load_json("blocking_metrics.json"))
        _publish(st.art, stg, ["block_ranker.txt"])
        _drop_embeddings(st.stg["dense"])
        log.info("restored %d candidate pairs from checkpoint", len(st.cand))
        return
    keys = block_keys(st.S1, st.X)
    tags = [t for *_, t in keys]
    eval_mask = st.is_hold if st.is_hold.any() else st.is_fit

    # 1) unpruned candidate union of every block (one checkpoint per block)
    with Timer(f"candidate union for all S1 entities ({len(keys)} blocks)"):
        for country, src, tag in keys:
            if stg.has(f"union/{tag}.parquet"):
                log.info("block %s / S%d restored from checkpoint", country, src)
                continue
            _require_embeddings(cfg, st.emb, "train_dense")
            stg.save_frame(f"union/{tag}.parquet",
                           block_union(st.S1, st.X, country, src, cfg.blocking, cfg.n_jobs, emb=st.emb))
    st.emb = None                             # embeddings are only used to build the union
    _drop_embeddings(st.stg["dense"])
    gc.collect()

    # 2) pruner trained on the union rows of the ranker entities: exactly the
    #    candidate distribution it will prune
    with Timer("blocking pruner"):
        if stg.has("block_ranker.txt"):
            ranker = lgb.Booster(model_file=str(stg.fetch("block_ranker.txt")))
        else:
            parts = []
            for tag in tags:
                u = stg.load_frame(f"union/{tag}.parquet")
                parts.append(u[st.is_ranker[u["s1_idx"].values]])
            union = pd.concat(parts, ignore_index=True)
            del parts
            yu = st.owner[union["x_idx"].values] == union["s1_idx"].values
            log.info("pruner training pairs: %d (%d true)", len(union), int(yu.sum()))
            ranker = fit_ranker(union, st.X, yu, seed=mc.seed)
            stg.save_model("block_ranker.txt", ranker)
            del union, yu

    # 3) prune every block (one checkpoint per block)
    with Timer("prune the candidate union of every block"):
        for country, src, tag in keys:
            if stg.has(f"cand/{tag}.json"):
                continue
            u = stg.load_frame(f"union/{tag}.parquet")
            hit = (st.owner[u["x_idx"].values] == u["s1_idx"].values) & eval_mask[u["s1_idx"].values]
            sparse = (u["via"].values & VIA_SPARSE) > 0
            info = {"union_pairs": int(len(u)), "union_pairs_sparse": int(sparse.sum()),
                    "union_true_eval": int(hit.sum()), "union_true_eval_dense_only": int((hit & ~sparse).sum())}
            block = prune_candidates(u, st.X, ranker, cfg.blocking.keep_per_source)
            del u, hit
            stg.save_frame(f"cand/{tag}.parquet", block)
            stg.save_json(f"cand/{tag}.json", info)     # written last: marks the block complete
            log.info("pruned %s / S%d: %d union -> %d kept", country, src, info["union_pairs"], len(block))

    st.cand = _read_cand(stg, tags)
    stats = [stg.load_json(f"cand/{t}.json") for t in tags]
    y = st.owner[st.cand["x_idx"].values] == st.cand["s1_idx"].values
    tot = st.n_gt[eval_mask].sum()
    n_union_true = sum(s["union_true_eval"] for s in stats)
    bm = {"union_pairs": int(sum(s["union_pairs"] for s in stats)), "cand_pairs": int(len(st.cand)),
          "cand_per_s1": float(len(st.cand) / len(st.S1)),
          "union_recall_eval": float(n_union_true / max(tot, 1)),
          # what the sparse views alone reach (the difference is the dense view's contribution)
          "union_recall_eval_sparse_only": float(
              (n_union_true - sum(s.get("union_true_eval_dense_only", 0) for s in stats)) / max(tot, 1)),
          "cand_recall_eval": float((y & eval_mask[st.cand["s1_idx"].values]).sum() / max(tot, 1))}
    st.metrics.update(bm)
    log.info("blocking: %s", bm)
    stg.save_json("blocks.json", tags)
    stg.save_json("blocking_metrics.json", bm)
    stg.finish(bm)
    for tag in tags:                          # the unions stay on S3; free the local disk
        stg.drop_local(f"union/{tag}.parquet")
    _publish(st.art, stg, ["block_ranker.txt"])


def train_stage1(st) -> None:
    cfg, mc, stg = st.cfg, st.cfg.model, st.stg["stage1"]
    cand = st.cand
    s1c = cand["s1_idx"].values
    st.y = (st.owner[cand["x_idx"].values] == s1c).astype(np.int8)
    fit_rows = np.flatnonzero(st.is_fit[s1c])
    hold_rows = np.flatnonzero(st.is_hold[s1c])
    st.model_rows = np.concatenate([fit_rows, hold_rows])   # fit rows FIRST
    st.n_fit_rows = len(fit_rows)
    st.fold_m = st.fold_s1[s1c[st.model_rows]]
    spans = _spans(len(st.model_rows))
    st.f1_parts = [f"F1/part_{i:04d}.npy" for i in range(len(spans))]
    arts = [f"stage1_fold{k}.txt" for k in range(mc.n_folds)] + ["stage1_cols.json", "stage1_importance.csv"]
    if stg.done:
        st.p1 = stg.load_npy("p1.npy")
        st.f1_cols = stg.load_json("stage1_cols.json")
        _publish(st.art, stg, arts)
        log.info("restored stage-1 predictions from checkpoint")
        return

    grp = None
    with Timer(f"stage-1 features for {len(st.model_rows):,} fit + hold-out rows ({len(spans)} parts)"):
        for rel, (a, b) in zip(st.f1_parts, spans):
            if stg.has(rel):
                continue
            if grp is None:
                grp = group_features(cand)
            cols = stg.load_json("stage1_cols.json") if stg.has("stage1_cols.json") else None
            F = stage1_matrix(cand, st.model_rows[a:b], st.S1, st.X, st.geo, grp, cfg.n_jobs, cols)
            if cols is None:
                stg.save_json("stage1_cols.json", list(F.columns))
            stg.save_npy(rel, F.to_numpy(np.float32))
            del F
            log.info("stage-1 feature part %s saved (%d / %d rows)", rel, b, len(st.model_rows))
    st.f1_cols = stg.load_json("stage1_cols.json")

    grp = None
    _release_frames(st)
    M = PartsMatrix(stg.fetch_many(st.f1_parts))
    with Timer("stage-1 LightGBM (grouped 5-fold OOF)"):
        models, p1m = train_oof(M, st.f1_cols, st.y[st.model_rows], st.fold_m, st.n_fit_rows,
                                mc.lgb_stage1, mc.n_folds, mc.num_boost_round,
                                mc.early_stopping, mc.seed, stg, "stage1")
    del M
    _reload_frames(st)
    st.p1 = np.full(len(cand), np.nan)
    st.p1[st.model_rows] = p1m
    other = np.flatnonzero(np.isnan(st.p1))
    if len(other):
        with Timer(f"stage-1 scoring of {len(other):,} competitor rows"):
            for i, (a, b) in enumerate(_spans(len(other))):
                rel = f"p1_other/part_{i:04d}.npy"
                if not stg.has(rel):
                    if grp is None:
                        grp = group_features(cand)
                    Fo = stage1_matrix(cand, other[a:b], st.S1, st.X, st.geo, grp, cfg.n_jobs, st.f1_cols)
                    stg.save_npy(rel, predict_avg(models, Fo.to_numpy(np.float32)))
                    del Fo
                st.p1[other[a:b]] = stg.load_npy(rel)
    stg.save_npy("p1.npy", st.p1)
    stg.finish()
    _publish(st.art, stg, arts)


def ce_stage(st):
    """The cross-encoder checkpoint stage of the current settings (also sets st.ce_enabled)."""
    ccfg = st.cfg.cross_encoder
    st.ce_enabled = _ce_usable(st.cfg)
    fp = {"stage1": st.stg["stage1"].hash, "enabled": st.ce_enabled}
    if st.ce_enabled:
        fp["ce"] = _no(asdict(ccfg), "infer_batch_size", "enabled", "num_workers")   # speed-only settings
    return st.ck.stage("ce", fp)


def missing_restores(st) -> list[str]:
    """Training steps up to stage 1 that are NOT restorable from the checkpoints (and would
    therefore be recomputed): the normalised frames, the dense encoder, blocking, stage 1."""
    need = {"prep": st.stg["prep"].done, "block": st.stg["block"].done, "stage1": st.stg["stage1"].done}
    if st.cfg.dense.enabled:                  # without the encoder train_dense retrains it
        need["dense"] = st.stg["dense"].has("dense_encoder.ok")
    return [k for k, ok in need.items() if not ok]


def train_cross_encoder(st) -> None:
    ccfg, mc = st.cfg.cross_encoder, st.cfg.model
    stg = st.stg["ce"] = ce_stage(st)
    st.ce_m = np.full(len(st.model_rows), np.nan, dtype=np.float32)
    if not st.ce_enabled:
        return
    if stg.done:
        st.ce_m = stg.load_npy("ce_m.npy")
        for h in (0, 1):
            _publish_dir(st.art, stg, f"ce_{h}")
        log.info("restored cross-encoder scores and models from checkpoint")
        return
    with Timer("cross-encoder (2-fold over fit entities)"):
        rows_all = st.model_rows
        p1m = st.p1[rows_all]
        y_m = st.y[rows_all]
        band = (p1m >= ccfg.band_low) & (p1m <= ccfg.band_high)
        is_fit_row = np.arange(len(rows_all)) < st.n_fit_rows
        half = np.where(is_fit_row, st.fold_m % 2, -1)
        dirs = []
        for h in (0, 1):
            d_rel, s_rel = f"ce_{h}", f"ce_scores_{h}.npy"
            sc = np.flatnonzero(is_fit_row & (half == 1 - h) & band)
            if stg.has(f"{d_rel}.ok"):
                d = stg.fetch_dir(d_rel)
                log.info("cross-encoder half %d restored from checkpoint", h)
            else:
                if stg.has(f"{d_rel}.model.ok"):  # trained earlier, its scoring was interrupted
                    d = stg.fetch_dir(d_rel)
                    log.info("cross-encoder half %d model restored from checkpoint - scoring it", h)
                else:
                    # the saved model + tokenizer take ~1.2 GB; check BEFORE hours of GPU training
                    _require_disk(stg.dir, 5, f"cross-encoder half {h}")
                    tr = is_fit_row & (half == h)
                    pick = ce.sample_training_pairs(np.flatnonzero(tr & band), np.flatnonzero(tr & ~band),
                                                    y_m, ccfg.max_train_pairs, mc.seed + h)
                    d = ce.train_cross_encoder(_ce_texts(st.S1, st.cand["s1_idx"].values[rows_all[pick]]),
                                               _ce_texts(st.X, st.cand["x_idx"].values[rows_all[pick]]),
                                               y_m[pick].astype(np.float32), ccfg, stg.path(d_rel), mc.seed + h)
                    stg.commit(d_rel)
                    # model files uploaded first, then this marker: a crash while scoring keeps the model
                    stg.save_json(f"{d_rel}.model.ok", {"pairs": int(len(pick))})
                stg.save_npy(s_rel, _ce_score([d], st.cand, st.S1, st.X, rows_all[sc], ccfg))
                stg.save_json(f"{d_rel}.ok", stg.load_json(f"{d_rel}.model.ok"))   # marks the half complete
            st.ce_m[sc] = stg.load_npy(s_rel)
            dirs.append(d)
        sc = np.flatnonzero(~is_fit_row & band)
        if not stg.has("ce_hold.npy"):
            stg.save_npy("ce_hold.npy", _ce_score(dirs, st.cand, st.S1, st.X, rows_all[sc], ccfg))
        st.ce_m[sc] = stg.load_npy("ce_hold.npy")
    stg.save_npy("ce_m.npy", st.ce_m)
    stg.finish()
    for h in (0, 1):
        _publish_dir(st.art, stg, f"ce_{h}")


CTX_VERSION = 2     # context-feature layout: 2 = + collective ownership / same-name rivalry features


def _other_rows(st) -> np.ndarray:
    """Candidate rows outside fit + hold-out (competitors only: the dense slice's entities,
    entities beyond max_train_s1)."""
    in_model = np.zeros(len(st.cand), bool)
    in_model[st.model_rows] = True
    return np.flatnonzero(~in_model)


def _ctx_matrix(ctx: pd.DataFrame, cols: list[str]) -> np.ndarray:
    """Context columns in the trained order (a newer library may compute extra columns)."""
    missing = [c for c in cols if c not in ctx.columns]
    assert not missing, f"context features missing: {missing}"
    return ctx[cols].to_numpy(np.float32)


def train_stage2(st) -> None:
    mc = st.cfg.model
    fp = {"ce": st.stg["ce"].hash, "n_folds": mc.n_folds, "lgb": _no(mc.lgb_stage2, "num_threads"),
          "rounds": mc.num_boost_round, "early_stopping": mc.early_stopping, "n_anchors": mc.n_anchors,
          "ctx": CTX_VERSION, "competitors": "p2"}
    stg = st.stg["stage2"] = st.ck.stage("stage2", fp)
    arts = [f"stage2_fold{k}.txt" for k in range(mc.n_folds)] + ["stage2_importance.csv"]
    if stg.done:
        st.p2m = stg.load_npy("p2m.npy")
        st.f2_cols = stg.load_json("stage2_cols.json")
        _publish(st.art, stg, arts)
        log.info("restored stage-2 predictions from checkpoint")
        comp = _competitor_p2(st, stg)        # restored from the checkpoint
    else:
        if stg.has("ctx.npy") and stg.has("ctx_cols.json"):
            C, ctx_cols = stg.load_npy("ctx.npy"), stg.load_json("ctx_cols.json")
        else:
            with Timer("context features"):
                ctx = context_features(st.cand, st.p1, st.X, mc.n_anchors, rows=st.model_rows, S1=st.S1)
                ctx["ce"] = st.ce_m
                ctx_cols, C = list(ctx.columns), ctx.to_numpy(np.float32)
                del ctx
                stg.save_npy("ctx.npy", C)
                stg.save_json("ctx_cols.json", ctx_cols)
        st.f2_cols = st.f1_cols + ctx_cols
        _release_frames(st)
        M = PartsMatrix(st.stg["stage1"].fetch_many(st.f1_parts), extra=C)
        with Timer("stage-2 LightGBM (grouped 5-fold OOF)"):
            _, st.p2m = train_oof(M, st.f2_cols, st.y[st.model_rows], st.fold_m, st.n_fit_rows,
                                  mc.lgb_stage2, mc.n_folds, mc.num_boost_round,
                                  mc.early_stopping, mc.seed + 100, stg, "stage2")
        del M, C
        _reload_frames(st)
        stg.save_npy("p2m.npy", st.p2m)
        stg.save_json("stage2_cols.json", st.f2_cols)
        _publish(st.art, stg, arts)           # the competitor scoring below loads these models
        # computed once and kept: without checkpoints (USE_CACHE=False) a second call would redo it
        comp = _competitor_p2(st, stg)
        stg.finish()
        if not mc.stage3:                     # stage 3 reads the stage-1 parts again
            _drop_stage1_parts(st)
            for rel in _f1o_parts(st):
                stg.drop_local(rel)
        stg.drop_local("ctx.npy")
    # p2 of EVERY candidate row, as at inference: out-of-fold for fit rows, fold average otherwise
    other, st.ce_other, p2o = comp
    st.p2_all = st.p1.copy()
    st.p2_all[st.model_rows] = st.p2m
    st.p2_all[other] = p2o


def _drop_stage1_parts(st) -> None:
    for rel in st.f1_parts:                   # stay on S3; free the local disk for inference
        st.stg["stage1"].drop_local(rel)


def _f1o_parts(st) -> list[str]:
    return [f"F1o/part_{i:04d}.npy" for i in range(len(_spans(len(_other_rows(st)))))]


def _competitor_p2(st, stg) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Stage-2 probabilities of the competitor-only rows, computed as at inference.

    Rows of entities outside fit / hold-out (the dense slice, entities beyond
    max_train_s1) get no p2 from the grouped OOF. At inference every row has the
    fold-average p2, so they get exactly that here: their stage-1 features, the
    stage-2 context from p1 (with the cross-encoder on their uncertain band) and the
    fold-average stage-2 models. Stage 3 and the decision step then see p2 for every
    competitor, as at inference. Checkpointed in the stage-2 stage; the stage-1
    feature parts of these rows (F1o) are kept for the stage-3 scoring.
    Returns (rows, cross-encoder scores, p2).
    """
    cfg, mc = st.cfg, st.cfg.model
    other = _other_rows(st)
    ce_o = np.full(len(other), np.nan, dtype=np.float32)
    if len(other) == 0:
        return other, ce_o, np.zeros(0)
    if st.ce_enabled:
        ccfg = cfg.cross_encoder
        band = np.flatnonzero((st.p1[other] >= ccfg.band_low) & (st.p1[other] <= ccfg.band_high))
        if not stg.has("ce_other.npy"):
            with Timer(f"cross-encoder scoring of {len(band):,} competitor rows"):
                dirs = [st.stg["ce"].fetch_dir(f"ce_{h}") for h in (0, 1)]
                stg.save_npy("ce_other.npy", _ce_score(dirs, st.cand, st.S1, st.X, other[band], ccfg))
        ce_o[band] = stg.load_npy("ce_other.npy")
    if stg.has("p2_other.npy"):
        return other, ce_o, stg.load_npy("p2_other.npy")
    with Timer(f"stage-2 scoring of {len(other):,} competitor rows"):
        ctx = context_features(st.cand, st.p1, st.X, mc.n_anchors, rows=other, S1=st.S1)
        ctx["ce"] = ce_o
        assert st.f1_cols + list(ctx.columns) == st.f2_cols, "feature order mismatch"
        C = ctx.to_numpy(np.float32)
        del ctx
        spans = _spans(len(other))
        rels = _f1o_parts(st)
        grp = None
        for rel, (a, b) in zip(rels, spans):
            if stg.has(rel):
                continue
            if grp is None:
                grp = group_features(st.cand)
            F = stage1_matrix(st.cand, other[a:b], st.S1, st.X, st.geo, grp, cfg.n_jobs, st.f1_cols)
            stg.save_npy(rel, F.to_numpy(np.float32))
            del F
        del grp
        p2o = predict_avg(load_models(st.art, "stage2", mc.n_folds), PartsMatrix(stg.fetch_many(rels), extra=C))
    stg.save_npy("p2_other.npy", p2o)
    return other, ce_o, p2o


def train_stage3(st) -> None:
    """Collective refinement: stage-1 features + the context recomputed from p2.

    p2 is out-of-fold for the fit rows and the fold average for every other row
    (st.p2_all, built by train_stage2), exactly as at inference. After training,
    the competitor-only rows are scored by the fold-average stage-3 models too, so
    the decision step sees the final probability of every candidate row.
    """
    mc = st.cfg.model
    st.p3m, st.f3_cols, st.p3_all = None, None, None
    if not mc.stage3:
        return
    fp = {"stage2": st.stg["stage2"].hash, "n_folds": mc.n_folds, "lgb": _no(mc.lgb_stage3, "num_threads"),
          "rounds": mc.num_boost_round, "early_stopping": mc.early_stopping, "n_anchors": mc.n_anchors,
          "competitors": "p2+p3"}
    stg = st.stg["stage3"] = st.ck.stage("stage3", fp)
    arts = [f"stage3_fold{k}.txt" for k in range(mc.n_folds)] + ["stage3_importance.csv"]
    other = _other_rows(st)
    if stg.done:
        st.p3m = stg.load_npy("p3m.npy")
        st.f3_cols = stg.load_json("stage3_cols.json")
        _publish(st.art, stg, arts)
        log.info("restored stage-3 predictions from checkpoint")
    else:
        if stg.has("ctx.npy") and stg.has("ctx_cols.json"):
            C, ctx_cols = stg.load_npy("ctx.npy"), stg.load_json("ctx_cols.json")
        else:
            with Timer("stage-3 context features (from p2)"):
                ctx = stage3_context(st.cand, st.p2_all, st.X, mc.n_anchors, rows=st.model_rows, S1=st.S1)
                ctx["ce"] = st.ce_m
                ctx_cols, C = list(ctx.columns), ctx.to_numpy(np.float32)
                del ctx
                stg.save_npy("ctx.npy", C)
                stg.save_json("ctx_cols.json", ctx_cols)
        st.f3_cols = st.f1_cols + ctx_cols
        _release_frames(st)
        M = PartsMatrix(st.stg["stage1"].fetch_many(st.f1_parts), extra=C)
        with Timer("stage-3 LightGBM (grouped 5-fold OOF)"):
            _, st.p3m = train_oof(M, st.f3_cols, st.y[st.model_rows], st.fold_m, st.n_fit_rows,
                                  mc.lgb_stage3, mc.n_folds, mc.num_boost_round,
                                  mc.early_stopping, mc.seed + 200, stg, "stage3")
        del M, C
        _reload_frames(st)
        stg.save_npy("p3m.npy", st.p3m)
        stg.save_json("stage3_cols.json", st.f3_cols)
        _publish(st.art, stg, arts)
        if len(other) and not stg.has("p3_other.npy"):
            with Timer(f"stage-3 scoring of {len(other):,} competitor rows"):
                ctx = stage3_context(st.cand, st.p2_all, st.X, mc.n_anchors, rows=other, S1=st.S1)
                ctx["ce"] = st.ce_other
                C = _ctx_matrix(ctx, st.f3_cols[len(st.f1_cols):])
                del ctx
                parts = st.stg["stage2"].fetch_many(_f1o_parts(st))
                stg.save_npy("p3_other.npy", predict_avg(load_models(st.art, "stage3", mc.n_folds),
                                                         PartsMatrix(parts, extra=C)))
                del C
        stg.finish()
        _drop_stage1_parts(st)
        for rel in _f1o_parts(st):            # stay on S3; free the local disk
            st.stg["stage2"].drop_local(rel)
        stg.drop_local("ctx.npy")
    st.p3_all = st.p2_all.copy()
    st.p3_all[st.model_rows] = st.p3m
    if len(other):
        st.p3_all[other] = stg.load_npy("p3_other.npy")


def train_decision(st) -> dict:
    """Tune the decision rule on out-of-fold predictions of each final-probability
    candidate (stage 2; stage 3 and the stage-2/3 mean when stage 3 was trained),
    keep the best one, then score the hold-out once with it."""
    dc = st.cfg.decision
    has3 = getattr(st, "p3_all", None) is not None
    stg = st.stg["decision"] = st.ck.stage("decision", {
        "stage2": st.stg["stage2"].hash, "stage3": st.stg["stage3"].hash if has3 else None,
        "decision": asdict(dc), "graph": "all candidate rows"})
    # final-probability candidates over EVERY candidate row (23 = mean of stages 2 and 3)
    probs = {2: st.p2_all, **({3: st.p3_all, 23: (st.p2_all + st.p3_all) / 2} if has3 else {})}
    s1a = st.cand["s1_idx"].values
    xa = st.cand["x_idx"].values
    nf = st.n_fit_rows
    fit_rows, hold_rows = st.model_rows[:nf], st.model_rows[nf:]
    fit_ents = np.flatnonzero(st.is_fit)
    hold_ents = np.flatnonzero(st.is_hold)
    if stg.done:
        st.metrics.update(stg.load_json("metrics.json"))
        meta = stg.load_json("meta.json")
        st.decision, st.final_stage = meta["decision"], meta.get("final_stage", 2)
        st.decision_grid = [tuple(r) for r in stg.load_json("grid.json")]
        st.sel_full = stg.load_npy("sel_full.npy")
        _publish(st.art, stg, ["meta.json", "metrics.json"])
        log.info("restored the decision rule and metrics from checkpoint")
    else:
        # The rule is applied to the whole candidate graph at once, exactly as on the test set
        # (probability sharing and exclusivity see every claimant of a record); it is scored on
        # the fit entities' out-of-fold rows and chosen on them only. Hold-out labels are never read.
        codes_f = np.searchsorted(fit_ents, s1a[fit_rows])
        y_f, n_gt_f = st.y[fit_rows], st.n_gt[fit_ents]

        def score_fit(sel):
            return macro_fbeta(codes_f, y_f, sel[fit_rows], n_gt_f, dc.beta)

        tuned = {}
        with Timer("decision tuning on out-of-fold predictions (whole candidate graph)"):
            for k, p in probs.items():
                tuned[k] = tune_decision(s1a, xa, p, dc, score_fit)
                st.metrics[f"oof_macro_f05_stage{k}"] = tuned[k][1]
        st.final_stage = max(tuned, key=lambda k: tuned[k][1])     # chosen on out-of-fold data only
        params, oof, grid = tuned[st.final_stage]
        st.decision, st.decision_grid = params, grid
        st.metrics.update(oof_macro_f05=oof, decision=params, final_stage=st.final_stage)
        log.info("final stage: %d (out-of-fold macro F0.5 %s)", st.final_stage,
                 {k: round(v[1], 5) for k, v in tuned.items()})
        st.sel_full = decide(s1a, xa, probs[st.final_stage], params, dc.beta, dc.max_items)
        if len(hold_ents):
            codes_h = np.searchsorted(hold_ents, s1a[hold_rows])
            y_h, n_gt_h = st.y[hold_rows], st.n_gt[hold_ents]
            for k, p in probs.items():        # reported for information; not used for any choice
                s = decide(s1a, xa, p, tuned[k][0], dc.beta, dc.max_items)
                st.metrics[f"holdout_macro_f05_stage{k}"] = macro_fbeta(codes_h, y_h, s[hold_rows], n_gt_h, dc.beta)
            sel = st.sel_full[hold_rows]
            tp = int((y_h * sel).sum())
            st.metrics.update(
                holdout_macro_f05=macro_fbeta(codes_h, y_h, sel, n_gt_h, dc.beta),
                holdout_macro_f05_stage1_t05=macro_fbeta(codes_h, y_h, st.p1[hold_rows] >= 0.5, n_gt_h, dc.beta),
                holdout_macro_f05_oracle=macro_fbeta(codes_h, y_h, y_h.astype(bool), n_gt_h, dc.beta),
                holdout_pair_precision=tp / max(int(sel.sum()), 1),
                holdout_pair_recall=tp / max(int(n_gt_h.sum()), 1))
        stg.save_npy("sel_full.npy", st.sel_full)
        meta = {"f1_cols": st.f1_cols, "f2_cols": st.f2_cols, "f3_cols": st.f3_cols if has3 else None,
                "final_stage": st.final_stage, "decision": st.decision,
                "ce_enabled": bool(st.ce_enabled), "n_folds": st.cfg.model.n_folds}
        stg.save_json("meta.json", meta)
        stg.save_json("metrics.json", st.metrics)
        stg.save_json("grid.json", [[s, p] for s, p in st.decision_grid])
        stg.finish()
        _publish(st.art, stg, ["meta.json", "metrics.json"])
        log.info("METRICS %s", json.dumps(st.metrics, default=float))
    # final probability / selection of the model rows (fit rows first, then hold-out), as the
    # notebook diagnostics expect them
    st.pfin_all = probs[st.final_stage]
    st.pfin = st.pfin_all[st.model_rows]
    st.holdout_sel = st.sel_full[hold_rows]
    return st.metrics


TRAIN_STEPS = [train_load, train_normalize, train_split, train_aliases, train_finalize, train_dense,
               train_blocking, train_stage1, train_cross_encoder, train_stage2, train_stage3,
               train_decision]


def train(data_dir, work_dir, cfg: Config, use_cache: bool = True, **ckpt) -> dict:
    st = train_init(data_dir, work_dir, cfg, use_cache, **ckpt)
    for step in TRAIN_STEPS:
        step(st)
    return st.metrics


# --------------------------------------------------------------------------
# PREDICT
# --------------------------------------------------------------------------
def _artifacts_id(art: Path) -> str:
    """Identity of a set of trained artefacts (content of small files, size of large ones)."""
    h = hashlib.sha1()
    for f in sorted(p for p in art.rglob("*") if p.is_file() and p.name != "metrics.json"):
        h.update(f.relative_to(art).as_posix().encode())
        size = f.stat().st_size
        h.update(f.read_bytes() if size < 64 * 2**20 else str(size).encode())
    return h.hexdigest()[:16]


def predict_init(data_dir, split: str, work_dir, out_dir, use_cache: bool = True,
                 ckpt_uri: str | None = None, s3_region: str | None = None,
                 s3_endpoint: str | None = None) -> SimpleNamespace:
    art = Path(work_dir) / "artifacts"
    ps = SimpleNamespace(data_dir=Path(data_dir), split=split, work=Path(work_dir),
                         out=Path(out_dir), art=art, cfg=Config.load(art / "config.json"),
                         meta=json.loads((art / "meta.json").read_text()),
                         aliases=json.loads((art / "aliases.json").read_text()))
    files = [ps.data_dir / split / f"{split}_source{k}.tsv" for k in (1, 2, 3)]
    ps.ck = Checkpoints(ps.work / "ckpt", ckpt_uri, s3_region, s3_endpoint, enabled=use_cache)
    ps.stg = ps.ck.stage(f"predict-{split}", {"v": CKPT_VERSION, "split": split,
                                               "data": {f.name: f.stat().st_size for f in files},
                                               "artifacts": _artifacts_id(art)})
    log.info("inference checkpoints: %s%s", ps.ck.s3_uri or f"{ps.ck.root}/", ps.stg.key)
    return ps


def predict_prepare(ps) -> None:
    cfg, al, stg = ps.cfg, ps.aliases, ps.stg
    if all(stg.has(f) for f in ("s1.parquet", "x.parquet", "geo.pkl")):
        ps.S1, ps.X = stg.load_frame("s1.parquet"), stg.load_frame("x.parquet")
        ps.geo = stg.load_pickle("geo.pkl")
        log.info("restored normalised %s frames from checkpoint", ps.split)
        return
    with Timer(f"read + normalise {ps.split}"):
        raw = read_split(ps.data_dir, ps.split)
        S1s, Xs = static_normalize(raw, cfg.n_jobs, cfg.geo_min_count)
        del raw
        ps.S1, ps.X = finalize(S1s, Xs, al["name_alias"], al["addr_alias"], set(al["noise_tokens"]))
        del S1s, Xs
        ps.geo = add_record_stats(ps.S1, ps.X, cfg.geo_min_count)
        stg.save_pickle("geo.pkl", ps.geo)
        stg.save_frame("x.parquet", ps.X)
        stg.save_frame("s1.parquet", ps.S1)


def predict_dense(ps) -> None:
    """Embed every record with the trained bi-encoder (only for blocks still to build)."""
    dc, stg = ps.cfg.dense, ps.stg
    ps.emb = None
    if not dc.enabled or all(stg.has(f"cand/{t}.json") for *_, t in block_keys(ps.S1, ps.X)):
        return
    if not ce.gpu_available():
        log.warning("no CUDA GPU: embedding %d records on the CPU will be slow", len(ps.S1) + len(ps.X))
    enc = ps.art / "dense_encoder"
    with Timer(f"dense embeddings of the {ps.split} records"):
        ps.emb = (_embed_parts(stg, "emb_s1", enc, dense.record_text(ps.S1), dc),
                  _embed_parts(stg, "emb_x", enc, dense.record_text(ps.X), dc))


def predict_blocking(ps) -> None:
    stg, bcfg = ps.stg, ps.cfg.blocking
    ranker = lgb.Booster(model_file=str(ps.art / "block_ranker.txt"))
    keys = block_keys(ps.S1, ps.X)
    tags = [t for *_, t in keys]
    with Timer("candidate generation"):
        for country, src, tag in keys:
            if stg.has(f"cand/{tag}.json"):
                continue
            emb = getattr(ps, "emb", None)
            _require_embeddings(ps.cfg, emb, "predict_dense")
            u = block_union(ps.S1, ps.X, country, src, bcfg, ps.cfg.n_jobs, emb=emb)
            n_union = len(u)
            block = prune_candidates(u, ps.X, ranker, bcfg.keep_per_source)
            del u
            stg.save_frame(f"cand/{tag}.parquet", block)
            stg.save_json(f"cand/{tag}.json", {"union_pairs": n_union})   # marks the block complete
        ps.emb = None
        _drop_embeddings(stg)                 # only needed for the blocks, which are all saved now
        if tags:
            ps.cand = _read_cand(stg, tags)
        else:
            ps.cand = pd.DataFrame({"s1_idx": np.zeros(0, np.int32), "x_idx": np.zeros(0, np.int32),
                                    "src": np.zeros(0, np.int8), "block_score": np.zeros(0, np.float32),
                                    **{c: np.zeros(0, np.float32) for c in COS_COLS},
                                    "via": np.zeros(0, np.uint16)})
        n_union = sum(stg.load_json(f"cand/{t}.json")["union_pairs"] for t in tags)
        log.info("union %d -> %d candidate pairs (%.1f per S1)", n_union,
                 len(ps.cand), len(ps.cand) / max(len(ps.S1), 1))


def predict_stage1(ps) -> None:
    cfg, meta, stg = ps.cfg, ps.meta, ps.stg
    cols = meta["f1_cols"]
    spans = _spans(len(ps.cand))
    ps.f1_parts = [f"F1/part_{i:04d}.npy" for i in range(len(spans))]
    p1_parts = [f"p1/part_{i:04d}.npy" for i in range(len(spans))]
    models, grp = load_models(ps.art, "stage1", meta["n_folds"]), None
    with Timer(f"stage-1 features + scoring ({len(ps.cand):,} rows, {len(spans)} parts on disk)"):
        for f_rel, p_rel, (a, b) in zip(ps.f1_parts, p1_parts, spans):
            if stg.has(p_rel):
                continue
            if grp is None:
                grp = group_features(ps.cand)
            Fn = stage1_matrix(ps.cand, np.arange(a, b), ps.S1, ps.X, ps.geo, grp, cfg.n_jobs,
                               cols).to_numpy(np.float32)
            stg.save_npy(f_rel, Fn)
            stg.save_npy(p_rel, predict_avg(models, Fn))   # written last: marks the part complete
            del Fn
    ps.p1 = np.concatenate([np.load(p) for p in stg.fetch_many(p1_parts)]) if spans else np.zeros(0)


def predict_cross_encoder(ps) -> None:
    ccfg = ps.cfg.cross_encoder
    ps.ce = np.full(len(ps.cand), np.nan, dtype=np.float32)
    if not ps.meta["ce_enabled"]:
        return
    if ps.stg.has("ce.npy"):
        ps.ce = ps.stg.load_npy("ce.npy")
        return
    if not ce.gpu_available():
        raise RuntimeError("These artefacts were trained with the cross-encoder; a CUDA GPU "
                           "is required at inference (or retrain with --no-cross-encoder).")
    with Timer("cross-encoder scoring"):
        rows = np.flatnonzero((ps.p1 >= ccfg.band_low) & (ps.p1 <= ccfg.band_high))
        ps.ce[rows] = _ce_score([ps.art / "ce_0", ps.art / "ce_1"], ps.cand, ps.S1, ps.X, rows, ccfg)
    ps.stg.save_npy("ce.npy", ps.ce)


def predict_stage2(ps) -> None:
    meta, stg = ps.meta, ps.stg
    if stg.has("p2.npy"):
        ps.p2 = stg.load_npy("p2.npy")
        return
    models = load_models(ps.art, "stage2", meta["n_folds"])
    with Timer("context features"):
        ctx = context_features(ps.cand, ps.p1, ps.X, ps.cfg.model.n_anchors, S1=ps.S1)
        ctx["ce"] = ps.ce
        C = _ctx_matrix(ctx, meta["f2_cols"][len(meta["f1_cols"]):])
        del ctx
    with Timer("stage-2 scoring"):
        ps.p2 = (predict_avg(models, PartsMatrix(stg.fetch_many(ps.f1_parts), extra=C))
                 if len(ps.cand) else np.zeros(0))
    stg.save_npy("p2.npy", ps.p2)


def predict_stage3(ps) -> None:
    """Stage-3 scoring when the decision step chose stage 3 (or the stage-2/3 mean, 23);
    ps.pfin is the final probability."""
    meta, stg = ps.meta, ps.stg
    final = meta.get("final_stage", 2)
    if final not in (3, 23):
        ps.pfin = ps.p2
        return
    if stg.has("p3.npy"):
        p3 = stg.load_npy("p3.npy")
    else:
        models = load_models(ps.art, "stage3", meta["n_folds"])
        with Timer("stage-3 context features (from p2)"):
            ctx = stage3_context(ps.cand, ps.p2, ps.X, ps.cfg.model.n_anchors, S1=ps.S1)
            ctx["ce"] = ps.ce
            C = _ctx_matrix(ctx, meta["f3_cols"][len(meta["f1_cols"]):])
            del ctx
        with Timer("stage-3 scoring"):
            p3 = (predict_avg(models, PartsMatrix(stg.fetch_many(ps.f1_parts), extra=C))
                  if len(ps.cand) else np.zeros(0))
        stg.save_npy("p3.npy", p3)
    ps.pfin = p3 if final == 3 else (ps.p2 + p3) / 2


def predict_write(ps) -> dict:
    dc = ps.cfg.decision
    p = getattr(ps, "pfin", None)
    p = ps.p2 if p is None else p
    sel = decide(ps.cand["s1_idx"].values, ps.cand["x_idx"].values, p, ps.meta["decision"],
                 dc.beta, dc.max_items)
    ps.selected = sel
    summary = write_outputs(ps.out, ps.S1, ps.X, ps.cand, sel, p)
    ps.stg.finish(summary)
    for rel in getattr(ps, "f1_parts", []):   # stay on S3; free the local disk
        ps.stg.drop_local(rel)
    return summary


PREDICT_STEPS = [predict_prepare, predict_dense, predict_blocking, predict_stage1, predict_cross_encoder,
                 predict_stage2, predict_stage3, predict_write]


def predict(data_dir, split: str, work_dir, out_dir, use_cache: bool = True, **ckpt) -> dict:
    ps = predict_init(data_dir, split, work_dir, out_dir, use_cache, **ckpt)
    out = None
    for step in PREDICT_STEPS:
        out = step(ps)
    return out

In [ ]:
#
# ▌2.13 Command-line entry point (`run.py`)
#
# Same pipeline from a terminal: `python src/run.py train|predict|package ...`.

In [ ]:
%%writefile {PKG}/src/run.py
"""Command-line entry point.

    python src/run.py train   --data-dir <student_resource>/dataset --work-dir work
    python src/run.py predict --data-dir <student_resource>/dataset --work-dir work --out-dir output
    python src/run.py package --out-dir output --team <team_name>

Add ``--ckpt-uri s3://bucket/prefix/`` to train / predict to mirror every
checkpoint to S3: an interrupted run then resumes from the last completed
block / feature part / fold, on the same machine or a new one.
"""
from __future__ import annotations

import argparse
import logging
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from ber.config import Config  # noqa: E402
from ber.pipeline import predict, train  # noqa: E402

CODE_ROOT = Path(__file__).resolve().parents[1]


def _setup_logging(work_dir: Path | None) -> None:
    handlers = [logging.StreamHandler(sys.stdout)]
    if work_dir is not None:
        work_dir.mkdir(parents=True, exist_ok=True)
        handlers.append(logging.FileHandler(work_dir / "run.log", encoding="utf-8"))
    logging.basicConfig(level=logging.INFO, handlers=handlers,
                        format="%(asctime)s %(levelname)s %(name)s | %(message)s")


def _package(out_dir: Path, team: str, dest: Path) -> Path:
    zpath = dest / f"{team}_submission.zip"
    doc = CODE_ROOT / "docs" / "Documentation_template.md"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for name in ("matching_results.tsv", "candidate_pairs.tsv"):
            z.write(out_dir / name, f"output/{name}")
        for p in CODE_ROOT.rglob("*"):
            if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc":
                z.write(p, f"code/business_entity_resolution/{p.relative_to(CODE_ROOT).as_posix()}")
        z.write(doc, "Documentation_template.md")
    return zpath


def main() -> None:
    ap = argparse.ArgumentParser(description="Business Entity Resolution pipeline")
    sub = ap.add_subparsers(dest="cmd", required=True)

    t = sub.add_parser("train", help="fit every model on dataset/train only")
    t.add_argument("--data-dir", type=Path, required=True)
    t.add_argument("--work-dir", type=Path, required=True)
    t.add_argument("--no-cross-encoder", action="store_true")
    t.add_argument("--no-dense", action="store_true", help="block without the learned dense view")
    t.add_argument("--max-train-s1", type=int, default=None)
    t.add_argument("--holdout-frac", type=float, default=None,
                   help="S1 fraction kept out for validation (default 0.15; 0 = train on all)")
    t.add_argument("--n-jobs", type=int, default=-1)
    t.add_argument("--ckpt-uri", default=None, help="s3://... prefix for checkpoints (optional)")
    t.add_argument("--s3-region", default=None)

    p = sub.add_parser("predict", help="apply trained artefacts to a split")
    p.add_argument("--data-dir", type=Path, required=True)
    p.add_argument("--split", default="test")
    p.add_argument("--work-dir", type=Path, required=True)
    p.add_argument("--out-dir", type=Path, required=True)
    p.add_argument("--ckpt-uri", default=None, help="s3://... prefix for checkpoints (optional)")
    p.add_argument("--s3-region", default=None)

    k = sub.add_parser("package", help="build <team>_submission.zip")
    k.add_argument("--out-dir", type=Path, required=True)
    k.add_argument("--team", required=True)
    k.add_argument("--dest", type=Path, default=Path("."))

    args = ap.parse_args()
    if args.cmd == "train":
        _setup_logging(args.work_dir)
        cfg = Config()
        cfg.n_jobs = args.n_jobs
        if args.no_cross_encoder:
            cfg.cross_encoder.enabled = False
        if args.no_dense:
            cfg.dense.enabled = False
        if args.holdout_frac is not None:
            cfg.model.holdout_frac = args.holdout_frac
        if args.max_train_s1:
            cfg.model.max_train_s1 = args.max_train_s1
        train(args.data_dir, args.work_dir, cfg, ckpt_uri=args.ckpt_uri, s3_region=args.s3_region)
    elif args.cmd == "predict":
        _setup_logging(args.work_dir)
        predict(args.data_dir, args.split, args.work_dir, args.out_dir,
                ckpt_uri=args.ckpt_uri, s3_region=args.s3_region)
    else:
        _setup_logging(None)
        print(_package(args.out_dir, args.team, args.dest))


if __name__ == "__main__":
    main()

In [ ]:
#
# ▌2.14 Package README
#
# Reproduction instructions shipped in the zip.

In [ ]:
%%writefile {PKG}/README.md
# Business Entity Resolution — reproducible pipeline

Links every Source 1 business to its Source 2 / Source 3 records with a
**blocking (sparse views + learned dense view) → two-stage GBDT (+ multilingual
cross-encoder) → expected-F0.5 decision** pipeline. All models are trained on `dataset/train` only; the test
split is touched only by `predict`.

## 1. Environment

```bash
python -m venv .venv && source .venv/bin/activate      # Python 3.11+
pip install -r requirements.txt
```

`torch` / `transformers` are needed for the dense retrieval view and the
cross-encoder. Both run on a CUDA GPU and are skipped automatically without
one (or with `--no-dense` / `--no-cross-encoder`); everything else is CPU
(multi-core) code.

## 2. Data layout

Point `--data-dir` at the challenge `dataset/` folder:

```
dataset/
  train/train_source1.tsv  train_source2.tsv  train_source3.tsv  train_ground_truth.tsv
  test/test_source1.tsv    test_source2.tsv   test_source3.tsv
```

## 3. Run end-to-end

```bash
# (a) Train + validate. Uses ONLY dataset/train. 15% of S1 entities are held out
#     and never trained on; their macro F0.5 is written to work/artifacts/metrics.json
python src/run.py train --data-dir ../../dataset --work-dir work

# (b) Optional final refit on 100% of the training entities (the OOF estimate
#     in metrics.json remains honest)
python src/run.py train --data-dir ../../dataset --work-dir work_full --holdout-frac 0

# (c) Inference on the test split -> output/matching_results.tsv + output/candidate_pairs.tsv
python src/run.py predict --data-dir ../../dataset --split test --work-dir work_full --out-dir ../../output

# (d) Validate the submission format (from the student_resource/ directory)
python utils/validate_submission.py --matching output/matching_results.tsv \
       --candidate output/candidate_pairs.tsv --test-dir dataset/test

# (e) Build <team>_submission.zip (output/ + code/ + filled Documentation_template.md)
python src/run.py package --out-dir ../../output --team <team_name> --dest ../..
```

### AWS SageMaker (S3)

`notebooks/Business_Entity_Resolution_Colab.ipynb` (despite the file name, it targets **AWS SageMaker**) runs the same pipeline cell by cell; `notebooks/sagemaker_cells.py` holds the identical cells as plain copy-paste blocks. The notebook downloads `dataset/train` and `utils/` from `s3://aws-ml-162599956323-ap-south-1-an/6ab10eb3b23ba_student_resource/student_resource/` to the instance's EBS volume. It then trains every stage with diagnostics and plots, downloads `dataset/test` only for inference, validates the submission and uploads the artefacts, both TSVs and the submission zip to `.../student_resource/ber_outputs/runs/<profile>_<timestamp>/`. Every checkpoint is mirrored to `.../student_resource/ber_outputs/checkpoints/`: after a crash, a kernel restart or on a new instance, run the notebook again from the top and it continues from the last completed block, feature part or fold. Recommended: `ml.g5.8xlarge` (128 GiB RAM, A10G GPU for the cross-encoder; `ml.g5.4xlarge` with 64 GiB also runs the full profile), EBS volume >= 100 GB, kernel `conda_pytorch_p310` (Notebook Instance) or Python 3 (Studio). The execution role needs `s3:ListBucket`, `s3:GetObject` and `s3:PutObject` on the bucket. The notebook is generated from the sources by `python tools/build_notebook.py`; re-run it after editing any module.

Every expensive step checkpoints into `<work-dir>/ckpt/<step>-<hash>/`, and with `--ckpt-uri s3://bucket/prefix/` (as the notebook does) each file is also uploaded to S3 the moment it is complete. The granularity is one blocking block, one 2-million-row feature part, one model fold or one cross-encoder half. Re-running `train` or `predict`, on the same machine or a new one, restores finished work and continues from the first missing file; a resumed run is bit-identical to an uninterrupted one. Each hash covers the training files, the settings the step depends on and its upstream steps, so changing a setting re-runs only the affected steps. After editing the library code itself, run once with caching off or bump `CKPT_VERSION` in `pipeline.py`.

Useful switches for `train`:

| flag | effect |
| --- | --- |
| `--no-cross-encoder` | skip the mDeBERTa stage |
| `--no-dense` | block without the learned dense view (with both flags: pure CPU pipeline) |
| `--holdout-frac F` | S1 fraction held out for validation (default 0.15, `0` = train on all) |
| `--max-train-s1 N` | fit the GBDTs on at most N S1 entities (candidates are still generated and scored for every entity, so competition features match the test set) |
| `--n-jobs N` | CPU workers for normalisation / blocking / features (default all cores) |

Every other hyper-parameter lives in `src/ber/config.py` and is saved to
`work/artifacts/config.json` for each run.

### SageMaker training jobs (alternative to the notebook)

`sagemaker/launch_jobs.py` runs the same pipeline as managed SageMaker training
jobs. It uses boto3 only, and should be run from a Studio or notebook-instance
terminal inside `code/business_entity_resolution`:

```bash
python sagemaker/launch_jobs.py check  --wait                     # ~5 min: packages, GPU, disk, model downloads, S3
python sagemaker/launch_jobs.py train  --wait --tail              # hold-out 0.15: honest score + diagnostics
python sagemaker/launch_jobs.py predict --run-name <run> --wait   # test inference -> submission files
python sagemaker/launch_jobs.py all --holdout-frac 0              # final refit, then predict
```

**What each job does**
- **The training job** mounts `dataset/train/` only. The test split exists only in the predict job.
- **Same configuration as the notebook.** It builds exactly the configuration of notebook cell 3.2, so it shares the notebook's S3 checkpoints. Stages the notebook already finished are restored, not recomputed. A failed, stopped or spot-interrupted job simply resumes when you launch it again.

**What each run folder contains** (`ber_outputs/runs/<run>/`)

| folder | content |
| --- | --- |
| `artifacts/` | the trained models (the predict job's input) |
| `reports/report.json` | the diagnostics of notebook cells 10.1/10.2/14.1/14.3–14.5/15.2 |
| `output/` | `matching_results.tsv`, `candidate_pairs.tsv` and `predict_summary.json` (internal checks and the official validator) |

**Launcher defaults**
- **Instance:** `ml.g6.8xlarge`, with a 300 GB volume and the AWS PyTorch 2.5.1 GPU image (Python 3.11).
- **Job dependencies:** installed from `sagemaker/requirements.txt`, which leaves the image's torch, numpy and CUDA untouched.
- **Spot pricing:** add `--spot`.
- **No internet access:** pass `--ce-model-s3` / `--dense-model-s3`.

## 4. Code map

| file | role |
| --- | --- |
| `src/run.py` | CLI: `train`, `predict`, `package` |
| `src/ber/io.py` | TSV readers/writers (tab-separated, no quoting) |
| `src/ber/normalize.py` | transliteration (any script → Latin), noise cleanup, legal-form / street-type canonicalisation, number parsing, locality-phrase joining, phonetic skeleton |
| `src/ber/aliases.py` | token aliases (tx↔texas, up↔uttar_pradesh, praivet↔pvt …) and noise tokens learned from training labels |
| `src/ber/blocking.py` | 5-view sparse TF-IDF top-k blocking (forward + reverse), the dense view's top-k, and learned candidate pruning |
| `src/ber/dense.py` | learned dense retrieval view: MiniLM-L12 bi-encoder (Apache-2.0) fine-tuned on training pairs, exact GPU top-k |
| `src/ber/features.py` | ~75 pairwise features + stage-2 context features |
| `src/ber/models.py` | entity-grouped K-fold LightGBM with out-of-fold predictions |
| `src/ber/cross_encoder.py` | optional mDeBERTa-v3-base cross-encoder (MIT) |
| `src/ber/decision.py` | exact expected-F0.5 selection, exclusivity, macro-F0.5 metric |
| `src/ber/checkpoint.py` | stage checkpoints on local disk, mirrored to S3 |
| `src/ber/diagnostics.py` | blocking / score / loss-breakdown / ambiguity-ceiling / corpus-size reports (JSON) |
| `sagemaker/launch_jobs.py` | starts and follows the SageMaker training jobs (boto3) |
| `sagemaker/sagemaker_entry.py` | what runs inside a job: `check`, `train`, `predict` |
| `sagemaker/requirements.txt` | the jobs' extra packages (installed by the container) |
| `src/ber/pipeline.py` | resumable train / predict steps (checkpointed, chunked, memory-safe) |
| `notebooks/Business_Entity_Resolution_Colab.ipynb` | step-by-step AWS SageMaker / S3 notebook (generated) |
| `notebooks/sagemaker_cells.py` | the same notebook as plain copy-paste cells (generated) |
| `tools/build_notebook.py` | regenerates the notebook from the sources |

## 5. Artefacts written by `train`

`work/artifacts/`: `config.json`, `aliases.json`, `block_ranker.txt`,
`dense_encoder/` (bi-encoder), `stage1_fold*.txt`, `stage2_fold*.txt`, `stage3_fold*.txt`,
`*_importance.csv`, `ce_0/`, `ce_1/` (cross-encoder), `meta.json` (feature lists + chosen decision rule) and
`metrics.json` (recall ceilings, OOF and hold-out macro F0.5). `work/run.log`
holds the full log.

## 6. Fair play / licences

No external data, APIs, gazetteers or geocoders are used. The only built-in
knowledge is generic abbreviation lists in `normalize.py` (Rd→road,
Pvt→private, SARL/SAS…). Every learned artefact comes from `dataset/train`.
Libraries are BSD / MIT / Apache / ISC licensed. The two neural models are
`microsoft/mdeberta-v3-base` (cross-encoder, MIT, 278M parameters) and
`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (dense retrieval
bi-encoder, Apache-2.0, 118M parameters), both far under the 8B limit and both
fine-tuned on `dataset/train` only.

In [ ]:
#
# ▌2.15 Pinned requirements
#
# Library versions used for the submission.

In [ ]:
%%writefile {PKG}/requirements.txt
# Core pipeline (blocking, features, GBDTs, decision) - these exact versions
# were used to run the pipeline end-to-end.
numpy==2.5.1
pandas==3.0.5
scipy==1.18.1
scikit-learn==1.9.1
lightgbm==4.7.0
rapidfuzz==3.14.6
anyascii==0.3.3
joblib==1.6.0
pyarrow==24.0.0

# Optional cross-encoder stage (disable with --no-cross-encoder).
# Model: microsoft/mdeberta-v3-base (MIT licence, 278M parameters).
torch==2.5.1
transformers==4.46.3
sentencepiece==0.2.0
protobuf==5.28.3

In [ ]:
#
# ▌2.16 Methodology document
#
# The filled challenge template, packaged at the root of the submission zip (fill in the team fields and the final metrics before submitting).

In [ ]:
%%writefile {PKG}/docs/Documentation_template.md
# ML Challenge 2026: Business Entity Resolution Solution Template

**Team Name:** Expecto Patronum
**Team Members:** Vedit Agrawal (BMS College of Engineering (BMSCE), Bangalore); Khushi Nawal (Ramaiah Institute of Technology, Bengaluru, Karnataka); Bhuvan Agrawal (M S Ramaiah Institute of Technology (MSRIT), Bangalore)
**Submission Date:** [Date]

---

## 1. Executive Summary

We resolve S2/S3 records to S1 businesses with a filter–verify pipeline:

- **Blocking.** Five sparse TF-IDF views (name tokens, name character n-grams, a transliteration-aware phonetic skeleton, address, and name×address conjunctions) with forward and reverse top-k retrieval, followed by a learned pruner.
- **Matching.** A two-stage LightGBM matcher. The second stage reasons over competing candidates, and an optional multilingual mDeBERTa-v3-base cross-encoder re-scores uncertain pairs.
- **Decision.** For each S1 entity, the matched subset maximizes the *exact expected F0.5*, which handles singletons natively.

Everything is learned from the training split only. All features are country-agnostic, so the pipeline transfers to the unseen French data.

---

## 2. Methodology

### 2.1 Problem Analysis

EDA on the full training set (2.2M S1, 5.0M S2, 5.3M S3):

- **Label structure.**
  - 5.6% of S1 entities are singletons; the rest have 1–11 matches (mean about 3.5, at most 5 from S2 and 6 from S3).
  - Every S2/S3 record belongs to at most one S1 entity.
  - About 27% of S2/S3 records match nothing and act as distractors.
  - True pairs always share the `country` label.
- **Signal coverage.** In true pairs:
  - 85% share a name token.
  - 82% share an address number.
  - 99.9% share a name token *or* an address word.

  No single blocking key is enough; several are needed together.
- **Name noise.**
  - Token shuffles, and legal forms moved, dropped or re-abbreviated (Pvt/Private, L.L.C.).
  - Appended fillers ("Center", "Services") and prefixed honorifics (Shri, M/s, Dr, The).
  - Punctuation debris (`--`, `<<`, `[...]`) and OCR substitutions (H0mes, Zi1via, 5ALINA).
  - Injected accents (Léarning), domain forms (redcafe.com, #naiktankers) and DBA names.
  - About 8–10% of rows contain Indic-script text, and many S2/S3 names are fully written in it. These are phonetic transliterations of the English name: ఇండో పవర్ ఎల్‌ఎల్‌పీ = Indo Power LLP.
- **Address noise.**
  - Component rotation and uppercase/abbreviated street types.
  - State code vs full name vs native script (TX/Texas, TG/Telangana/తెలంగాణ).
  - House-number formats (5-402, D/402, ##308, 004060, DOOR NO 72) and off-by-one or truncated numbers.
  - Wrong ordinals (89nd), city typos and aliases (Madras, ATL), `null`/`<NULL>` tokens, added PO boxes and landmarks.
- **Hard negatives.**
  - Generic names: identical normalized names are about 25× more often non-matches than matches.
  - Different businesses at the exact same address.
  - Near-duplicate siblings: same name with a slightly different house number or legal form.

  Name and address must therefore be judged jointly and in the context of competing candidates.

### 2.2 Solution Strategy

**Approach Type:** Hybrid. Multi-view blocking, then a two-stage GBDT (plus an optional cross-encoder), then expected-F0.5 set selection.

**Core Innovation:**

1. Aliases learned from the training labels (abbreviations, transliterations, state codes) combined with a phonetic skeleton that bridges Indic-script names to their English forms.
2. Stage-2 relational features over out-of-fold stage-1 probabilities.
3. An exact expected-F0.5 decision per entity, instead of a global threshold.

---

## 3. Candidate Generation (Blocking)

- **Normalization before blocking.**
  - NFKC and transliteration of any script to Latin (anyascii); lowercase.
  - Removal of null tokens, ID noise, "m/s" and punctuation debris; OCR digit→letter repair inside words.
  - Domain names collapsed to their stem; dotted acronyms joined (L.L.C.→llc).
  - Legal forms (English, Indian and French: pvt, ltd, llc, inc, sarl, sas, sci…) and street types (Rd/Road, St/Street/Saint, Bd/Boulevard, rue…) canonicalized.
  - Ordinal suffixes stripped, leading zeros removed from numbers, PO boxes separated.
  - Multi-word locality phrases joined (uttar pradesh → uttar_pradesh). The phrase list is mined without labels from the S1 records of the corpus being processed.
  - Token aliases learned from training true pairs, keeping the S1 form as canonical: texas→tx, up→uttar_pradesh, praivet→pvt, sophtveyr→software. A plausibility check rejects co-occurrence artefacts.
  - Noise name tokens learned from insertion rates (center, services…).
- **Blocking keys used.** One TF-IDF space per (country, target source) block, with IDF computed on that block, so the procedure adapts to any country label including France. The five views:
  1. **name tokens** — core-name words plus the whole name without spaces;
  2. **name char 4-grams** — for typos and OCR noise;
  3. **name phonetic skeleton** — ph→f, bh→v, voicing merged, vowels dropped, so "teknalaji" ~ "technology";
  4. **address** — words, house numbers, and number×word conjunctions within one component, robust to reordering;
  5. **name×address conjunctions** — very specific keys that survive even when each token alone is common.

  Tokens above a document-frequency cap are dropped. The sparse matrix products run in parallel row chunks. For each view we take the forward top-15 S2/S3 records per S1 and the reverse top-5 S1 entities per S2/S3 record, then union them.
  6. **dense view (learned)**: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (Apache-2.0, 118M parameters) fine-tuned as a siamese bi-encoder on "name | address" (normalized text).
     - **Loss.** Symmetric in-batch-negative contrastive loss (InfoNCE, as in DPR and Sentence-BERT).
     - **Batches.** Country-homogeneous batches of 512 pairs, so the negatives are hard. Records of the same entity are masked, because they are not negatives.
     - **Retrieval.** Exact cosine top-30 per S1 and top-10 per record on the GPU.
     - **Why it is needed.** It has no document-frequency cap. At full scale, with about a million entities per block, the absolute caps of the sparse views strip most of a generic name and a fixed top-k is crowded out. Candidate recall fell from 99.9% on a 20k-entity slice to 98.1% on the full set.
     - **No leakage.** The encoder is trained only on a separate slice of training entities (15% of the non-hold-out ones) on which no other model is fitted, so its cosine is an honest feature.
- **Learned pruning.** A LightGBM ranker keeps the top 16 candidates per (S1, source). Its features are the six view cosines and the number of sparse and dense lists that retrieved the pair, and it is trained on training labels. Every record's single best S1 is always kept. The pruned set is written to `candidate_pairs.tsv`, and it is exactly what the matcher scores.
- **Candidate pairs generated:** [fill from the `predict` log: "pruning: … -> N candidate pairs"]. About 24 per S1 entity on training slices.
- **How we ensured true matches were not lost.**
  - The five complementary views cover typos, transliteration, missing addresses and rotated components.
  - Reverse retrieval protects records whose owner is crowded out by generic look-alikes.
  - Recall is measured at every stage (`union_recall_holdout`, `cand_recall_holdout` in `metrics.json`). On a 20k-entity training slice the hold-out candidate recall was 99.88% before and after pruning, with an oracle F0.5 ceiling of 0.9993.

---

## 4. Matching Model

**Features used** (about 75 in stage 1, about 90 in stage 2):

- **Name features.**
  - Levenshtein / indel ratio, token-set, token-sort and partial ratios on the full and core names.
  - Jaro-Winkler, prefix similarity and exact match on the space-free core name.
  - Skeleton ratio and token-set; token-set against the DBA alternative name.
  - Name lengths and token counts.
  - Legal-form agreement and conflict.
  - Flags: transliterated script, domain-style name, DBA.
  - Genericness: corpus frequency of the exact core name, and min/max document frequency of its tokens.
- **Address features.**
  - Ratio, token-set and token-sort on the full address; token-set and partial on address words.
  - Similarity of the street part after removing locality components.
  - House numbers: count, intersection, Jaccard, first-number equality and membership, best Levenshtein similarity between numbers, log absolute difference of first numbers, overlap of numbers with 3+ digits.
  - Locality components (a vocabulary mined from S1): intersection, conflict, subset, component Jaccard.
  - Empty-address and PO-box flags; joint name×address evidence.
- **Blocking / group features.**
  - The five sparse view cosines, the dense embedding cosine and the pruner score.
  - How many sparse and dense retrieval lists found the pair.
  - Rank and gap of the pruner score and of the dense cosine, within the S1 entity and within the S2/S3 record.
  - The number of S1 entities claiming the record and the number of candidates per S1.
- **Stage-2 context features** (from out-of-fold stage-1 probabilities p1):
  - Rank, gap, sum and high-confidence count of p1 within the S1 entity and within (S1, source).
  - Rank and gap of p1 among all S1 entities claiming the same record, the second-best claimant, and high-confidence claimant count.
  - **Sibling-anchor agreement:** name and address similarity of the record to the S1 entity's most probable *other* candidates, weighted by their p1.
  - The cross-encoder score (NaN outside its band).

**Model type:**

- Two LightGBM binary classifiers:
  - stage 1: 255 leaves, learning rate 0.05;
  - stage 2: 127 leaves, learning rate 0.03.
- Each is trained with 5-fold cross-validation **grouped by S1 entity**, with early stopping. Out-of-fold predictions feed stage 2 and the decision tuning.
- **Optional cross-encoder:** `microsoft/mdeberta-v3-base` (MIT, 278M parameters), a multilingual tokenizer covering Indic scripts and French. It is fine-tuned on "name | address" pairs (up to 128 pieces) for uncertain stage-1 pairs (0.01 ≤ p1 ≤ 0.99, plus 20% confident pairs), 2-fold over entities, so its scores are also out-of-fold.
- **Stage 3 (collective refinement):** the competition and sibling-anchor context is recomputed from the stage-2 probabilities, which include the cross-encoder. The record's margin over its strongest competing claimant is added. A third grouped 5-fold LightGBM (127 leaves, learning rate 0.03) is trained on it, with the same folds so it stays out-of-fold. This is stacked collective classification. The final stage (2 or 3) is the one with the better out-of-fold macro F0.5.
- For test inference, predictions are averaged over the fold models.

**Threshold selection method:**

- For each S1 entity with candidate probabilities p (after a temperature calibration T), choose k* = argmax E[F0.5 | predict top-k]. The expectation is computed exactly with Poisson-binomial DPs over the selected and unselected candidates. E[F0.5 | k=0] = P(no true match), which protects singletons.
- The grid covers expected-F with T ∈ {0.6…2.0}, a global threshold ∈ [0.30, 0.96], and each with or without one-owner exclusivity (every S2/S3 record goes only to its most probable S1).
- The winner is chosen by out-of-fold macro F0.5 on the fit entities, then measured once on the untouched hold-out entities.

---

## 5. Results & Error Analysis

- **F_0.5 score (macro).**
  - Full training run: hold-out = [fill from `work/artifacts/metrics.json → holdout_macro_f05`]; out-of-fold = [`oof_macro_f05`].
  - Development slice (20k training S1 entities, 15% hold-out, GBDT only): hold-out **0.9931**, out-of-fold 0.9948, stage 1 alone at 0.5: 0.9889.
  - A separate 6k-entity training-derived slice scored end-to-end through `predict`: 0.996.
- **Common false positives (wrong merges).**
  - Planted siblings: same or near-same name at the same street with a different house number or legal form (Baverex LLC 3103 vs Baverex Ltd 3107 Poplar Pl).
  - A different business in the same building whose name is in Indic script (Sai Constructions vs సాయి హెల్త్‌కేర్, same flat and street).
  - Same generic name with an empty S2/S3 address.
- **Common false negatives (missed matches).**
  - The S2/S3 name is an unrelated brand (Umbramirazeph, Gildiri) *and* the address is partial.
  - Generic names with an empty address, which the precision-oriented decision correctly leaves out.
  - Rarely, Indic-script names with very sparse addresses missed by blocking. The skeleton view was added for exactly these cases.

---

## 6. Conclusion

- Careful, largely data-driven normalization (aliases, noise tokens and locality phrases all learned from training data) makes simple sparse retrieval reach near-complete recall.
- A second-stage model that sees competing candidates, plus an exact expected-F0.5 decision, turns strong pairwise scores into precise per-entity match lists.
- The main lesson: in this dataset the hardest errors are relational (siblings, shared addresses, generic names), so context features and per-entity decoding matter more than a heavier pairwise model.

---

## Appendix

### A. Code Artefacts

`code/business_entity_resolution/`:

- `src/run.py` — CLI: `train`, `predict`, `package`.
- `src/ber/normalize.py`, `aliases.py` — normalization and learned aliases.
- `src/ber/blocking.py` — five-view blocking and pruning.
- `src/ber/features.py` — features.
- `src/ber/models.py` — grouped-OOF LightGBM.
- `src/ber/cross_encoder.py` — optional mDeBERTa.
- `src/ber/decision.py` — expected-F0.5 decoding and the macro-F0.5 metric.
- `src/ber/pipeline.py` — orchestration.
- `README.md` — exact commands; `requirements.txt` — pinned versions.

Reproduce:

```bash
pip install -r requirements.txt
python src/run.py train   --data-dir <dataset> --work-dir work [--holdout-frac 0]
python src/run.py predict --data-dir <dataset> --split test --work-dir work --out-dir output
```

This writes `output/matching_results.tsv` and `output/candidate_pairs.tsv`. No external data, APIs or geocoders are used.

### B. Additional Results

`work/artifacts/metrics.json` reports:

- union and candidate recall;
- out-of-fold and hold-out macro F0.5;
- hold-out pair precision and recall;
- the oracle ceiling on candidates;
- the chosen decision rule.

`stage1_importance.csv` and `stage2_importance.csv` hold feature importances. On the development slice the top stage-2 features were p1, the pruner score, joint name×address evidence, the within-source p1 gap, the record's number of confident claimants, and address-anchor agreement.

In [ ]:
# ── 2.17 Import the library (and make it visible to worker processes) ──────
import importlib
src_dir = str(Path(PKG) / "src")
ber_dir = Path(src_dir) / "ber"
missing = [m for m in ("__init__", "config", "io", "normalize", "aliases", "blocking", "dense", "features",
                       "models", "cross_encoder", "decision", "diagnostics", "checkpoint", "pipeline")
           if not (ber_dir / f"{m}.py").exists()]
assert not missing, f"library files missing in {ber_dir}: {missing} - run the %%writefile cells of Section 2 first"

# The library folder must come first on the path, in this kernel and in worker processes. Any other
# folder named "ber" - e.g. LOCAL_ROOT itself when the notebook runs from its parent folder - would
# otherwise be imported as an empty namespace package ("cannot import name 'config' from 'ber'").
sys.path[:] = [src_dir] + [p for p in sys.path if p != src_dir]
os.environ["PYTHONPATH"] = os.pathsep.join(
    [src_dir] + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p and p != src_dir])
# Forget any earlier import of ber (stale code, or the wrong folder) so that the modules below are
# exactly the files written above, and stop idle joblib workers that still hold the old modules.
for _m in [m for m in sys.modules if m == "ber" or m.startswith("ber.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
try:
    from joblib.externals.loky import get_reusable_executor
    get_reusable_executor().shutdown(wait=True)
except Exception:
    pass

import ber
from ber import config as ber_config, normalize, aliases, blocking, features, models, \
    cross_encoder, dense, decision, checkpoint, pipeline, io as ber_io
_loaded = Path(getattr(ber, "__file__", None) or "?").resolve().parent
assert _loaded == ber_dir.resolve(), f"'ber' was imported from {_loaded}, expected {ber_dir}"
print("ber library loaded from", _loaded)

In [ ]:
#
# ▌3. CONFIGURATION
#
# Edit the switches in 3.1. `PROFILE="auto"` picks the largest setting that fits the instance's memory:
#
# | profile | RAM | example SageMaker instance | entities used to fit the GBDTs | notes |
# |---|---|---|---|---|
# | `full` | ≥ 60 GB | `ml.g5.4xlarge`, `ml.g5.8xlarge`, `ml.r6i.2xlarge`, `ml.m6i.4xlarge` | all non-hold-out (~1.9 M) | best accuracy |
# | `large` | ≥ 40 GB | 48 GiB instances | 900 000 | near-full accuracy |
# | `medium` | ≥ 24 GB | `ml.g5.2xlarge`, `ml.m6i.2xlarge` | 400 000 | |
# | `dev` | any | any | train-derived subset (`DEV_SAMPLE_S1`) | workflow check only |
#
# Whatever the profile, **candidates are generated and scored for every Source 1 entity**, so the "competition" features look exactly as they will on the test set; the profile only limits how many entities the models are *fitted* on. Feature matrices are kept on disk and streamed into LightGBM, and the text tables are released while the models train, which is why 64 GiB is enough for the full profile.
#
# **Checkpoints.** Each step's checkpoint is keyed by a hash of the data and of the settings it depends on. Changing a setting re-runs only the steps that depend on it (for example, changing `PROFILE` reuses the normalisation and blocking checkpoints). `USE_CACHE = False` ignores all checkpoints.

In [ ]:
# ── 3.1 Main switches (EDIT HERE) ────────────────────────────────────────────
PROFILE           = os.environ.get("BER_PROFILE", "auto")   # "auto" | "full" | "large" | "medium" | "dev"
USE_CROSS_ENCODER = True     # used only when a CUDA GPU is present
HOLDOUT_FRAC      = 0.15     # S1 entities kept out for validation (0.0 = final refit on everything)
DEV_SAMPLE_S1     = int(os.environ.get("BER_DEV_SAMPLE", 0)) or None   # e.g. 20000 forces DEV mode
USE_CACHE         = True     # resume from the checkpoints (local copy first, then S3_CKPT_URI)
# Stop at once if the finished work up to stage 1 is not found on S3 (4.2), instead of silently recomputing
# it (10-20 hours), or if the cross-encoder is disabled (12.1). Set False only for a deliberately fresh run.
REQUIRE_RESTORE   = True
SEED              = 2026
CE_MODEL_S3_URI   = None     # e.g. "s3://<bucket>/models/mdeberta-v3-base/" if the instance has no internet
USE_DENSE         = True     # learned dense retrieval view (GPU); see Section 9.2
DENSE_MODEL_S3_URI = None    # e.g. "s3://<bucket>/models/paraphrase-multilingual-MiniLM-L12-v2/" (no internet)
# Sparse blocking breadth. The previous run used 10 / 3 (hold-out candidate recall 98.0%); cell 10.3
# measures the hold-out recall of other values. Changing any of them re-runs blocking and everything after it.
BLOCK_K_FWD       = 15       # forward top-k per sparse view
BLOCK_K_REV       = 5        # reverse top-k per sparse view
BLOCK_CAP_X       = 1        # multiplier on the document-frequency caps (max_df_*)
USE_STAGE3        = True     # collective refinement (13.2); kept only if it wins on out-of-fold data
# Cross-encoder backbone: "microsoft/mdeberta-v3-base" (MIT, 278M) or the stronger
# "FacebookAI/xlm-roberta-large" (MIT, 560M; about 3x slower to train and score).
CE_MODEL          = "microsoft/mdeberta-v3-base"
# Cross-encoder training pairs per half. None = sized by GPU class in 3.2 (1.2M on L4 / A10G: ~2-2.5 h per
# half). 600_000 about halves the cross-encoder's GPU time at a small accuracy cost. Changing it retrains
# the cross-encoder (only when it is not already finished on S3 with the current value).
CE_TRAIN_PAIRS    = None
# Read-only robustness checks (training data only; they never change the trained models):
SIZE_CHECK_FRACS  = ()           # 15.2: e.g. (0.1, 0.3) re-runs inference on train-derived corpora of these
                                 # sizes (~1-2 extra GPU hours); off by default to save credits
RUN_COUNTRY_CHECK = False        # 21.1: train on one country, score the other (a separate, long run)
RUN_BLOCKING_WHATIF = False      # 10.3: sparse-view recall at other k / caps (rebuilds every block's TF-IDF twice: slow)

In [ ]:
# ── 3.2 Resolve the profile and build the configuration ─────────────────────
from dataclasses import asdict
from ber.config import Config

if PROFILE == "auto":
    PROFILE = "full" if RAM_GB >= 60 else "large" if RAM_GB >= 40 else "medium" if RAM_GB >= 24 else "dev"
if DEV_SAMPLE_S1:
    PROFILE = "dev"
if PROFILE == "dev" and not DEV_SAMPLE_S1:
    DEV_SAMPLE_S1 = 40_000
    print("WARNING: not enough RAM for the full training corpus -> DEV mode on a train-derived subset.")

cfg = Config()
cfg.n_jobs = -1
cfg.model.seed = SEED
cfg.model.holdout_frac = HOLDOUT_FRAC
cfg.model.max_train_s1 = {"full": None, "large": 900_000, "medium": 400_000, "dev": None}[PROFILE]
cfg.model.ranker_s1 = 150_000
for _v in ("name_tok", "name_char", "name_skel", "addr", "name_addr"):
    setattr(cfg.blocking, f"k_{_v}", int(BLOCK_K_FWD))
    setattr(cfg.blocking, f"max_df_{_v}", int(getattr(cfg.blocking, f"max_df_{_v}") * BLOCK_CAP_X))
cfg.blocking.k_reverse = int(BLOCK_K_REV)
cfg.model.stage3 = USE_STAGE3
# LightGBM is fastest with one thread per physical core (speed only: not part of any checkpoint key)
N_PHYS = psutil.cpu_count(logical=False) or N_CPU
for _p in (cfg.model.lgb_stage1, cfg.model.lgb_stage2, cfg.model.lgb_stage3):
    _p["num_threads"] = N_PHYS

# Cross-encoder sizing by GPU class (training pairs per half, batch sizes).
HAS_GPU = TORCH_CUDA                       # PyTorch must see the GPU, not just the driver
cfg.cross_encoder.enabled = USE_CROSS_ENCODER and HAS_GPU
gpu = (torch.cuda.get_device_name(0) if TORCH_CUDA else GPU_INFO).upper()
if "A100" in gpu or "H100" in gpu:
    cfg.cross_encoder.max_train_pairs, cfg.cross_encoder.batch_size, cfg.cross_encoder.infer_batch_size = 2_000_000, 128, 1024
elif "L4" in gpu or "A10" in gpu:
    cfg.cross_encoder.max_train_pairs, cfg.cross_encoder.batch_size, cfg.cross_encoder.infer_batch_size = 1_200_000, 96, 512
else:  # T4 / V100 / other 16 GB cards
    cfg.cross_encoder.max_train_pairs, cfg.cross_encoder.batch_size, cfg.cross_encoder.infer_batch_size = 350_000, 64, 256
cfg.cross_encoder.model_name = CE_MODEL
if "large" in CE_MODEL.lower():            # 560M backbone: smaller batches and learning rate
    cfg.cross_encoder.batch_size = max(16, cfg.cross_encoder.batch_size // 3)
    cfg.cross_encoder.infer_batch_size = max(64, cfg.cross_encoder.infer_batch_size // 2)
    cfg.cross_encoder.lr = 1e-5
if CE_TRAIN_PAIRS:
    cfg.cross_encoder.max_train_pairs = int(CE_TRAIN_PAIRS)
if PROFILE == "dev":
    cfg.cross_encoder.max_train_pairs = min(cfg.cross_encoder.max_train_pairs, 60_000)

# Dense retrieval view: needs the GPU and the pretrained encoder. Decided here, before the
# checkpoint keys are computed in 4.2, so that the keys describe what actually runs.
cfg.dense.enabled = USE_DENSE and HAS_GPU
if cfg.dense.enabled:
    try:
        if DENSE_MODEL_S3_URI:                     # offline instances: pretrained weights from S3
            _local = LOCAL_ROOT / "hf_models" / "paraphrase-multilingual-MiniLM-L12-v2"
            s3_download(DENSE_MODEL_S3_URI, _local)
            cfg.dense.model_name = str(_local)
        cross_encoder._quiet_transformers()
        from transformers import AutoTokenizer
        AutoTokenizer.from_pretrained(cfg.dense.model_name)   # fail fast (no internet / no weights)
    except Exception as e:
        print(f"dense view disabled - cannot load {cfg.dense.model_name}: {type(e).__name__}: {e}")
        cfg.dense.enabled = False
elif USE_DENSE:
    print("dense view disabled - PyTorch cannot use a GPU in this kernel")

# every run uploads its artefacts and submission files to its own S3 folder
RUN_NAME = f"{PROFILE}_{time.strftime('%Y%m%d-%H%M%S')}"
RUN_URI = f"{S3_OUTPUT_URI}runs/{RUN_NAME}/"

print(f"PROFILE={PROFILE}  dense view={'on' if cfg.dense.enabled else 'off'}  "
      f"cross-encoder={'on' if cfg.cross_encoder.enabled else 'off'}  "
      f"hold-out={HOLDOUT_FRAC}  max_train_s1={cfg.model.max_train_s1}")
print("results will be uploaded to", RUN_URI)
print("checkpoints (resume point)  :", S3_CKPT_URI if USE_CACHE else "disabled (USE_CACHE=False)")
display(pd.json_normalize(asdict(cfg), sep=".").T.rename(columns={0: "value"}).astype(str))

In [ ]:
#
# ▌4. LOAD THE TRAINING DATA
#
# In DEV mode a train-derived subset is built first: `DEV_SAMPLE_S1` Source 1 entities, all their true S2/S3 records, and a proportional share of other records as distractors. A second disjoint subset, `deveval`, stands in for the test set in Section 17 so that the whole workflow can be scored. Both come from `dataset/train`; the real test data is never used for training or tuning.

In [ ]:
# ── 4.1 (DEV mode only) build train-derived subsets ─────────────────────────
def build_dev_dataset(src: Path, dst: Path, n_train: int, n_eval: int, seed: int = 7) -> Path:
    if (dst / "deveval" / "deveval_ground_truth.tsv").exists():
        print("dev dataset already present:", dst); return dst
    rng = random.Random(seed)
    gt = {}
    with open(src / "train" / "train_ground_truth.tsv", encoding="utf-8") as f:
        header = next(f)
        for line in f:
            a, _, b = line.rstrip("\n").partition("\t"); gt[a] = b
    ids = list(gt); rng.shuffle(ids)
    splits = {"train": set(ids[:n_train]), "deveval": set(ids[n_train:n_train + n_eval])}
    owner = {x: sp for sp, S in splits.items() for a in S for x in gt[a].split(",") if x}
    rate = {sp: 0.5 * len(S) / len(ids) for sp, S in splits.items()}
    outs = {}
    for sp in splits:
        (dst / sp).mkdir(parents=True, exist_ok=True)
        for k in (1, 2, 3):
            outs[sp, k] = open(dst / sp / f"{sp}_source{k}.tsv", "w", encoding="utf-8", newline="\n")
    for k in (1, 2, 3):
        with open(src / "train" / f"train_source{k}.tsv", encoding="utf-8") as f:
            h = next(f)
            for sp in splits: outs[sp, k].write(h)
            for line in f:
                i = line[:line.find("\t")]
                if k == 1:
                    for sp, S in splits.items():
                        if i in S: outs[sp, k].write(line)
                else:
                    o = owner.get(i)
                    if o: outs[o, k].write(line)
                    else:
                        r = rng.random()
                        if r < rate["train"]: outs["train", k].write(line)
                        elif r < rate["train"] + rate["deveval"]: outs["deveval", k].write(line)
    for v in outs.values(): v.close()
    for sp, S in splits.items():
        with open(dst / sp / f"{sp}_ground_truth.tsv", "w", encoding="utf-8", newline="\n") as g:
            g.write(header)
            for a in S: g.write(f"{a}\t{gt[a]}\n")
    return dst

if PROFILE == "dev":
    TRAIN_DATA_DIR = build_dev_dataset(DATA_DIR, LOCAL_ROOT / "dev_dataset", DEV_SAMPLE_S1,
                                       max(DEV_SAMPLE_S1 // 4, 2000))
    EVAL_SPLIT, EVAL_DATA_DIR = "deveval", TRAIN_DATA_DIR
else:
    TRAIN_DATA_DIR = DATA_DIR
    EVAL_SPLIT, EVAL_DATA_DIR = "test", DATA_DIR
print("training data :", Path(TRAIN_DATA_DIR) / "train")
print("inference on  :", Path(EVAL_DATA_DIR) / EVAL_SPLIT)

In [ ]:
# ── 4.2 Initialise the training state (checkpoints on S3) and read the data ─
st = pipeline.train_init(TRAIN_DATA_DIR, WORK_DIR, cfg, use_cache=USE_CACHE, ckpt_uri=S3_CKPT_URI,
                         s3_region=S3_REGION, s3_endpoint=S3_ENDPOINT, legacy_work_dir=LEGACY_WORK_DIR)
(Path(WORK_DIR) / "run_uri.txt").unlink(missing_ok=True)   # rewritten by 16.1 for these artefacts
display(pd.DataFrame([{"stage": k, "checkpoint": v.key, "status": "complete" if v.done else "to do"}
                      for k, v in st.stg.items()]))
_missing = pipeline.missing_restores(st)
if _missing and REQUIRE_RESTORE:
    raise RuntimeError(
        f"not restorable from {S3_CKPT_URI}: {_missing} - these steps would be recomputed from scratch. "
        "Check S3_URI / the settings of 3.1 (HOLDOUT_FRAC, SEED, BLOCK_K_*, BLOCK_CAP_X, USE_DENSE) and that "
        "the earlier run finished them; set REQUIRE_RESTORE = False in 3.1 only for a deliberately fresh run.")
print("restorable up to stage 1:", "yes" if not _missing else f"NO - will recompute {_missing}")
pipeline.train_load(st)
print(f"ground-truth rows: {len(st.gt):,}")
if getattr(st, "raw", None) is not None:
    for k, v in st.raw.items():
        print(f"{k}: {len(v):,} rows")

In [ ]:
#
# ▌5. EXPLORATORY DATA ANALYSIS
#
# Findings that drive the design:
# - **Singletons** (no true match) score 1.0 only with an empty prediction.
# - Every S2/S3 record belongs to **at most one** Source 1 entity, which motivates the exclusivity option.
# - True matches always share the **country** label, so blocking runs per country.
# - Names and addresses are heavily noised (see examples), and many names are written in **Indic scripts**.

In [ ]:
# ── 5.1 Ground-truth structure ───────────────────────────────────────────────
sizes = np.array([len(v) for v in st.gt.values()])
n_s2 = np.array([sum(x.startswith("S2-") for x in v) for v in st.gt.values()])
n_s3 = sizes - n_s2
all_ids = [x for v in st.gt.values() for x in v]
print(f"S1 entities            : {len(sizes):,}")
print(f"singleton rate         : {np.mean(sizes == 0):.2%}")
print(f"mean matches / entity  : {sizes.mean():.2f}   (max {sizes.max()})")
print(f"max S2 / S3 per entity : {n_s2.max()} / {n_s3.max()}")
print(f"records matched twice  : {len(all_ids) - len(set(all_ids))}")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
vals, cnts = np.unique(sizes, return_counts=True)
ax[0].bar(vals, cnts, color="#4C72B0"); ax[0].set_title("True matches per Source 1 entity")
ax[0].set_xlabel("number of matches"); ax[0].set_ylabel("entities")
w = 0.4
for off, arr, lab in ((-w/2, n_s2, "from S2"), (w/2, n_s3, "from S3")):
    v, c = np.unique(arr, return_counts=True); ax[1].bar(v + off, c, width=w, label=lab)
ax[1].set_title("Matches per source"); ax[1].legend(); ax[1].set_xlabel("matches")
plt.tight_layout(); plt.show()
del all_ids

In [ ]:
# ── 5.2 Field quality per source ─────────────────────────────────────────────
import re
NONLATIN = r"[^\u0000-\u024f\u1e00-\u1eff\u2000-\u206f]"   # anything outside Latin + punctuation

def eda_frames():
    # raw frames if still in memory, otherwise the restored normalised frames (they keep the raw text)
    if getattr(st, "raw", None) is not None:
        return {k: v.rename(columns={"business_name": "raw_name", "business_address": "raw_addr"})
                for k, v in st.raw.items()}
    S1 = st.S1 if st.S1 is not None else st.S1s
    X = st.X if st.X is not None else st.Xs
    return {"s1": S1, "s2": X[X["src"] == 2], "s3": X[X["src"] == 3]}

rows = []
for k, f in eda_frames().items():
    samp = f.sample(min(len(f), 200_000), random_state=0)
    rows.append({"source": k.upper(), "rows": len(f),
                 **{f"country={c}": f"{p:.1%}" for c, p in f["country"].value_counts(normalize=True).items()},
                 "empty address": f"{(samp['raw_addr'].str.strip() == '').mean():.2%}",
                 "non-Latin script": f"{samp['raw_name'].str.contains(NONLATIN, regex=True).mean():.2%}"})
display(pd.DataFrame(rows).fillna("-"))

In [ ]:
# ── 5.3 Examples of true-match groups (the noise the model must see through) ─
frames = eda_frames()
lookup = pd.concat([f[["entity_id", "raw_name", "raw_addr", "country"]] for f in frames.values()]).set_index("entity_id")
rng = np.random.default_rng(3)
multi = [k for k, v in st.gt.items() if len(v) >= 2]
for s1 in rng.choice(multi, min(4, len(multi)), replace=False):
    ids = [s1] + st.gt[s1]
    display(Markdown(f"**{s1}**"))
    display(lookup.loc[[i for i in ids if i in lookup.index]])
del frames, lookup, multi; gc.collect()

In [ ]:
#
# ▌6. TEXT NORMALISATION
#
# Deterministic and country-agnostic:
# - **Transliteration.** Every script becomes Latin (`anyascii`): ఇండో పవర్ ఎల్‌ఎల్‌పీ → `imdo pavar elelpi`.
# - **Cleanup.** Accent folding, case folding, OCR-digit repair (`H0mes` → `homes`), `m/s`, domains (`redcafe.com` → `redcafe`), dotted acronyms (`L.L.C.` → `llc`) and DBA splitting.
# - **Canonical forms.** Legal forms (`Private`→`pvt`, `SARL`, `SAS`, ...), street types (`Road`→`rd`, `Boulevard`/`Bd`→`blvd`, `rue`, ...), ordinals and house numbers.
# - **Phonetic skeleton.** A consonant skeleton that bridges transliterations to their English source.

In [ ]:
# ── 6.1 Normalisation on real-looking examples ──────────────────────────────
from ber.normalize import normalize_name, normalize_address, skeleton
names = ["Pvt. Sacha Téchnologies Ltd.", "LLC FRANKLIN, DANIEL + ARRINGTNO", "Lappin Gbn H0mes LLC",
         "Marshallbuilders.C0m", "Brixumbra dba Crystal Emera LLC", "ఇండో పవర్ ఎల్‌ఎల్‌పీ",
         "इनोवेटिव इंफ्रास्ट्रक्चर प्राइवेट लिमिटेड", "SCI Ptit Àmicale"]
display(pd.DataFrame([{"raw name": n, "normalised tokens": normalize_name(n)[0],
                       "phonetic skeleton": " ".join(skeleton(t) for t in normalize_name(n)[0].split())}
                      for n in names]))
addrs = ["##308, VIJAYA TOWERS NAGARJUNANAGAR COLONY, HYDERABAD, Telangana",
         "IL, 1606 11TH STREET, SPRINGFIELD CPD", "11003 89nd Avenue, Vancouver, Washington",
         "<NULL>, 5Th Main Road, Plot D/427 #A3&A4 Brundavana, KA",
         "175 Bd du Président Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine"]
display(pd.DataFrame([{"raw address": a, "tokens": normalize_address(a)[0],
                       "components": normalize_address(a)[1]} for a in addrs]))

In [ ]:
# ── 6.2 Static normalisation of the training corpus ─────────────────────────
pipeline.train_normalize(st)
if st.S1s is not None:
    display(st.S1s[["raw_name", "name_tok", "raw_addr", "addr_comps"]].head(5))
else:
    print("restored from checkpoint - static pass skipped")

In [ ]:
#
# ▌7. ENTITY SPLIT
#
# Source 1 entities are split **before anything is learned**:
# - **hold-out** (`HOLDOUT_FRAC`): never used for fitting anything. It gives the honest validation score.
# - **dense** (`dense_frac`, only with the dense view): their true pairs train the bi-encoder of 9.2 and nothing else. Their candidates are still generated and scored as competitors, exactly like the test set's.
# - **fit**: the entities whose candidates train the GBDTs, in 5 folds grouped by entity.
# - **ranker**: a sample (disjoint from the fit entities whenever enough remain) whose candidates train the blocking pruner. It does not depend on the profile, so the blocking checkpoint survives a profile change.

In [ ]:
# ── 7.1 Split entities and build the label arrays ───────────────────────────
pipeline.train_split(st)
display(pd.Series({k: st.metrics[k] for k in ("n_s1", "n_x", "singleton_rate", "n_fit_s1",
                                              "n_holdout_s1", "n_ranker_s1", "n_dense_s1")}, name="value").to_frame())

In [ ]:
#
# ▌8. LEARNED ALIASES & NOISE TOKENS
#
# These are learned from **training-split true pairs only**. For a record matched to a Source 1 entity, tokens the record has and the entity lacks are paired with tokens the entity has and the record lacks. Pairs that recur consistently become aliases, with the clean Source 1 form as the canonical side: state codes (`tg→telangana`), transliterations (`praivet→pvt`, `sophtveyr→software`) and typos. Tokens that sources routinely *insert* into names (`center`, `services`) become noise tokens and are dropped from the core-name view.

In [ ]:
# ── 8.1 Learn (or restore) aliases and show samples ─────────────────────────
pipeline.train_aliases(st)
al = json.loads((st.art / "aliases.json").read_text())
print(f"name aliases: {len(al['name_alias'])}   address aliases: {len(al['addr_alias'])}   "
      f"noise tokens: {al['noise_tokens']}")
show = lambda d, n: pd.DataFrame(list(d.items())[:n], columns=["variant", "canonical"])
display(pd.concat([show(al["name_alias"], 25).add_prefix("name "),
                   show(al["addr_alias"], 25).add_prefix("address ")], axis=1))

In [ ]:
#
# ▌9. FINAL VIEWS & CORPUS STATISTICS
#
# The learned aliases are applied, then the views used downstream are derived: core name, space-free name, phonetic skeleton, legal forms, address words and numbers, and components. Corpus statistics are computed on the corpus being matched. The test set gets its own statistics in Section 17, which is what lets the model adapt to the unseen French data:
# - name genericness: exact-name duplicate counts and token document frequencies;
# - the **locality vocabulary**: address components that recur in Source 1.

In [ ]:
# ── 9.1 Finalise the normalised frames (checkpointed) ───────────────────────
pipeline.train_finalize(st)
display(st.S1[["raw_name", "name_core", "name_skel", "name_legal", "addr_full", "addr_nums"]].head(6))
print(f"locality vocabulary: {len(st.geo):,} components")
print(f"memory: S1 {st.S1.memory_usage(deep=True).sum()/1e9:.2f} GB,  "
      f"S2+S3 {st.X.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
# **9.2 Dense retrieval view (GPU).** The previous run lost most of its score at blocking: hold-out candidate recall was 98.0%, while the same code reached 99.9% on a 20k-entity slice. The cause is scale. The sparse views drop every token whose document frequency exceeds an absolute cap (3,000). In a block of about a million entities that removes most of a generic name, and a fixed top-k gets crowded out by look-alikes. The dense view has no cap:
#
# - **Model.** `paraphrase-multilingual-MiniLM-L12-v2` (Apache-2.0, 118M parameters) embeds "name | address" (normalised text) into 384 dimensions.
# - **Training.** Symmetric in-batch-negative contrastive loss (InfoNCE, as in DPR / Sentence-BERT) on the true pairs of a separate **dense slice** of training entities (`dense_frac`, 15% of the non-hold-out entities). Batches stay inside one country, so the negatives are hard. Records of the same entity are masked, because they are not negatives.
# - **No leakage.** No other model is fitted on the dense slice: not the pruner, the GBDTs or the cross-encoder. The embedding cosine therefore stays an honest feature, the same reason the cross-encoder is trained 2-fold. The hold-out never touches the encoder.
# - **Retrieval.** Exact cosine top-30 per entity and top-10 per record, computed on the GPU per (country, source) block. The cosine and the list of views that retrieved each pair become pruner and matcher features.
#
# The encoder and every 2M-record embedding part are checkpointed to S3. When the blocking checkpoint already exists, the embeddings are not recomputed.

In [ ]:
# ── 9.2 Dense retrieval view: fine-tune the bi-encoder, embed every record (GPU, checkpointed) ──
pipeline.train_dense(st)
if cfg.dense.enabled:
    pairs = st.stg["dense"].load_json("dense_encoder.ok")["pairs"]
    print(f"dense encoder: trained on {pairs:,} true pairs of {st.is_dense.sum():,} dense-slice entities "
          f"(kept out of every other fit)")
    if st.emb is not None:
        print(f"embeddings: S1 {st.emb[0].shape}, S2+S3 {st.emb[1].shape} ({st.emb[0].dtype})")
    else:
        print("blocking results already saved - embeddings not needed")
else:
    print("dense view off (USE_DENSE=False, no GPU, or the model could not be loaded) - sparse views only")

In [ ]:
#
# ▌10. CANDIDATE GENERATION (BLOCKING)
#
# The five TF-IDF views are built per (country, target source) block, and IDF is always fitted on the whole block:
#
# | view | catches |
# |---|---|
# | name tokens (+ space-free name) | exact tokens, word-order swaps, domain names |
# | name character 4-grams | typos, OCR noise |
# | name phonetic skeleton | Indic-script names transliterated to Latin |
# | address words / numbers / number×word | reordered components, missing parts |
# | name×address conjunctions | very specific keys when every single token is common |
#
# Tokens above a document-frequency cap are dropped. Each sparse view contributes the forward top-15 records per entity and the reverse top-5 entities per record. The **dense view** (9.2) adds its exact embedding-cosine top-30 per entity and top-10 per record, with no frequency cap. Every pair records which lists retrieved it (`via`). The steps are:
#
# 1. **Union, one block at a time, for every Source 1 entity.** The five sparse views of a block are built in parallel, the dense top-k runs on the GPU, and each block's union is checkpointed to S3 as soon as it is done.
# 2. **Pruner.** A LightGBM is trained on the union rows of the ranker entities, which is exactly the candidate distribution it will prune. Its features are the view cosines, including the dense one, and the number of sparse and dense lists that retrieved the pair.
# 3. **Pruning.** Each block keeps **16 candidates per entity per source**, plus every record's single best entity, again checkpointed block by block.
#
# **Recall here is the ceiling for everything downstream.**

In [ ]:
# ── 10.1 Candidate union per block, pruner, pruning (checkpointed per block) ─
pipeline.train_blocking(st)
bm = {k: st.metrics.get(k) for k in ("union_pairs", "cand_pairs", "cand_per_s1", "union_recall_eval_sparse_only",
                                     "union_recall_eval", "cand_recall_eval")}
display(pd.Series(bm, name="value").to_frame())
print("previous run (sparse top-10 / top-3, 12 kept): union_recall_eval 0.98111, cand_recall_eval 0.97956")

In [ ]:
# ── 10.2 Candidate-set diagnostics ───────────────────────────────────────────
cand = st.cand
per_s1 = cand.groupby("s1_idx").size()
is_true = st.owner[cand["x_idx"].values] == cand["s1_idx"].values
# candidates are stored sorted by (entity, source, pruner score desc) inside each block
rank = cand.groupby(["s1_idx", "src"], sort=False).cumcount().values + 1
rt = rank[is_true]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].hist(per_s1.values, bins=range(0, per_s1.max() + 2), color="#55A868")
ax[0].set_title("Candidates per S1 entity"); ax[0].set_xlabel("candidates")
ax[1].hist(np.minimum(rt, 15), bins=range(1, 17), color="#C44E52")
ax[1].set_title("Pruner rank of the TRUE matches (per source)"); ax[1].set_xlabel("rank (15 = 15+)")
plt.tight_layout(); plt.show()
print(f"true matches ranked 1-3: {(rt <= 3).mean():.2%}   ranked 1-6: {(rt <= 6).mean():.2%}")
# which retrieval lists found the true candidates (evaluation entities only)
ev = (st.is_hold if st.is_hold.any() else st.is_fit)[cand["s1_idx"].values] & is_true
via = cand["via"].values[ev]
sp, dn = (via & blocking.VIA_SPARSE) > 0, (via & blocking.VIA_DENSE) > 0
display(pd.Series({"sparse and dense": (sp & dn).mean(), "sparse views only": (sp & ~dn).mean(),
                   "dense view only": (~sp & dn).mean()}, name="share of true candidates").to_frame().round(4))
display(pd.Series({v: ((via & (blocking.VIA_BIT[(v, "fwd")] | blocking.VIA_BIT[(v, "rev")])) > 0).mean()
                   for v in (*blocking.VIEWS, "dense")}, name="found by view").to_frame().round(4))
del rank, rt, is_true, per_s1, ev, via, sp, dn

In [ ]:
# **10.3 (optional) Blocking what-if, sparse views only.** Recall is the ceiling of everything downstream. This read-only cell measures, on a sample of hold-out entities, the rank of every true record in every **sparse** view. That gives the exact sparse-union recall for any forward / reverse top-k and for higher frequency caps, together with the union size each setting would produce. Set `BLOCK_K_FWD`, `BLOCK_K_REV` and `BLOCK_CAP_X` in 3.1 from its table. It is off by default (`RUN_BLOCKING_WHATIF` in 3.1) because it rebuilds every block's TF-IDF matrices twice.
#
# - **The dense view comes on top.** Its extra recall is added to these numbers: compare `union_recall_eval` with `union_recall_eval_sparse_only` in 10.1.
# - **Without a hold-out** (`HOLDOUT_FRAC = 0`) the sample is drawn from the fit entities instead. The sparse views learn nothing from labels except the token aliases, so this is only slightly optimistic.
# - **Union sizes** are calibrated on each block's sparse-only union. Blocks built before that count was recorded show a rough, uncalibrated estimate.

In [ ]:
# ── 10.3 (optional, read-only) Blocking what-if: sparse-view recall at larger k / higher caps ──
# Training data only. For a sample of evaluation entities (hold-out; fit entities
# when HOLDOUT_FRAC = 0) the rank of every true record is measured in every sparse
# view - forward (among the block's records, for the entity) and reverse (among the
# country's entities, for the record) - so the union recall of any (k, k_reverse)
# is known exactly without rebuilding the union. Each cap multiplier rebuilds a
# block's TF-IDF matrices once. Nothing is written and no checkpoint changes.
import time
from dataclasses import replace
from ber import blocking

S1, X, bc0 = st.S1, st.X, st.cfg.blocking
SAMPLE_PER_COUNTRY = 30_000          # evaluation entities analysed per country
K_FWD = tuple(sorted({10, 15, 20, 30, 50, bc0.k_name_tok}))    # forward top-k per sparse view (current included)
K_REV = tuple(sorted({3, 5, 10, 20, bc0.k_reverse}))           # reverse top-k per sparse view (current included)
CAP_MULTS = (1, 3)                   # max_df caps x this factor (1 = current caps)

t_all = time.time()
KMAX = max(max(K_FWD), max(K_REV))
rng = np.random.default_rng(0)
s1_country, x_country, x_src = S1["country"].values, X["country"].values, X["src"].values
if st.is_hold.any():
    pool, pool_name = st.is_hold, "hold-out"
else:                                # HOLDOUT_FRAC = 0: sparse views use no labels except the aliases
    pool, pool_name = st.is_fit, "fit (no hold-out in this run; slightly optimistic)"
sample = np.zeros(len(S1), bool)
for c in np.unique(s1_country[pool]):
    pos = np.flatnonzero(pool & (s1_country == c))
    sample[rng.choice(pos, min(SAMPLE_PER_COUNTRY, len(pos)), replace=False)] = True
gx = pd.Index(np.asarray(X["entity_id"].values, dtype=object)).get_indexer(st.gt_x)
in_s = (st.gt_gi >= 0) & sample[np.clip(st.gt_gi, 0, None)]
n_true = int(in_s.sum())                                   # every true pair of the sample (recall denominator)
tp_ = pd.DataFrame({"s1": st.gt_gi[in_s], "x": gx[in_s]})
tp_ = tp_[tp_["x"] >= 0].reset_index(drop=True)
same_ctry = x_country[tp_["x"].values] == s1_country[tp_["s1"].values]
print(f"sample: {int(sample.sum()):,} {pool_name} entities, {n_true:,} true pairs "
      f"({n_true - len(tp_):,} with an id absent from the files, {int((~same_ctry).sum()):,} cross-country)")
RUN_WHATIF = RUN_BLOCKING_WHATIF and n_true > 0
if not RUN_BLOCKING_WHATIF:
    print("blocking what-if skipped (RUN_BLOCKING_WHATIF=False in 3.1)")
elif not RUN_WHATIF:
    print("no true pairs in the sample - nothing to measure, cell skipped")


def _ranks(A, B, r, c, chunk):
    # rank (1-based, ties counted ahead) of column c[i] in row r[i] of A @ B.T - inf when
    # the pair shares no token - and every row's top-KMAX as (row, col, rank0)
    BT = B.T.tocsc()
    out = np.full(len(r), np.inf)
    ur, inv = np.unique(r, return_inverse=True)
    top = ([], [], [])
    for s in range(0, len(ur), chunk):
        rows = ur[s:s + chunk]
        M = (A[rows] @ BT).tocsr()
        M.sum_duplicates()
        rr = np.repeat(np.arange(M.shape[0]), np.diff(M.indptr))
        order = np.lexsort((-M.data, rr))
        rs, ds = rr[order], M.data[order].astype(np.float64)
        key = rs * 4.0 + (2.0 - ds)                          # ascending: row, then cosine descending
        start = M.indptr[:-1]
        rank0 = np.arange(len(order)) - start[rs]
        keep = rank0 < KMAX
        top[0].append(rows[rs[keep]]); top[1].append(M.indices[order][keep]); top[2].append(rank0[keep])
        sel = np.flatnonzero((inv >= s) & (inv < s + len(rows)))
        if len(sel):
            lr = inv[sel] - s
            tv = np.asarray(M[lr, c[sel]]).ravel().astype(np.float64)
            ge = np.searchsorted(key, lr * 4.0 + (2.0 - tv), side="right") - start[lr]
            out[sel] = np.where(tv > 0, ge, np.inf)
    return out, tuple(np.concatenate(t) if t else np.zeros(0, np.int64) for t in top)


res = {m: {v: [np.full(len(tp_), np.inf), np.full(len(tp_), np.inf)] for v in blocking.VIEWS} for m in CAP_MULTS}
size_rows, calibrated = [], True
for country, src, tag in (blocking.block_keys(S1, X) if RUN_WHATIF else []):
    s1_pos = np.flatnonzero(s1_country == country)
    x_pos = np.flatnonzero((x_country == country) & (x_src == src))
    x_in = np.zeros(len(X), bool)
    x_in[x_pos] = True
    pm = np.flatnonzero(same_ctry & (s1_country[tp_["s1"].values] == country) & x_in[tp_["x"].values])
    la = np.searchsorted(s1_pos, tp_["s1"].values[pm])
    lb = np.searchsorted(x_pos, tp_["x"].values[pm])
    s_rows = np.searchsorted(s1_pos, np.flatnonzero(sample & (s1_country == country)))
    n_xs = max(len(np.unique(lb)), 1)
    info = st.stg["block"].load_json(f"cand/{tag}.json")
    # this cell estimates SPARSE unions: calibrate on the block's sparse-only union when it was recorded
    real_union = info.get("union_pairs_sparse")
    calibrated &= real_union is not None
    for m in CAP_MULTS:
        t0 = time.time()
        bc = replace(bc0, **{f"max_df_{v}": getattr(bc0, f"max_df_{v}") * m for v in blocking.VIEWS})
        mats = blocking.build_view_matrices(S1.iloc[s1_pos], X.iloc[x_pos], bc, st.cfg.n_jobs)
        t_build, t1 = time.time() - t0, time.time()
        chunk = max(200, 1000 // m)
        fwd_top, rev_top = [], []
        for v in blocking.VIEWS:
            A, B = mats[v]
            # one product for the true pairs and the union-size rows (same sampled entities)
            f, ft = _ranks(A, B, np.r_[la, s_rows], np.r_[lb, np.zeros(len(s_rows), np.int64)], chunk)
            r, rt = _ranks(B, A, lb, la, chunk)
            res[m][v][0][pm], res[m][v][1][pm] = f[:len(la)], r
            fwd_top.append(ft)
            rev_top.append(rt)
        t_rank = time.time() - t1
        for kf in K_FWD:
            fk = len(np.unique(np.concatenate([t[0][t[2] < kf] * len(x_pos) + t[1][t[2] < kf] for t in fwd_top])))
            for kr in K_REV:
                n_rev = sum(int((t[2] < kr).sum()) for t in rev_top) / n_xs
                size_rows.append({"cap_x": m, "k_fwd": kf, "k_rev": kr, "block": tag, "real_union": real_union,
                                  "raw": len(s1_pos) * fk / max(len(s_rows), 1) + len(x_pos) * n_rev})
        print(f"{country}/S{src} caps x{m}: {len(s1_pos):,} S1 x {len(x_pos):,} records, {len(pm):,} true pairs "
              f"| TF-IDF build {t_build:.0f}s, ranks {t_rank:.0f}s  (elapsed {time.time() - t_all:.0f}s)")
        del mats
        gc.collect()

if RUN_WHATIF:
    # Union size per setting: the sample-based estimate, scaled per block so that the CURRENT
    # setting reproduces the block's real sparse-only union (rough, unscaled estimate otherwise)
    sz = pd.DataFrame(size_rows)
    sz["real_union"] = pd.to_numeric(sz["real_union"], errors="coerce")   # None (not recorded) -> NaN
    cur = sz[(sz["cap_x"] == 1) & (sz["k_fwd"] == bc0.k_name_tok) & (sz["k_rev"] == bc0.k_reverse)]
    sz = sz.merge(cur[["block", "raw"]].rename(columns={"raw": "raw_cur"}), on="block", how="left")
    ok = sz["real_union"].notna() & (sz["raw_cur"] > 0)
    sz["est"] = np.where(ok, sz["raw"] / sz["raw_cur"].where(ok, 1.0) * sz["real_union"].where(ok, 0.0), sz["raw"])
    size = sz.groupby(["cap_x", "k_fwd", "k_rev"])["est"].sum().div(1e6).rename("union_M").reset_index()

    tab = []
    for m in CAP_MULTS:
        fmin = np.vstack([res[m][v][0] for v in blocking.VIEWS]).min(0)
        rmin = np.vstack([res[m][v][1] for v in blocking.VIEWS]).min(0)
        for kf in K_FWD:
            for kr in K_REV:
                tab.append({"cap_x": m, "k_fwd": kf, "k_rev": kr,
                            "recall": float(((fmin <= kf) | (rmin <= kr)).sum() / n_true)})
        never = np.isinf(fmin) & np.isinf(rmin)
        print(f"caps x{m}: true pairs sharing no token in any SPARSE view (unreachable by them at any k): "
              f"{int(never.sum()):,} = {never.sum() / n_true:.3%}")
    tab = pd.DataFrame(tab).merge(size, on=["cap_x", "k_fwd", "k_rev"])
    print("\nreal union_recall_eval (all views):", round(st.metrics["union_recall_eval"], 5),
          "  sparse views only:", round(st.metrics.get("union_recall_eval_sparse_only", float("nan")), 5),
          "  real union pairs (M):", round(st.metrics["union_pairs"] / 1e6, 1))
    print(f"{pool_name} sample, sparse-union recall (ties counted against the true record; "
          f"current = caps x1, k_fwd {bc0.k_name_tok}, k_rev {bc0.k_reverse}):")
    display(tab.pivot_table(index=["cap_x", "k_fwd"], columns="k_rev", values="recall").round(5))
    print("estimated SPARSE union pairs over all blocks (millions)"
          + ("" if calibrated else " - rough: some blocks predate the sparse-only count, re-run blocking to calibrate") + ":")
    display(tab.pivot_table(index=["cap_x", "k_fwd"], columns="k_rev", values="union_M").round(0))
    best = tab["recall"].max()
    near = tab[tab["recall"] >= best - 0.0005].sort_values("union_M")
    if len(near):
        pick = near.iloc[0]
        print(f"best recall {best:.5f}; cheapest setting within 0.05 pt of it: caps x{int(pick.cap_x)}, "
              f"k_fwd {int(pick.k_fwd)}, k_rev {int(pick.k_rev)} -> recall {pick.recall:.5f}, ~{pick.union_M:.0f}M union pairs")
print(f"done in {time.time() - t_all:.0f}s")
# global references to the text frames would keep them in memory while LightGBM trains
# (the pipeline releases st.S1 / st.X there): drop them and the sample arrays
del S1, X, s1_country, x_country, x_src, sample, gx, in_s, tp_, same_ctry, res
gc.collect()

In [ ]:
#
# ▌11. STAGE-1 MATCHER
#
# About 80 features per candidate pair:
# - **Names.** Several string similarities on the full, core, space-free and phonetic-skeleton views; legal-form agreement or conflict; name genericness.
# - **Addresses.** Full, word and street-part similarity; house-number agreement (exact, fuzzy, numeric distance); locality-component agreement or conflict.
# - **Blocking.** View cosines, pruner score, and ranks and gaps within the entity and within the record.
#
# LightGBM is trained with **5 folds grouped by Source 1 entity**, so no entity appears in both the training and validation folds. The out-of-fold probability `p1` of every fitting row feeds stage 2. Every other candidate (hold-out entities and competing entities) is scored by the fold average, exactly as the test set will be.
#
# **Scale and resume.** The features are written to disk in parts of 2 million rows, and each part is checkpointed to S3. LightGBM reads the parts as a stream into its compact binned format, so the float matrix never has to fit in RAM. Each finished fold (model and predictions) is checkpointed too, so an interrupted training continues at the next fold.

In [ ]:
# ── 11.1 Stage-1 features + grouped 5-fold LightGBM (checkpointed per part / fold)
pipeline.train_stage1(st)
print(f"stage-1 features: {len(st.f1_cols)}   model rows: {len(st.model_rows):,}   "
      f"(fit rows {st.n_fit_rows:,})")

In [ ]:
# ── 11.2 Stage-1 diagnostics: out-of-fold quality + feature importance ──────
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
nf = st.n_fit_rows
y_fit = st.y[st.model_rows][:nf]; p_fit = st.p1[st.model_rows][:nf]
print(f"OOF  AUC {roc_auc_score(y_fit, p_fit):.5f}   PR-AUC {average_precision_score(y_fit, p_fit):.5f}   "
      f"log-loss {log_loss(y_fit, np.clip(p_fit, 1e-7, 1-1e-7)):.5f}")
imp = pd.read_csv(st.art / "stage1_importance.csv", index_col=0)["gain"].head(25)[::-1]
imp.plot.barh(figsize=(8, 7), color="#4C72B0", title="Stage-1 feature importance (gain, top 25)")
plt.tight_layout(); plt.show()

In [ ]:
#
# ▌12. CROSS-ENCODER (OPTIONAL, GPU)
#
# `microsoft/mdeberta-v3-base` (MIT licence, 278M parameters, well under the 8B limit) reads both records jointly as "name | address" pairs. Its multilingual vocabulary natively covers Devanagari, Telugu, Tamil, Bengali and French.
# - **Training.** Uncertain stage-1 pairs (0.01 ≤ p1 ≤ 0.99), plus 20% confident ones for calibration, with sequences up to 128 pieces. It is trained 2-fold over the fit entities, so its scores are also out-of-fold.
# - **Use.** Its score is a stage-2 feature (NaN outside the band), so stage 2 learns how far to trust it.
#
# The step is skipped automatically without a GPU. The model is downloaded from the Hugging Face Hub, or from `CE_MODEL_S3_URI` when the instance has no internet access. If it cannot be loaded, the step is skipped with a message instead of failing. Each trained half (weights and scores) is checkpointed to S3.

In [ ]:
# ── 12.1 Train / score the cross-encoder (skipped without GPU) ──────────────
if cfg.cross_encoder.enabled:
    try:
        if CE_MODEL_S3_URI:                        # offline instances: pretrained weights from S3
            local_model = LOCAL_ROOT / "hf_models" / "mdeberta-v3-base"
            s3_download(CE_MODEL_S3_URI, local_model)
            cfg.cross_encoder.model_name = str(local_model)
        cross_encoder._quiet_transformers()       # expected messages only (new head, tokenizer note)
        from transformers import AutoTokenizer
        AutoTokenizer.from_pretrained(cfg.cross_encoder.model_name)   # fail fast (no internet / no weights)
        print("cross-encoder will train on:", torch.cuda.get_device_name(0))
    except Exception as e:
        print(f"cross-encoder disabled - cannot load {cfg.cross_encoder.model_name}: {type(e).__name__}: {e}")
        cfg.cross_encoder.enabled = False
    cfg.save(st.art / "config.json")
if REQUIRE_RESTORE and not cfg.cross_encoder.enabled:
    raise RuntimeError(
        "the cross-encoder is disabled (no GPU, or the model could not be loaded - see the message above): "
        "stage 2 would train without it. Fix the cause, or set REQUIRE_RESTORE = False in 3.1 to go on without it.")
_ce_restored = cfg.cross_encoder.enabled and pipeline.ce_stage(st).done
print("cross-encoder:", "restored from checkpoint" if _ce_restored else
      "not on S3 yet - training it now (both halves: roughly 3-5 GPU hours; each finished half is saved to S3). "
      "Never run a second notebook on the same checkpoints meanwhile.")
pipeline.train_cross_encoder(st)
band = np.isfinite(st.ce_m)
print(f"cross-encoder enabled: {st.ce_enabled}   scored pairs: {band.sum():,}")
ym = st.y[st.model_rows]
if band.any() and len(np.unique(ym[band])) == 2:
    print(f"cross-encoder AUC on its band: {roc_auc_score(ym[band], st.ce_m[band]):.4f}  "
          f"(stage-1 AUC on the same pairs: {roc_auc_score(ym[band], st.p1[st.model_rows][band]):.4f})")

In [ ]:
#
# ▌13. STAGE-2 MATCHER
#
# Stage 2 adds **context** computed from the out-of-fold `p1` of *all* candidates:
# - **Within the entity.** The pair's rank and gap versus the entity's best candidate per source; the sum of p1, which estimates the expected number of matches; the count of confident candidates.
# - **Within the record.** The pair's rank among every entity that claims the same record, the second-best claimant, and the number of confident claimants. This targets generic names.
# - **Sibling agreement.** Name and address similarity of the record to the entity's most probable *other* candidates, weighted by their p1. This catches records renamed to an unrelated brand at the same address, and rejects look-alikes.
# - **Ownership (collective evidence).** Most remaining errors are records, usually with an empty address, that several look-alike entities compete for. The record's strongest rival claimant and its probability; how well that rival is already served in the same source; the entity's own best other candidate in this source and in the other source; how many Source 1 entities of the country share exactly this name; and how many of the record's claimants carry exactly this name, with their probability mass. An exact-name tie is a coin flip, which a precision-weighted metric prefers to leave out.
# - **Cross-encoder score.**
#
# After training, the rows of the competitor-only entities (the dense slice) are scored by the fold-average stage-2 models, cross-encoder included, exactly as every test row is. Stage 3 and the decision step then see a stage-2 probability for every candidate. The context features, every finished fold and these scores are checkpointed to S3.

In [ ]:
# ── 13.1 Context features + grouped 5-fold LightGBM ─────────────────────────
pipeline.train_stage2(st)
nf = st.n_fit_rows
y_fit = st.y[st.model_rows][:nf]
print(f"stage-2 OOF AUC {roc_auc_score(y_fit, st.p2m[:nf]):.5f}   "
      f"PR-AUC {average_precision_score(y_fit, st.p2m[:nf]):.5f}")
imp = pd.read_csv(st.art / "stage2_importance.csv", index_col=0)["gain"].head(25)[::-1]
imp.plot.barh(figsize=(8, 7), color="#8172B2", title="Stage-2 feature importance (gain, top 25)")
plt.tight_layout(); plt.show()

In [ ]:
# **13.2 Stage 3: collective refinement.** Most remaining errors are ownership questions: several look-alike Source 1 entities (often the same name, and a record with no address) compete for one record. Stage 2 judged those competitions with **stage-1** probabilities of the competitors. Stage 3 recomputes the same competition and sibling-anchor features from the **stage-2** probabilities, which already include the cross-encoder. It adds the record's margin over its strongest competing claimant (`q_claim_margin`) and trains another grouped 5-fold LightGBM. This is stacked collective classification: each round sees better estimates of its neighbours.
#
# - **Consistency.** The stage-2 inputs are out-of-fold for the fit entities, and the fold assignment is the same, so stage 3 stays honest. Competitors outside the fit and hold-out entities (the dense slice) get the fold-average stage-2 probability, cross-encoder included, exactly as every test row does, so stage 3 sees the same kind of input in training and at inference. After training, those competitor rows are scored by the fold-average stage-3 models too.
# - **Safety.** Section 14 tunes the decision rule on stage 2, on stage 3 and on their mean, and keeps the one that wins on **out-of-fold** data. The hold-out scores of all three are reported and are never used for the choice.

In [ ]:
# ── 13.2 Stage 3: context from p2 + grouped 5-fold LightGBM (checkpointed per fold) ──
pipeline.train_stage3(st)
if st.p3m is not None:
    nf = st.n_fit_rows
    y_fit = st.y[st.model_rows][:nf]
    print(f"stage-3 OOF AUC {roc_auc_score(y_fit, st.p3m[:nf]):.5f}   PR-AUC {average_precision_score(y_fit, st.p3m[:nf]):.5f}"
          f"   (stage 2: {roc_auc_score(y_fit, st.p2m[:nf]):.5f} / {average_precision_score(y_fit, st.p2m[:nf]):.5f})")
    imp = pd.read_csv(st.art / "stage3_importance.csv", index_col=0)["gain"].head(20)[::-1]
    imp.plot.barh(figsize=(8, 6), color="#937860", title="Stage-3 feature importance (gain, top 20)")
    plt.tight_layout(); plt.show()
else:
    print("stage 3 off (USE_STAGE3=False)")

In [ ]:
#
# ▌14. DECISION RULE & HOLD-OUT VALIDATION
#
# The metric is F0.5 per Source 1 entity, averaged, and a singleton scores 1 **only** for an empty list. For each entity the pipeline selects the top-k candidates that maximise the **exact expected F0.5**:
#
# $$E[F_{0.5}\mid k]=\sum_{a,b}P(TP=a\mid\text{top-}k)\,P(FN=b\mid\text{rest})\,\frac{1.25\,a}{0.25\,(a+b)+k},\qquad E[F_{0.5}\mid 0]=P(\text{no true match})$$
#
# Both distributions are Poisson-binomial, computed exactly by dynamic programming. The grid search also tries:
# - **Recalibration.** A temperature and a calibration offset on the logit scale (Platt scaling). The previous run's best rule was a plain threshold, a sign that the probabilities were shifted.
# - **Probability sharing.** A record belongs to at most one entity, but each pair was scored on its own. The rule either uses the raw probabilities, divides each record's claims by their total when it exceeds 1, or lets only the record's most probable claimant keep its probability.
# - **A plain global threshold.**
# - **One-owner exclusivity.** A record goes only to its most probable entity.
#
# The rule is applied to the **whole candidate graph at once**, exactly as on the test set, so sharing and exclusivity see every entity claiming a record, not only the evaluated ones. The winner, and the final probability (stage 2, stage 3 or their mean), are chosen on **out-of-fold** data of the fit entities and then scored once on the untouched **hold-out** entities.

In [ ]:
# ── 14.1 Tune on OOF, evaluate on the hold-out ──────────────────────────────
metrics = pipeline.train_decision(st)
keys = ["cand_recall_eval", "oof_macro_f05_stage2", "oof_macro_f05_stage3", "oof_macro_f05_stage23", "final_stage",
        "oof_macro_f05", "holdout_macro_f05_stage2", "holdout_macro_f05_stage3", "holdout_macro_f05_stage23",
        "holdout_macro_f05", "holdout_macro_f05_stage1_t05", "holdout_macro_f05_oracle",
        "holdout_pair_precision", "holdout_pair_recall"]
display(pd.Series({k: metrics.get(k) for k in keys}, name="value").to_frame())
print("chosen decision rule:", metrics["decision"], "  on stage", metrics.get("final_stage", 2),
      "(23 = mean of stages 2 and 3)")
print("previous run: holdout_macro_f05 0.98843, oracle 0.99350, leaderboard 0.985")

In [ ]:
# ── 14.2 Decision grid: macro F0.5 of every rule tried (out-of-fold) ────────
g = pd.DataFrame([{**p, "score": s} for s, p in st.decision_grid])
g["prob"] = g["prob"].fillna("raw") if "prob" in g else "raw"
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for (prob, excl), sub in g[g["rule"] == "threshold"].groupby(["prob", "exclusive"]):
    axes[0].plot(sub["threshold"], sub["score"], marker=".", label=f"{prob}, exclusive={excl}")
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("OOF macro F0.5"); axes[0].set_title("Global threshold")
axes[0].legend(fontsize=8)
ef = g[(g["rule"] == "expf") & (g.get("bias", 0.0) == 0.0)]
for (prob, excl), sub in ef.groupby(["prob", "exclusive"]):
    axes[1].plot(sub["temperature"], sub["score"], marker=".", label=f"{prob}, exclusive={excl}")
axes[1].set_xlabel("temperature"); axes[1].set_title("Expected-F (calibration offset 0)")
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()
print("best 10 rules (out-of-fold):")
display(g.sort_values("score", ascending=False).head(10))

In [ ]:
# ── 14.3 F0.5 broken down by the number of true matches ─────────────────────
from types import SimpleNamespace


def per_entity_f(codes, label, sel, n_gt, beta=0.5):
    n = len(n_gt)
    tp = np.bincount(codes, weights=(label * sel).astype(float), minlength=n)
    npred = np.bincount(codes, weights=sel.astype(float), minlength=n)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.where(npred > 0, tp / npred, 0); R = np.where(n_gt > 0, tp / n_gt, 0)
        f = np.where(tp > 0, (1 + beta**2) * P * R / (beta**2 * P + R), 0.0)
    return np.where((n_gt == 0) & (npred == 0), 1.0, f)


def eval_view():
    # The entities 14.3-15.1 evaluate: the hold-out when this run has one; otherwise
    # (HOLDOUT_FRAC = 0) the fit entities scored by their OUT-OF-FOLD predictions with
    # the chosen decision rule - honest per entity, the rule itself was tuned on them.
    nf = st.n_fit_rows
    if st.is_hold.any():
        ents, rows, p = np.flatnonzero(st.is_hold), st.model_rows[nf:], st.pfin[nf:]
        sel, name = np.asarray(st.holdout_sel, bool), "hold-out"
    else:
        ents, rows, p = np.flatnonzero(st.is_fit), st.model_rows[:nf], st.pfin[:nf]
        sel, name = None, "out-of-fold (fit entities; no hold-out in this run)"
    codes = np.searchsorted(ents, st.cand["s1_idx"].values[rows])
    if sel is None:                   # the rule applied to the whole candidate graph, as on the test set
        sel = st.sel_full[rows]
    return SimpleNamespace(ents=ents, rows=rows, p=p, sel=np.asarray(sel, bool), codes=codes,
                           y=st.y[rows].astype(bool), n_gt=st.n_gt[ents], name=name)


EV = eval_view()
f = per_entity_f(EV.codes, EV.y, EV.sel, EV.n_gt)
print(f"evaluated on: {EV.name} - {len(EV.ents):,} entities, macro F0.5 {f.mean():.5f}")
br = pd.DataFrame({"true matches": np.minimum(EV.n_gt, 8), "F0.5": f}) \
       .groupby("true matches")["F0.5"].agg(["mean", "size"])
br.index = [str(i) if i < 8 else "8+" for i in br.index]
display(br.rename(columns={"mean": "F0.5", "size": "entities"}))

In [ ]:
# ── 14.4 (optional, read-only) Where the F0.5 is lost ────────────────────────
# Splits (1 - macro F0.5) of the evaluated entities (14.3: hold-out, or out-of-fold
# fit entities without a hold-out) into the part lost at blocking (true records
# that never became candidates) and the part lost by the model / decision, the
# latter by kind of error and per country. Training data only; nothing is written.
def _ent_f(codes, label, sel, n_gt, beta=0.5):
    n = len(n_gt)
    tp = np.bincount(codes, weights=(label & sel).astype(float), minlength=n)
    npred = np.bincount(codes, weights=sel.astype(float), minlength=n)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.where(npred > 0, tp / npred, 0.0)
        R = np.where(n_gt > 0, tp / n_gt, 0.0)
        f = np.where(tp > 0, (1 + beta**2) * P * R / (beta**2 * P + R), 0.0)
    return np.where((n_gt == 0) & (npred == 0), 1.0, f), tp, npred


codes, yh, selh, n_gt = EV.codes, EV.y, EV.sel, EV.n_gt
f_act, tp, npred = _ent_f(codes, yh, selh, n_gt)
f_orc, reach, _ = _ent_f(codes, yh, yh, n_gt)
n = len(EV.ents)
kind = np.select(
    [(n_gt == 0) & (npred > 0), (n_gt > 0) & (npred == 0) & (reach > 0),
     (n_gt > 0) & (npred > tp), (n_gt > 0) & (tp < reach)],
    ["singleton given a match (scores 0)", "matchable entity left empty (scores 0)",
     "entity with >=1 wrong match", "entity missing reachable matches (no wrong one)"], "correct")
print(f"[{EV.name}] macro F0.5 {f_act.mean():.5f} | ceiling with these candidates {f_orc.mean():.5f} | "
      f"lost at blocking {1 - f_orc.mean():.5f} | lost by model + decision {f_orc.mean() - f_act.mean():.5f}")
d = pd.DataFrame({"kind": kind, "loss": f_orc - f_act})
display(d.groupby("kind")["loss"].agg(entities="size", loss_points=lambda s: s.sum() / n)
        .sort_values("loss_points", ascending=False).round(5))

xi = st.cand["x_idx"].values[EV.rows]
fp, fn = selh & ~yh, ~selh & yh
own = st.owner[xi[fp]]
print(f"wrong matches: {int(fp.sum()):,}   record owned by another S1 entity: "
      f"{(own >= 0).mean() if fp.any() else 0:.1%}")
if fn.any():
    print("final probability of the missed reachable true matches (10/25/50/75/90 %):",
          np.round(np.quantile(EV.p[fn], [0.1, 0.25, 0.5, 0.75, 0.9]), 3))
if fp.any():
    print("final probability of the wrong matches (10/25/50/75/90 %):",
          np.round(np.quantile(EV.p[fp], [0.1, 0.25, 0.5, 0.75, 0.9]), 3))

ctry = st.S1["country"].values[EV.ents]
by = pd.DataFrame({"country": ctry, "F0.5": f_act, "ceiling": f_orc}).groupby("country").mean()
by["entities"] = pd.Series(ctry).value_counts()
display(by.round(5))


def _share(col, mask):
    return float(st.X[col].values[xi[mask]].mean()) if mask.any() else np.nan


display(pd.DataFrame({name: [_share("name_nonlatin", m), _share("addr_empty", m)]
                      for name, m in (("all true pairs", yh), ("missed (FN)", fn), ("wrong (FP)", fp))},
                     index=["record name non-Latin", "record address empty"]).round(3))

In [ ]:
# **14.5 Ambiguity floor: the best score any model could reach.** Some true records carry no information that picks their owner. A record with an **empty address**, whose Source 1 entity shares its exact normalised name with **other Source 1 entities of the same country**, reads identically against all of them. Under a precision-weighted F0.5 the best action for such a record is to leave it out. Scoring the hold-out as if every other true record were found, and every such record were left out, estimates the ceiling for *any* pipeline on this data. If this number is below your target, no model change can reach the target.
#
# It is an estimate, not a strict bound:
# - a model may still resolve some of these records from small name differences (legal form, word order);
# - other kinds of ambiguity are not counted.

In [ ]:
# ── 14.5 (read-only) Ambiguity floor ─────────────────────────────────────────
# Uses the entities of 14.3 (hold-out, or out-of-fold fit entities without a hold-out);
# the ceiling itself needs labels only, no model.
key = pd.Series(st.S1["country"].values).astype(str) + "|" + pd.Series(st.S1["name_core"].values)
n_same = key.map(key.value_counts()).values                 # S1 entities with this (country, core name)
gx = pd.Index(np.asarray(st.X["entity_id"].values, dtype=object)).get_indexer(st.gt_x)
ok = (st.gt_gi >= 0) & (gx >= 0)
gi, gxx = st.gt_gi[ok], gx[ok]
in_ev = np.zeros(len(st.S1), bool); in_ev[EV.ents] = True
gi, gxx = gi[in_ev[gi]], gxx[in_ev[gi]]
amb = (st.X["addr_empty"].values[gxx] == 1) & (n_same[gi] >= 2)
n_gt_e = EV.n_gt
tp_floor = np.bincount(np.searchsorted(EV.ents, gi[~amb]), minlength=len(EV.ents)).astype(float)
with np.errstate(divide="ignore", invalid="ignore"):
    R = np.where(n_gt_e > 0, tp_floor / n_gt_e, 0.0)
    f_ceiling = np.where(tp_floor > 0, 1.25 * R / (0.25 + R), 0.0)   # precision 1 on what is predicted
f_ceiling = np.where(n_gt_e == 0, 1.0, f_ceiling)
has_amb = np.bincount(np.searchsorted(EV.ents, gi[amb]), minlength=len(EV.ents)) > 0
f_now = per_entity_f(EV.codes, EV.y, EV.sel, n_gt_e)
print(f"[{EV.name}] true pairs: {len(amb):,}   ambiguous (empty address, name shared by >=2 S1 of the country): "
      f"{int(amb.sum()):,} = {amb.mean() if len(amb) else 0:.3%}")
print(f"entities with >=1 ambiguous true record: {has_amb.sum():,} of {len(EV.ents):,} ({has_amb.mean():.2%})")
print(f"AMBIGUITY CEILING (every resolvable record found, every ambiguous one left out): {f_ceiling.mean():.5f}")
print(f"current macro F0.5 on the same entities: {f_now.mean():.5f}")
lost = (1 - f_now)
display(pd.DataFrame({"entities": [int(has_amb.sum()), int((~has_amb).sum())],
                      "loss points (of 1 - F0.5)": [lost[has_amb].sum() / len(lost), lost[~has_amb].sum() / len(lost)],
                      "irreducible part": [(1 - f_ceiling[has_amb]).sum() / len(lost), 0.0]},
                     index=["with ambiguous records", "without"]).round(5))
print("leaderboard ~ hold-out - 0.003 so far: a leaderboard above 0.996 needs a hold-out near 0.999, "
      "so this ceiling must be at least that high")

In [ ]:
#
# ▌15. ERROR ANALYSIS
#
# A random sample of wrong merges (false positives) and missed links (false negatives), shown side by side. They are drawn from the hold-out, or from the out-of-fold fit entities when the run has no hold-out.

In [ ]:
# ── 15.1 False positives and false negatives on the hold-out ────────────────
def show_pairs(mask, rows, ph, title, n=10):
    idx = np.flatnonzero(mask)
    idx = np.random.default_rng(0).choice(idx, min(n, len(idx)), replace=False) if len(idx) else idx
    r = rows[idx]
    s1i, xi = st.cand["s1_idx"].values[r], st.cand["x_idx"].values[r]
    df = pd.DataFrame({"S1 name": st.S1["raw_name"].values[s1i], "S1 address": st.S1["raw_addr"].values[s1i],
                       "record name": st.X["raw_name"].values[xi], "record address": st.X["raw_addr"].values[xi],
                       "p": np.round(ph[idx], 3)})
    display(Markdown(f"**{title}** ({mask.sum():,} in total)")); display(df)

print("evaluated on:", EV.name)
show_pairs(EV.sel & ~EV.y, EV.rows, EV.p, "False positives (predicted, not a true match)")
show_pairs(~EV.sel & EV.y, EV.rows, EV.p, "False negatives among candidates (true match, not predicted)")

In [ ]:
# **15.2 Corpus-size check (read-only, training data only).** The test corpus has its own size, and several signals depend on how many look-alikes a corpus holds: name-duplicate counts, claimants per record, IDF, top-k crowding. The hold-out is always scored inside the full training corpus. This cell measures whether the score moves with corpus size, which is one possible cause of the hold-out → leaderboard gap:
#
# 1. For each fraction in `SIZE_CHECK_FRACS`, a corpus is rebuilt **from the training files**: that fraction of the Source 1 entities, every record they own, and the same fraction of the records that belong to no entity. The composition matches the training data.
# 2. The trained pipeline runs on it **exactly as on the test set**: its own normalisation statistics, dense embeddings, blocking, all stages and the decision rule.
# 3. Only the **hold-out** entities inside it are scored, and they are compared with the same entities' scores in the full corpus (14.1).
#
# Nothing is trained or tuned here. Each fraction costs roughly that fraction of a test inference.

In [ ]:
# ── 15.2 (optional, read-only) Corpus-size check on train-derived corpora ────
def _dump(path, frame, mask):
    # the raw text exactly as read (no field contains a tab), in the challenge's TSV layout
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        fh.write("entity_id\tbusiness_name\tbusiness_address\tcountry\n")
        for r in zip(*(frame[c].values[mask] for c in ("entity_id", "raw_name", "raw_addr", "country"))):
            fh.write("\t".join(r) + "\n")


def _f05(pred: set, gt: set) -> float:
    if not gt:
        return float(not pred)
    tp = len(pred & gt)
    if tp == 0:
        return 0.0
    P, R = tp / len(pred), tp / len(gt)
    return 1.25 * P * R / (0.25 * P + R)


if SIZE_CHECK_FRACS and st.is_hold.any():
    sim_root = LOCAL_ROOT / "sim_data"
    owned = st.owner >= 0
    hold_ents = np.flatnonzero(st.is_hold); nf = st.n_fit_rows
    codes = np.searchsorted(hold_ents, st.cand["s1_idx"].values[st.model_rows][nf:])
    f_full = pd.Series(per_entity_f(codes, st.y[st.model_rows][nf:], st.holdout_sel, st.n_gt[hold_ents]),
                       index=st.S1["entity_id"].values[hold_ents])
    rng = np.random.default_rng(11)
    res = []
    for frac in SIZE_CHECK_FRACS:
        name = f"sim{int(round(frac * 1000)):04d}"
        pick = rng.random(len(st.S1)) < frac                               # Source 1 entities kept
        keep_x = np.where(owned, pick[np.clip(st.owner, 0, None)], rng.random(len(st.X)) < frac)
        d = sim_root / name
        if not (d / f"{name}_source3.tsv").exists():
            d.mkdir(parents=True, exist_ok=True)
            _dump(d / f"{name}_source1.tsv", st.S1, pick)
            for s in (2, 3):
                _dump(d / f"{name}_source{s}.tsv", st.X, keep_x & (st.X["src"].values == s))
        out = LOCAL_ROOT / "sim_out" / name
        t0 = time.time()
        pipeline.predict(sim_root, name, st.work, out, use_cache=USE_CACHE, ckpt_uri=S3_CKPT_URI,
                         s3_region=S3_REGION, s3_endpoint=S3_ENDPOINT)
        pred = pd.read_csv(out / "matching_results.tsv", sep="\t", dtype=str, keep_default_na=False)
        pm = dict(zip(pred["source1_entity_id"], pred["matched_entity_ids"]))
        ids = st.S1["entity_id"].values[pick & st.is_hold]
        f_sim = np.array([_f05(set(filter(None, pm.get(i, "").split(","))), set(st.gt.get(i, []))) for i in ids])
        res.append({"corpus size": f"{frac:.0%}", "S1 entities": int(pick.sum()), "records": int(keep_x.sum()),
                    "hold-out entities scored": len(ids), "F0.5 in this corpus": f_sim.mean(),
                    "same entities, full corpus (14.1)": f_full.loc[ids].mean(),
                    "minutes": (time.time() - t0) / 60})
        print(res[-1])
    display(pd.DataFrame(res).round(5))
    print("If the score changes clearly with size, corpus-size effects explain part of the leaderboard gap.")
else:
    print("corpus-size check skipped (SIZE_CHECK_FRACS empty or no hold-out)")

In [ ]:
#
# ▌16. UPLOAD THE TRAINED ARTEFACTS TO S3
#
# `WORK_DIR/artifacts` holds everything inference needs: config, aliases, pruner, fold models, cross-encoder weights, feature lists, the decision rule and the metrics. The same files already live in the S3 checkpoints; this step also uploads them as one self-contained folder, `RUN_URI/artifacts/`, which Section 17 can restore on any instance.

In [ ]:
# ── 16.1 Upload artefacts + metrics to S3 ────────────────────────────────────
s3_upload(st.art, RUN_URI + "artifacts/")
(st.work / "run_uri.txt").write_text(RUN_URI)       # 17.1 / 19.1 upload next to these artefacts after a restart
print("artefacts saved to", RUN_URI + "artifacts/")
print(json.dumps(json.loads((st.art / "metrics.json").read_text()), indent=1)[:1500])

In [ ]:
#
# ▌17. INFERENCE ON THE TEST SET
#
# The frozen artefacts are applied to the test split with **the same transforms**. Corpus statistics (IDF, genericness, locality vocabulary and phrases) are computed on the test corpus itself; this needs no labels and adapts the features to France. Candidate rows are processed in chunks, and stage-1 features are staged on disk. The training state is released first to free memory.
#
# In DEV mode the train-derived `deveval` subset is used instead, and it is scored in 17.8.
#
# Inference is checkpointed too: the normalised test frames, each block's candidates and each feature part are uploaded to `S3_CKPT_URI/predict-test-<hash>/`, so an interrupted inference also resumes where it stopped. These files are derived from the test split for inference only; nothing is trained or tuned on them.
#
# > **After a kernel restart** you do not need to retrain. Either run the notebook from the top (every training step restores from its checkpoint in minutes), or run Sections 1–3 and cell 4.1 and continue from here: the trained artefacts are read from `WORK_DIR/artifacts`, or restored from the latest S3 run uploaded in 16.1.

In [ ]:
# ── 17.1 Free the training state and locate the trained artefacts ──────────
TRAINED_WORK_DIR = st.work if "st" in globals() else WORK_DIR
for _name in ("st", "cand", "y_fit", "p_fit", "ym", "band", "rows", "sel", "yh", "ph", "codes", "f", "g", "imp",
              "EV", "selh", "fp", "fn", "xi", "own", "d", "kind", "f_act", "f_orc", "tp", "npred", "reach",
              "amb", "gi", "gxx", "gx", "in_ev", "n_same", "key", "f_now", "f_ceiling", "tp_floor", "has_amb", "lost",
              "tp_", "res", "sample", "in_s", "same_ctry", "S1", "X"):
    globals().pop(_name, None)
gc.collect()

if not (TRAINED_WORK_DIR / "artifacts" / "meta.json").exists():
    latest = s3_latest_run(PROFILE)
    assert latest, f"no trained artefacts locally or under {S3_OUTPUT_URI}runs/ - run Sections 4-16 first"
    print("restoring artefacts from", latest)
    s3_download(latest + "artifacts/", TRAINED_WORK_DIR / "artifacts")
    RUN_URI = latest                               # outputs go next to the artefacts that produced them
    (TRAINED_WORK_DIR / "run_uri.txt").write_text(RUN_URI)
elif (TRAINED_WORK_DIR / "run_uri.txt").exists():  # local artefacts: their run folder, not this session's timestamp
    RUN_URI = (TRAINED_WORK_DIR / "run_uri.txt").read_text().strip()
print("results go to", RUN_URI)
print("artefacts :", TRAINED_WORK_DIR / "artifacts")
print(f"RAM in use: {psutil.virtual_memory().used/1e9:.1f} GB of {RAM_GB:.1f} GB")

In [ ]:
# ── 17.2 Download the test split from S3, then load + normalise it ──────────
if EVAL_SPLIT == "test":
    s3_download(S3_URI.rstrip("/") + "/dataset/test/", DATA_DIR / "test")
ps = pipeline.predict_init(EVAL_DATA_DIR, EVAL_SPLIT, TRAINED_WORK_DIR, OUT_DIR, use_cache=USE_CACHE,
                          ckpt_uri=S3_CKPT_URI, s3_region=S3_REGION, s3_endpoint=S3_ENDPOINT)
pipeline.predict_prepare(ps)
print(f"{EVAL_SPLIT}: {len(ps.S1):,} S1 entities, {len(ps.X):,} S2+S3 records")
display(ps.S1["country"].value_counts().rename("S1 entities per country").to_frame())

In [ ]:
# ── 17.3 Dense embeddings + candidate generation with the trained pruner ────
pipeline.predict_dense(ps)                         # skipped when the dense view was not trained
pipeline.predict_blocking(ps)
print(f"candidate pairs: {len(ps.cand):,}  ({len(ps.cand)/len(ps.S1):.1f} per S1 entity)")
display(pd.DataFrame({"country": ps.S1["country"].values[ps.cand["s1_idx"].values],
                      "source": ps.cand["src"].values})
          .value_counts().rename("candidate pairs").to_frame())

In [ ]:
# ── 17.4 Stage-1 features + scoring (parts on disk, checkpointed) ───────────
pipeline.predict_stage1(ps)
print(f"p1 > 0.5 for {(ps.p1 > 0.5).mean():.2%} of candidate pairs")

In [ ]:
# ── 17.5 Cross-encoder scoring (only if it was trained) ─────────────────────
pipeline.predict_cross_encoder(ps)
print(f"cross-encoder scored pairs: {np.isfinite(ps.ce).sum():,}")

In [ ]:
# ── 17.6 Stage-2 (+ stage-3 when it was chosen) scoring ─────────────────────
pipeline.predict_stage2(ps)
pipeline.predict_stage3(ps)                        # keeps p2 when stage 2 was the final stage
print("final stage:", ps.meta.get("final_stage", 2))
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(ps.pfin, bins=50, log=True, color="#8172B2"); ax.set_title("Final probabilities (log scale)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 17.7 Decision rule + write both submission files ─────────────────────────
summary = pipeline.predict_write(ps)
display(pd.Series(summary, name="value").to_frame())
print(f"entities predicted as singletons: {1 - summary['s1_with_matches']/summary['s1_entities']:.2%}")
display(pd.read_csv(OUT_DIR / "matching_results.tsv", sep="\t", nrows=5, keep_default_na=False))

In [ ]:
# ── 17.8 (DEV mode) score the train-derived evaluation subset ───────────────
if EVAL_SPLIT == "deveval":
    gt_eval = ber_io.read_ground_truth(Path(EVAL_DATA_DIR) / "deveval" / "deveval_ground_truth.tsv")
    pred = pd.read_csv(OUT_DIR / "matching_results.tsv", sep="\t", keep_default_na=False, dtype=str)
    F = []
    for s1, m in zip(pred["source1_entity_id"], pred["matched_entity_ids"]):
        p, g = set(filter(None, m.split(","))), set(gt_eval.get(s1, []))   # absent = singleton
        if not g:
            F.append(float(not p)); continue
        tp = len(p & g)
        F.append(0.0 if tp == 0 else 1.25 * (tp/len(p)) * (tp/len(g)) / (0.25 * tp/len(p) + tp/len(g)))
    print(f"DEV evaluation macro F0.5 = {np.mean(F):.5f} on {len(F):,} entities")

In [ ]:
#
# ▌18. VALIDATE THE SUBMISSION FILES
#
# This runs the challenge's official validator, `utils/validate_submission.py`, and an internal consistency check:
# - one row per Source 1 entity, with no duplicates;
# - S2-/S3- IDs only;
# - matches ⊆ candidates.

In [ ]:
# ── 18.1 Internal checks ─────────────────────────────────────────────────────
def read_lists(path):
    df = pd.read_csv(path, sep="\t", keep_default_na=False, dtype=str)
    return dict(zip(df.iloc[:, 0], df.iloc[:, 1].map(lambda s: [x for x in s.split(",") if x])))
m = read_lists(OUT_DIR / "matching_results.tsv"); c = read_lists(OUT_DIR / "candidate_pairs.tsv")
s1_required = set(ps.S1["entity_id"])
x_valid = set(ps.X["entity_id"])
checks = {
    "one row per S1 entity": set(m) == s1_required and len(m) == len(s1_required),
    "candidate file covers every S1": set(c) == s1_required,
    "no duplicate IDs in a list": all(len(v) == len(set(v)) for v in m.values()),
    "only S2-/S3- IDs": all(x[:3] in ("S2-", "S3-") for v in m.values() for x in v),
    "all IDs exist": all(x in x_valid for v in m.values() for x in v),
    "matches are a subset of candidates": all(set(m[k]) <= set(c.get(k, ())) for k in m),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values()), "submission check failed"
del m, c, x_valid

In [ ]:
# ── 18.2 Official validator (test split only) ────────────────────────────────
if EVAL_SPLIT == "test" and VALIDATOR.exists():
    r = subprocess.run([sys.executable, str(VALIDATOR), "--matching", str(OUT_DIR / "matching_results.tsv"),
                        "--candidate", str(OUT_DIR / "candidate_pairs.tsv"),
                        "--test-dir", str(DATA_DIR / "test")], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0, f"official validator failed (exit code {r.returncode})"
elif EVAL_SPLIT == "test":
    print("utils/validate_submission.py not found - only the internal checks in 18.1 were run")
else:
    print("official validator runs on the real test split only (skipped in DEV mode)")

In [ ]:
#
# ▌19. PACKAGE THE SUBMISSION ZIP AND UPLOAD IT TO S3
#
# This builds `<team>_submission.zip` in the required layout and uploads it, together with the two TSVs, to `RUN_URI`:
#
# ```
# <team>_submission.zip
# ├── output/  matching_results.tsv, candidate_pairs.tsv
# ├── code/business_entity_resolution/  src/, README.md, requirements.txt, docs/
# └── Documentation_template.md
# ```
#
# Before submitting, fill in the team fields and the final metrics in `Documentation_template.md`.

In [ ]:
# ── 19.1 Build the zip ────────────────────────────────────────────────────────
TEAM = "Expecto Patronum"                              # <── EDIT
# <team_name>_submission.zip; spaces become underscores so the file name is safe in any upload / shell
zpath = LOCAL_ROOT / f"{'_'.join(TEAM.split())}_submission.zip"
pkg = Path(PKG)
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("matching_results.tsv", "candidate_pairs.tsv"):
        z.write(OUT_DIR / name, f"output/{name}")
    for p in pkg.rglob("*"):
        if p.is_file() and "__pycache__" not in p.parts:
            z.write(p, f"code/business_entity_resolution/{p.relative_to(pkg).as_posix()}")
    z.write(pkg / "docs" / "Documentation_template.md", "Documentation_template.md")
print("wrote", zpath, f"({zpath.stat().st_size/1e6:.1f} MB)")
s3_upload(zpath, RUN_URI)
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    s3_upload(OUT_DIR / name, RUN_URI + "output/")
print("leaderboard file:", RUN_URI + "output/matching_results.tsv")
print("submission zip  :", RUN_URI + zpath.name)

In [ ]:
#
# ▌20. FINAL REFIT & TUNING LEVERS
#
# **Final refit.** After you are happy with the hold-out score, set `HOLDOUT_FRAC = 0.0` in 3.1 and run the notebook again (each run gets its own `ber_outputs/runs/...` folder on S3). Every entity is then used for fitting; the out-of-fold score in `metrics.json` stays an honest estimate. The static normalisation checkpoint is reused; every later step gets new checkpoints, because its hash includes the hold-out setting, so old and new work are never mixed.
#
# **Reading the metrics**
#
# | metric | meaning |
# |---|---|
# | `cand_recall_eval` | share of true matches that survive blocking: the recall ceiling |
# | `holdout_macro_f05_oracle` | best achievable macro F0.5 with these candidates |
# | `holdout_macro_f05` | the honest validation score of the full pipeline |
# | `holdout_pair_precision` / `_recall` | where the remaining errors are |
#
# **Levers** (in `cfg`, Section 3)
# - **Recall-bound** (candidate recall well below the pair recall you need): raise `cfg.blocking.k_*`, `k_dense*` or `keep_per_source`, raise the `max_df_*` caps (`BLOCK_CAP_X`), or give the dense encoder more pairs / epochs (`cfg.model.dense_frac`, `cfg.dense.epochs`).
# - **Precision-bound:** enable the cross-encoder (GPU), switch `CE_MODEL` to `FacebookAI/xlm-roberta-large`, increase `cross_encoder.max_train_pairs`, or widen `band_low` / `band_high`.
# - **Ceiling:** 14.5 estimates the best score any model could reach on this data. If it is below a target, no model change will reach that target.
# - **More training signal:** use a larger profile or `max_train_s1`.
# - **Checkpoints:** `S3_CKPT_URI/<step>-<hash>/` (local copy in `WORK_DIR/ckpt/`). Changing a setting creates new checkpoints only for the steps that depend on it. After editing the library code itself (e.g. `normalize.py`), set `USE_CACHE = False` once, or bump `CKPT_VERSION` in `pipeline.py`. Old checkpoint folders can be deleted from S3 at any time; they are only read, never required.

In [ ]:
#
# ▌21. UNSEEN-COUNTRY CHECK (OPTIONAL, `RUN_COUNTRY_CHECK`)
#
# The test set contains **France**, which never appears in training, and the leaderboard has been about 0.003 below the hold-out for both pipelines. This section measures how much a *country the models never saw* costs:
#
# 1. Train the whole pipeline on the entities of one training country only. The aliases, noise tokens, dense encoder, pruner, GBDTs, cross-encoder and decision rule are all learned from that country.
# 2. Score a sample of the other country, exactly as the test set would be scored.
# 3. Compare with the trained country's own hold-out score.
#
# Everything comes from `dataset/train`. It is a separate, long run: one full training per direction, at `LOCO_TRAIN_S1` entities, in its own work folder and S3 checkpoint prefix. It never touches the main run's models. India and the US differ more from each other than France differs from the US, so treat the drop as an upper estimate of France's cost.

In [ ]:
# ── 21.1 (optional) Unseen-country check: train on one country, score the other ──
LOCO_PAIRS    = (("US", "India"), ("India", "US"))   # (training country, scored country)
LOCO_TRAIN_S1 = 300_000                                # Source 1 entities of the training country
LOCO_EVAL_S1  = 100_000                                # Source 1 entities of the scored country
import re


def _f05(pred: set, gt: set) -> float:
    # per-entity F0.5 of the challenge (same as in 15.2; repeated so this cell runs on its own)
    if not gt:
        return float(not pred)
    tp = len(pred & gt)
    if tp == 0:
        return 0.0
    P, R = tp / len(pred), tp / len(gt)
    return 1.25 * P * R / (0.25 * P + R)


def _loco_split(raw, gt_owner, country, n_s1, rng):
    # n_s1 entities of one country, every record they own, and the same share of that country's unowned records
    s1 = raw["s1"][raw["s1"]["country"].values == country]
    pick = set(rng.choice(s1["entity_id"].values, min(n_s1, len(s1)), replace=False))
    rate = len(pick) / max(len(s1), 1)
    out = {"s1": s1[s1["entity_id"].isin(pick).values]}
    for k in ("s2", "s3"):
        f = raw[k][raw[k]["country"].values == country]
        own = f["entity_id"].map(gt_owner)
        out[k] = f[(own.isin(pick) | (own.isna() & (rng.random(len(f)) < rate))).values]
    return out, pick


def _loco_write(root, split, frames, gt_all, pick):
    d = root / split
    d.mkdir(parents=True, exist_ok=True)
    for k, f in frames.items():
        with open(d / f"{split}_source{k[-1]}.tsv", "w", encoding="utf-8", newline="\n") as fh:
            fh.write("entity_id\tbusiness_name\tbusiness_address\tcountry\n")
            for r in zip(f["entity_id"].values, f["business_name"].values, f["business_address"].values,
                         f["country"].values):
                fh.write("\t".join(r) + "\n")
    with open(d / f"{split}_ground_truth.tsv", "w", encoding="utf-8", newline="\n") as fh:
        fh.write("source1_entity_id\tmatched_entity_ids\n")
        for s in frames["s1"]["entity_id"].values:
            fh.write(f"{s}\t{','.join(gt_all.get(s, []))}\n")


if RUN_COUNTRY_CHECK:
    import copy
    raw = ber_io.read_split(TRAIN_DATA_DIR, "train")
    gt_all = ber_io.read_ground_truth(Path(TRAIN_DATA_DIR) / "train" / "train_ground_truth.tsv")
    gt_owner = pd.Series({x: s for s, lst in gt_all.items() for x in lst})
    loco_rows = []
    for train_c, eval_c in LOCO_PAIRS:
        root = LOCAL_ROOT / "loco" / re.sub(r"[^A-Za-z0-9]+", "_", f"{train_c}_to_{eval_c}")
        rng = np.random.default_rng(5)
        tr, tr_ids = _loco_split(raw, gt_owner, train_c, LOCO_TRAIN_S1, rng)
        ev, ev_ids = _loco_split(raw, gt_owner, eval_c, LOCO_EVAL_S1, rng)
        if not (root / "train" / "train_ground_truth.tsv").exists():
            _loco_write(root, "train", tr, gt_all, tr_ids)
        if not (root / "loeval" / "loeval_ground_truth.tsv").exists():
            _loco_write(root, "loeval", ev, gt_all, ev_ids)          # its labels are used only to score
        del tr, ev
        cfg_l = copy.deepcopy(cfg)
        cfg_l.model.holdout_frac = 0.15          # the trained country's own hold-out is the comparison point
        cfg_l.model.max_train_s1 = None
        cfg_l.model.ranker_s1 = min(cfg.model.ranker_s1, LOCO_TRAIN_S1 // 2)
        ck = dict(ckpt_uri=f"{S3_CKPT_URI}loco/{root.name}/", s3_region=S3_REGION, s3_endpoint=S3_ENDPOINT)
        m = pipeline.train(root, root / "work", cfg_l, use_cache=USE_CACHE, **ck)
        pipeline.predict(root, "loeval", root / "work", root / "out", use_cache=USE_CACHE, **ck)
        gt_e = ber_io.read_ground_truth(root / "loeval" / "loeval_ground_truth.tsv")
        pred = pd.read_csv(root / "out" / "matching_results.tsv", sep="\t", dtype=str, keep_default_na=False)
        pm = dict(zip(pred["source1_entity_id"], pred["matched_entity_ids"]))
        f_ev = np.mean([_f05(set(filter(None, pm.get(s, "").split(","))), set(g)) for s, g in gt_e.items()])
        loco_rows.append({"trained on": train_c, "scored on (never seen)": eval_c,
                          "hold-out F0.5, trained country": m.get("holdout_macro_f05"),
                          "F0.5, unseen country": f_ev})
        print(loco_rows[-1])
        gc.collect()
    display(pd.DataFrame(loco_rows).round(5))
    del raw, gt_all, gt_owner
else:
    print("unseen-country check skipped (RUN_COUNTRY_CHECK=False)")